# Global Automotive Investment Database — Block 7

## Hong Kong and China: HKEX listings, HKEXnews disclosures, CNINFO filings and Chinese issuer enrichment

This notebook builds the point-in-time Hong Kong and China component of the Global Automotive Investment Database.

It performs four linked tasks:

1. identifies ETF-held Hong Kong and Chinese securities and their economic issuers;
2. resolves HKEX, A-share, H-share, ADR and offshore listing relationships;
3. acquires and deterministically extracts official HKEXnews and CNINFO filings;
4. standardises the resulting accounting facts into the global canonical concept schema.

All extraction and mapping in this block is deterministic. Ambiguous or unresolved records are retained in explicit review queues for Block 9 — AI Enrichment and Quality Control.

### Primary upstream dependencies

- Block 2 — Security Master
- Block 3 — USA / SEC fundamentals
- Block 4 — Europe / ESMA fundamentals and canonical concept schema
- Block 5 — Japan / EDINET fundamentals
- Block 6 — Korea / DART fundamentals

### Primary downstream dependency

- Block 9 — AI-Assisted Quality Control
- Block 10 — Global Fundamentals


In [ ]:
# 1. INSTALLS AND IMPORTS

!pip -q install pandas numpy requests tqdm pyarrow beautifulsoup4 lxml pypdf pymupdf psutil rapidfuzz unidecode

from __future__ import annotations

import ctypes
import gc
import hashlib
import html as html_lib
import json
import os
import re
import time

from datetime import datetime, timezone
from pathlib import Path
from typing import Optional
from urllib.parse import urlparse, urljoin

import fitz
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests

from bs4 import BeautifulSoup
from pypdf import PdfReader
from tqdm.auto import tqdm
from rapidfuzz import fuzz, process
from unidecode import unidecode

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 300)

# Import self-check

required_symbols = {
    "fuzz": fuzz,
    "process": process,
    "unidecode": unidecode,
    "BeautifulSoup": BeautifulSoup,
    "PdfReader": PdfReader,
    "tqdm": tqdm,
}

missing_symbols = [
    name
    for name, value in required_symbols.items()
    if value is None
]

if missing_symbols:
    raise RuntimeError(
        f"Missing imported symbols: {missing_symbols}"
    )

print(
    "Import self-check passed:",
    ", ".join(required_symbols),
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 9.9 MB/s eta 0:00:00
Import self-check passed: fuzz, process, unidecode, BeautifulSoup, PdfReader, tqdm


In [ ]:
# 2. SETTINGS, DIRECTORIES AND UPSTREAM INPUTS

import unicodedata

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy"
    )
else:
    PROJECT_ROOT = Path("/content/global_automotive_investment_database")

DATA_ROOT = PROJECT_ROOT / "data"

BLOCK_2_MANIFEST_PATH = DATA_ROOT / "interim" / "block_2" / "block_2_manifest.json"
BLOCK_3_MANIFEST_PATH = DATA_ROOT / "interim" / "block_3" / "block_3_manifest.json"
BLOCK_4_MANIFEST_PATH = DATA_ROOT / "interim" / "block_4" / "block_4_manifest.json"
BLOCK_5_MANIFEST_PATH = DATA_ROOT / "interim" / "block_5" / "block_5_manifest.json"
BLOCK_6_MANIFEST_PATH = DATA_ROOT / "interim" / "block_6" / "block_6_manifest.json"
BLOCK_9_MANIFEST_PATH = DATA_ROOT / "interim" / "block_9" / "block_9_manifest.json"

APPLY_BLOCK_9_SYNONYM_REGISTRY = True

BLOCK_7_OUTPUT_DIR = DATA_ROOT / "interim" / "block_7"
BLOCK_7_MANIFEST_PATH = BLOCK_7_OUTPUT_DIR / "block_7_manifest.json"

HKEX_RAW_DIR = DATA_ROOT / "raw" / "hkexnews"
HKEX_REFERENCE_CACHE_DIR = HKEX_RAW_DIR / "reference"
HKEX_SEARCH_CACHE_DIR = HKEX_RAW_DIR / "search"
HKEX_DOCUMENT_CACHE_DIR = HKEX_RAW_DIR / "documents"
HKEX_TEXT_CACHE_DIR = HKEX_RAW_DIR / "text"

HKEX_EXTERNAL_DIR = DATA_ROOT / "external" / "hkexnews"
HKEX_OVERRIDE_PATH = HKEX_EXTERNAL_DIR / "hkex_stock_id_overrides.csv"
HKEX_HISTORICAL_LISTINGS_PATH = (
    HKEX_EXTERNAL_DIR / "hkex_historical_listings.csv"
)
HKEX_NAME_ALIASES_PATH = (
    HKEX_EXTERNAL_DIR / "hkex_name_aliases.csv"
)

for directory in [
    BLOCK_7_OUTPUT_DIR,
    HKEX_REFERENCE_CACHE_DIR,
    HKEX_SEARCH_CACHE_DIR,
    HKEX_DOCUMENT_CACHE_DIR,
    HKEX_TEXT_CACHE_DIR,
    HKEX_EXTERNAL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

HKEX_BASE = "https://www1.hkexnews.hk"
HKEX_TITLE_SEARCH_URL = f"{HKEX_BASE}/search/titlesearch.xhtml"
HKEX_ACTIVE_STOCKS_URL = (
    f"{HKEX_BASE}/ncms/script/eds/activestock_sehk_e.json"
)

DISCOVERY_START_DATE = "2019-10-01"
DISCOVERY_END_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")

REQUEST_INTERVAL_SECONDS = 0.40
REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 5
MAX_SEARCH_PAGES = 20

MAX_SECURITIES = None
MAX_DOCUMENTS_TO_DOWNLOAD = None
MAX_PDF_PAGES = None


# Seed records initialise persistent reference files only when those files
# do not already exist. Users can extend them without editing the notebook.
HISTORICAL_LISTING_SEED = [
    {
        "stock_code": "00489",
        "hkex_stock_id": "10226",
        "hkex_display_name": "DONGFENG GROUP",
        "listing_start_date": pd.NA,
        "listing_end_date": "2026-03-18",
        "listing_status": "DELISTED",
        "historical_reason": (
            "Historical HKEX listing retained after privatisation/delisting"
        ),
    },
]

NAME_ALIAS_SEED = [
    {
        "stock_code": "02333",
        "security_master_name": "GREAT WALL MOTOR",
        "hkex_name_alias": "GWMOTOR",
        "alias_reason": "HKEX abbreviated issuer display name",
    },
    {
        "stock_code": "00175",
        "security_master_name": "GEELY",
        "hkex_name_alias": "GEELY AUTO",
        "alias_reason": "HKEX abbreviated issuer display name",
    },
    {
        "stock_code": "01211",
        "security_master_name": "BYD",
        "hkex_name_alias": "BYD COMPANY",
        "alias_reason": "HKEX issuer display name",
    },
    {
        "stock_code": "02238",
        "security_master_name": "GUANGZHOU AUTOMOBILE GROUP",
        "hkex_name_alias": "GAC GROUP",
        "alias_reason": "HKEX abbreviated issuer display name",
    },
]

SEARCH_LANGUAGE = "EN"
INCLUDE_RESULT_ANNOUNCEMENTS = True
INCLUDE_LONG_FORM_REPORTS = True
SUPPRESS_DUPLICATE_STRUCTURED_FUNDAMENTALS = True
MIN_STRUCTURED_FACT_ROWS = 20
MIN_STRUCTURED_UNIQUE_CONCEPTS = 5

PERSIST_BLOCK_7_OUTPUTS = True
OVERWRITE_PERSISTED_OUTPUTS = True
PARQUET_CHUNK_ROWS = 5_000

TARGET_TITLE_PATTERN = re.compile(
    r"(annual report|interim report|half[- ]year report|"
    r"annual results|final results|interim results|half[- ]year results|"
    r"年度報告|年報|中期報告|半年報|年度業績|全年業績|中期業績)",
    flags=re.IGNORECASE,
)

RESULT_ANNOUNCEMENT_PATTERN = re.compile(
    r"(annual results|final results|interim results|half[- ]year results|"
    r"年度業績|全年業績|中期業績)",
    flags=re.IGNORECASE,
)

LONG_FORM_REPORT_PATTERN = re.compile(
    r"(annual report|interim report|half[- ]year report|"
    r"年度報告|年報|中期報告|半年報)",
    flags=re.IGNORECASE,
)

EXCLUDED_INSTRUMENT_PATTERN = re.compile(
    r"(FUT|FUTURE|INDEX|STOXX|S\s*&?\s*P|E[- ]?MINI|"
    r"HONG KONG DOLLAR|CURRENCY|FX|FORWARD|SWAP|OPTION)",
    flags=re.IGNORECASE,
)


def load_manifest_tables(manifest_path, required_names):
    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing upstream manifest: {manifest_path}")

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    records = {
        item["table_name"]: item
        for item in manifest.get("tables", [])
    }

    missing = set(required_names).difference(records)

    if missing:
        raise RuntimeError(
            f"{manifest_path.name} is missing required tables: {sorted(missing)}"
        )

    loaded = {}

    for name in required_names:
        path = Path(records[name]["path"])

        if not path.exists():
            raise FileNotFoundError(path)

        loaded[name] = pd.read_parquet(path)

    return loaded, manifest


block_2_inputs, block_2_manifest = load_manifest_tables(
    BLOCK_2_MANIFEST_PATH,
    {
        "security_master_df",
        "issuer_master_df",
        "security_identifier_history_df",
    },
)

security_master_df = block_2_inputs["security_master_df"]
issuer_master_df = block_2_inputs["issuer_master_df"]
security_identifier_history_df = block_2_inputs["security_identifier_history_df"]



def load_optional_manifest_tables(
    manifest_path,
    requested_names,
):
    """
    Load only tables that are present in an upstream manifest.

    A missing regional manifest does not prevent HKEX ingestion; it simply
    means that the corresponding structured source cannot suppress duplicate
    HKEX fundamental extraction during that run.
    """
    if not manifest_path.exists():
        return {}, {
            "manifest_path": str(manifest_path),
            "status": "MANIFEST_NOT_FOUND",
            "tables": [],
        }

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    records = {
        item["table_name"]: item
        for item in manifest.get("tables", [])
    }

    loaded = {}

    for name in requested_names:
        record = records.get(name)

        if record is None:
            continue

        path = Path(record["path"])

        if path.exists():
            loaded[name] = pd.read_parquet(path)

    manifest["manifest_path"] = str(manifest_path)
    manifest["status"] = "LOADED"

    return loaded, manifest


block_3_inputs, block_3_manifest = load_optional_manifest_tables(
    BLOCK_3_MANIFEST_PATH,
    {
        "sec_fundamentals_security_linked_df",
        "sec_fundamentals_standardised_df",
        "sec_standard_concept_dictionary_df",
    },
)

block_5_inputs, block_5_manifest = load_optional_manifest_tables(
    BLOCK_5_MANIFEST_PATH,
    {
        "japan_fundamentals_standardised_df",
        "japan_standard_concept_dictionary_df",
        "japan_source_concept_mapping_df",
    },
)

block_6_inputs, block_6_manifest = load_optional_manifest_tables(
    BLOCK_6_MANIFEST_PATH,
    {
        "korea_fundamentals_standardised_df",
        "korea_standard_concept_dictionary_df",
        "korea_source_account_mapping_df",
        "korea_observed_account_mapping_df",
    },
)


block_4_inputs, block_4_manifest = load_manifest_tables(
    BLOCK_4_MANIFEST_PATH,
    {"europe_standard_concept_dictionary_df"},
)

global_canonical_schema_df = (
    block_4_inputs["europe_standard_concept_dictionary_df"][
        [
            "standard_concept",
            "statement_type",
            "expected_period_type",
            "expected_unit_family",
            "core_tier",
            "is_core",
            "aggregation_policy",
        ]
    ]
    .drop_duplicates("standard_concept")
    .reset_index(drop=True)
)



# ------------------------------------------------------------
# Block 9 accepted-synonym feedback
# ------------------------------------------------------------

BLOCK9_SYNONYM_REGISTRY_COLUMNS = [
    "normalised_source_account_label",
    "standard_concept",
    "statement_type",
    "reporting_scope",
    "accepted_observations",
    "unique_issuers",
    "unique_filings",
    "first_accepted_at_utc",
    "last_accepted_at_utc",
    "registry_generation",
]


def normalise_block9_synonym_label(value):
    """Use the same Unicode-safe label form for registry and live labels."""
    if pd.isna(value):
        return pd.NA

    text = unicodedata.normalize("NFKC", str(value)).casefold()
    text = re.sub(r"[\u00a0\s]+", " ", text)
    text = re.sub(
        r"[^\w\u3400-\u9fff]+",
        " ",
        text,
        flags=re.UNICODE,
    )
    text = re.sub(r"\s+", " ", text).strip()

    return text if text else pd.NA


def _manifest_output_records(payload):
    records = []

    for key in ["tables", "outputs"]:
        value = payload.get(key)

        if isinstance(value, list):
            records.extend(value)

    return [record for record in records if isinstance(record, dict)]


def load_block9_synonym_registry(manifest_path):
    """Load Block 9's cumulative accepted-synonym registry if available."""
    empty = pd.DataFrame(columns=BLOCK9_SYNONYM_REGISTRY_COLUMNS)
    manifest_path = Path(manifest_path)

    status = {
        "status": "NOT_AVAILABLE",
        "manifest_path": str(manifest_path),
        "loaded_table_name": None,
        "loaded_file": None,
        "loaded_rows": 0,
        "error": pd.NA,
    }

    if not APPLY_BLOCK_9_SYNONYM_REGISTRY:
        status["status"] = "DISABLED"
        return empty, status

    if not manifest_path.exists():
        status["status"] = "MANIFEST_NOT_FOUND"
        return empty, status

    try:
        payload = json.loads(
            manifest_path.read_text(encoding="utf-8")
        )
    except Exception as exc:
        status["status"] = "MANIFEST_READ_FAILED"
        status["error"] = repr(exc)
        return empty, status

    aliases = {
        "accepted_synonym_registry",
        "accepted_synonym_registry_df",
    }

    candidates = []

    for record in _manifest_output_records(payload):
        table_name = str(
            record.get(
                "table_name",
                record.get("name", record.get("table", "")),
            )
        ).strip()

        if table_name not in aliases:
            continue

        record_status = str(record.get("status", "PASSED")).upper()

        if not (
            record_status.startswith("PASS")
            or record_status in {"SUCCESS", "SUCCEEDED", "LOADED"}
        ):
            continue

        file_path = (
            record.get("saved_file")
            or record.get("path")
            or record.get("file_path")
            or record.get("output_path")
        )

        if not file_path:
            continue

        candidates.append({
            "table_name": table_name,
            "file_type": str(record.get("file_type", "")).lower(),
            "path": Path(file_path),
        })

    if not candidates:
        for filename in [
            "accepted_synonym_registry.parquet",
            "accepted_synonym_registry_df.parquet",
            "accepted_synonym_registry.csv",
            "accepted_synonym_registry_df.csv",
        ]:
            candidate_path = manifest_path.parent / filename

            if candidate_path.exists():
                candidates.append({
                    "table_name": candidate_path.stem,
                    "file_type": candidate_path.suffix.lstrip(".").lower(),
                    "path": candidate_path,
                })

    if not candidates:
        status["status"] = "TABLE_NOT_FOUND"
        return empty, status

    candidates = sorted(
        candidates,
        key=lambda item: (
            item["file_type"] != "parquet",
            item["table_name"] != "accepted_synonym_registry",
        ),
    )

    selected = candidates[0]

    if not selected["path"].exists():
        status["status"] = "FILE_NOT_FOUND"
        status["loaded_table_name"] = selected["table_name"]
        status["loaded_file"] = str(selected["path"])
        return empty, status

    try:
        if (
            selected["file_type"] == "csv"
            or selected["path"].suffix.lower() == ".csv"
        ):
            registry = pd.read_csv(
                selected["path"],
                low_memory=False,
            )
        else:
            registry = pd.read_parquet(selected["path"])
    except Exception as exc:
        status["status"] = "LOAD_FAILED"
        status["loaded_table_name"] = selected["table_name"]
        status["loaded_file"] = str(selected["path"])
        status["error"] = repr(exc)
        return empty, status

    for column in BLOCK9_SYNONYM_REGISTRY_COLUMNS:
        if column not in registry.columns:
            registry[column] = pd.NA

    registry = registry[BLOCK9_SYNONYM_REGISTRY_COLUMNS].copy()

    status.update({
        "status": "LOADED",
        "loaded_table_name": selected["table_name"],
        "loaded_file": str(selected["path"]),
        "loaded_rows": int(len(registry)),
    })

    return registry, status


(
    block9_accepted_synonym_registry_raw_df,
    block9_synonym_registry_load_status,
) = load_block9_synonym_registry(
    BLOCK_9_MANIFEST_PATH
)


if block9_accepted_synonym_registry_raw_df.empty:
    block9_accepted_synonym_registry_df = pd.DataFrame(
        columns=BLOCK9_SYNONYM_REGISTRY_COLUMNS
    )
    block9_synonym_registry_conflicts_df = pd.DataFrame(
        columns=BLOCK9_SYNONYM_REGISTRY_COLUMNS
    )
else:
    registry = block9_accepted_synonym_registry_raw_df.copy()

    registry["normalised_source_account_label"] = registry[
        "normalised_source_account_label"
    ].map(normalise_block9_synonym_label)

    registry["standard_concept"] = (
        registry["standard_concept"]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "<NA>": pd.NA})
    )

    for column in [
        "statement_type",
        "reporting_scope",
    ]:
        registry[column] = (
            registry[column]
            .astype("string")
            .str.strip()
            .replace({"": pd.NA, "<NA>": pd.NA})
        )

    for column in [
        "accepted_observations",
        "unique_issuers",
        "unique_filings",
    ]:
        registry[column] = pd.to_numeric(
            registry[column],
            errors="coerce",
        ).fillna(0)

    valid_concepts = set(
        global_canonical_schema_df["standard_concept"]
        .dropna()
        .astype("string")
    )

    registry = registry.loc[
        registry["normalised_source_account_label"].notna()
        & registry["standard_concept"].isin(valid_concepts)
    ].copy()

    concept_counts = (
        registry.groupby(
            "normalised_source_account_label",
            dropna=False,
        )["standard_concept"]
        .nunique(dropna=True)
    )

    unambiguous_labels = set(
        concept_counts.loc[concept_counts == 1].index
    )
    conflicting_labels = set(
        concept_counts.loc[concept_counts > 1].index
    )

    block9_synonym_registry_conflicts_df = (
        registry.loc[
            registry["normalised_source_account_label"].isin(
                conflicting_labels
            )
        ]
        .sort_values(
            [
                "normalised_source_account_label",
                "accepted_observations",
                "last_accepted_at_utc",
            ],
            ascending=[True, False, False],
        )
        .reset_index(drop=True)
    )

    block9_accepted_synonym_registry_df = (
        registry.loc[
            registry["normalised_source_account_label"].isin(
                unambiguous_labels
            )
        ]
        .sort_values(
            [
                "normalised_source_account_label",
                "accepted_observations",
                "last_accepted_at_utc",
            ],
            ascending=[True, False, False],
        )
        .drop_duplicates(
            "normalised_source_account_label",
            keep="first",
        )
        .reset_index(drop=True)
    )


block9_synonym_lookup = (
    block9_accepted_synonym_registry_df
    .set_index("normalised_source_account_label")
    .to_dict(orient="index")
    if not block9_accepted_synonym_registry_df.empty
    else {}
)

block9_synonym_registry_status_df = pd.DataFrame([
    {
        **block9_synonym_registry_load_status,
        "usable_registry_rows": int(
            len(block9_accepted_synonym_registry_df)
        ),
        "conflicting_registry_rows": int(
            len(block9_synonym_registry_conflicts_df)
        ),
        "unique_usable_labels": int(
            block9_accepted_synonym_registry_df[
                "normalised_source_account_label"
            ].nunique(dropna=True)
            if not block9_accepted_synonym_registry_df.empty
            else 0
        ),
    }
])

print("Block 9 synonym feedback status:")
display(block9_synonym_registry_status_df)


print("Canonical concepts:", global_canonical_schema_df["standard_concept"].nunique())
print("Block 7 output directory:", BLOCK_7_OUTPUT_DIR)

Mounted at /content/drive
Block 9 synonym feedback status:


,status,manifest_path,loaded_table_name,loaded_file,loaded_rows,error,usable_registry_rows,conflicting_registry_rows,unique_usable_labels
0,LOADED,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,accepted_synonym_registry,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,531,<NA>,435,5,435


Canonical concepts: 100
Block 7 output directory: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_7


In [ ]:
# 3. BUILD THE HONG KONG LISTING-CANDIDATE UNIVERSE

def first_existing_column(dataframe, candidates):
    return next(
        (column for column in candidates if column in dataframe.columns),
        None,
    )


def normalise_country(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().upper()

    aliases = {
        "HONG KONG": "HK",
        "HONG KONG SAR": "HK",
        "HKG": "HK",
        "CHINA - HONG KONG": "HK",
        "CHINA HONG KONG": "HK",
    }

    return aliases.get(text, text)


def extract_numeric_listing_code(value):
    """
    Extract a plausible one-to-five-digit listing code.

    Accepted examples include 1211, 01211, 1211.HK, HK:1211 and 1211 HK.
    Alphabetic tickers such as VOW3, AMV0 and PAH3 are rejected.
    """
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().upper()

    patterns = [
        r"^(?:HK|HKG|SEHK|HKEX)[:\s\-]*(\d{1,5})$",
        r"^(\d{1,5})(?:\.HK|-HK|\s+HK)$",
        r"^(\d{1,5})$",
    ]

    for pattern in patterns:
        match = re.fullmatch(pattern, text)

        if match:
            return match.group(1).zfill(5)

    return pd.NA


country_col = first_existing_column(
    security_master_df,
    ["country", "issuer_country", "domicile_country"],
)
exchange_col = first_existing_column(
    security_master_df,
    ["exchange", "primary_exchange", "listing_exchange", "market"],
)
ticker_col = first_existing_column(
    security_master_df,
    ["ticker", "primary_ticker", "source_ticker"],
)
name_col = first_existing_column(
    security_master_df,
    ["issuer_name", "security_name", "name"],
)

candidates = pd.DataFrame(index=security_master_df.index)

candidates["security_id"] = security_master_df.get("security_id")
candidates["issuer_id"] = security_master_df.get("issuer_id")
candidates["issuer_name"] = (
    security_master_df[name_col] if name_col else pd.NA
)
candidates["ticker"] = (
    security_master_df[ticker_col] if ticker_col else pd.NA
)
candidates["country"] = (
    security_master_df[country_col].map(normalise_country)
    if country_col
    else pd.NA
)
candidates["exchange"] = (
    security_master_df[exchange_col].astype("string")
    if exchange_col
    else pd.NA
)

ticker_text = candidates["ticker"].astype("string").str.upper().str.strip()
exchange_text = candidates["exchange"].astype("string").str.upper().fillna("")
name_text = candidates["issuer_name"].astype("string").fillna("")

candidates["stock_code"] = (
    candidates["ticker"].map(extract_numeric_listing_code)
)

candidates["signal_hk_ticker_suffix"] = ticker_text.str.contains(
    r"(\.HK|-HK|\sHK)$",
    regex=True,
    na=False,
)

candidates["signal_hk_exchange"] = exchange_text.str.contains(
    r"HKEX|SEHK|HONG KONG|XHKG",
    regex=True,
    na=False,
)

candidates["signal_hk_country"] = candidates["country"].eq("HK")

candidates["explicit_hk_signal"] = (
    candidates[
        [
            "signal_hk_ticker_suffix",
            "signal_hk_exchange",
            "signal_hk_country",
        ]
    ]
    .fillna(False)
    .any(axis=1)
)

candidates["excluded_instrument_signal"] = name_text.str.contains(
    EXCLUDED_INSTRUMENT_PATTERN,
    na=False,
)

if "security_id" in security_identifier_history_df.columns:
    identifier_type_col = first_existing_column(
        security_identifier_history_df,
        ["identifier_type", "id_type"],
    )
    identifier_value_col = first_existing_column(
        security_identifier_history_df,
        ["identifier_value", "id_value"],
    )

    if identifier_type_col and identifier_value_col:
        history = security_identifier_history_df[
            [
                "security_id",
                identifier_type_col,
                identifier_value_col,
            ]
        ].copy()

        history["identifier_type_upper"] = (
            history[identifier_type_col]
            .astype("string")
            .str.upper()
        )
        history["identifier_value_upper"] = (
            history[identifier_value_col]
            .astype("string")
            .str.upper()
        )

        history["history_stock_code"] = (
            history[identifier_value_col]
            .map(extract_numeric_listing_code)
        )

        history["history_hk_signal"] = (
            history["identifier_type_upper"].str.contains(
                r"HKEX|SEHK|XHKG|HONG KONG",
                regex=True,
                na=False,
            )
            |
            history["identifier_value_upper"].str.contains(
                r"(\.HK|-HK|\sHK|HK:|HKG:|SEHK:)",
                regex=True,
                na=False,
            )
        )

        history_summary = (
            history.groupby("security_id", dropna=False)
            .agg(
                history_stock_code=(
                    "history_stock_code",
                    lambda series: series.dropna().iloc[0]
                    if not series.dropna().empty
                    else pd.NA,
                ),
                history_hk_signal=("history_hk_signal", "max"),
            )
            .reset_index()
        )

        candidates = candidates.merge(
            history_summary,
            on="security_id",
            how="left",
            validate="1:1",
        )

        candidates["stock_code"] = (
            candidates["stock_code"]
            .combine_first(candidates["history_stock_code"])
        )

        candidates["explicit_hk_signal"] = (
            candidates["explicit_hk_signal"]
            | candidates["history_hk_signal"].fillna(False)
        )

        candidates = candidates.drop(
            columns=[
                "history_stock_code",
                "history_hk_signal",
            ]
        )

candidate_mask = (
    candidates["explicit_hk_signal"]
    | candidates["stock_code"].notna()
)

candidate_mask &= ~candidates["excluded_instrument_signal"]

hong_kong_security_candidate_df = (
    candidates[candidate_mask]
    .drop_duplicates()
    .reset_index(drop=True)
)

hong_kong_universe_exclusions_df = (
    candidates[~candidate_mask]
    .copy()
    .reset_index(drop=True)
)

if MAX_SECURITIES is not None:
    hong_kong_security_candidate_df = (
        hong_kong_security_candidate_df.head(int(MAX_SECURITIES))
    )

print("Broad Hong Kong listing candidates:", len(hong_kong_security_candidate_df))
print(
    "Candidates with explicit HK signals:",
    int(hong_kong_security_candidate_df["explicit_hk_signal"].sum()),
)
print(
    "Ambiguous numeric-ticker candidates requiring name validation:",
    int((~hong_kong_security_candidate_df["explicit_hk_signal"]).sum()),
)

display(
    hong_kong_security_candidate_df[
        [
            "security_id",
            "issuer_id",
            "issuer_name",
            "ticker",
            "country",
            "exchange",
            "stock_code",
            "explicit_hk_signal",
        ]
    ].head(150)
)

/tmp/ipykernel_1172/995700960.py:100: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  candidates["signal_hk_ticker_suffix"] = ticker_text.str.contains(
/tmp/ipykernel_1172/995700960.py:126: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  candidates["excluded_instrument_signal"] = name_text.str.contains(
/tmp/ipykernel_1172/995700960.py:173: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  history["identifier_value_upper"].str.contains(


Broad Hong Kong listing candidates: 45
Candidates with explicit HK signals: 23
Ambiguous numeric-ticker candidates requiring name validation: 22


,security_id,issuer_id,issuer_name,ticker,country,exchange,stock_code,explicit_hk_signal
0,GAS_025E74C831EDA3481EFF,GAI_DFA494ABBAA0C6DE156F,Black Sesame International Hol,2533,CN,HKEX,02533,True
1,GAS_034768D81B72F9474880,GAI_09A6CB2659CE9A5A88D6,Nissan Motor Co Ltd,7201,JP,TSE,07201,False
2,GAS_069EC9AC410A90F3CF02,GAI_31167928D5EA6BDD77F6,TIANNENG POWER INTERNATIONAL LIMITED,None,HK,<NA>,<NA>,True
3,GAS_0AD6D2407B9614C625F3,GAI_5586FB58001D48BE17BA,XPENG INC.,9868,KY,HKEX,09868,True
4,GAS_0F0802D8983EE892AD09,GAI_118982B3D7A3654914FD,Ganfeng Lithium Group Co Ltd,1772,CN,HKEX,01772,True
5,GAS_1154405F1CBDE8821F51,GAI_A9415D33586CC4D4C537,Panasonic Holdings Corporation,6752,JP,TSE,06752,False
6,GAS_13B2314857701B0279D9,GAI_6C11D1870AA7C551AAA2,Yulon Motor Co Ltd,2201,TW,TWSE,02201,False
7,GAS_13C65F140E096987B8D9,GAI_E7D0EF071B325FF8D2F9,Suzuki Motor Corp,7269,JP,TSE,07269,False
8,GAS_14866004A8FB48938772,GAI_2DBE2C08A8284536411A,Jinchuan Group International Resources Co Ltd,None,HK,<NA>,<NA>,True
9,GAS_1D88ED8C4181248BCD3E,GAI_C2B1739103598D1255E1,Subaru Corp,7270,JP,TSE,07270,False


In [ ]:
# 4. HKEX HTTP CLIENT AND PERSISTENT CACHE

session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/142.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/json,"
              "application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-GB,en;q=0.9,zh-HK;q=0.8,zh;q=0.7",
})


def cache_key(url, params=None, data=None):
    payload = json.dumps(
        {
            "url": url,
            "params": params or {},
            "data": data or {},
        },
        sort_keys=True,
        default=str,
        ensure_ascii=False,
    )

    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def request_with_retries(method, url, **kwargs):
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.request(
                method,
                url,
                timeout=REQUEST_TIMEOUT_SECONDS,
                **kwargs,
            )

            response.raise_for_status()
            time.sleep(REQUEST_INTERVAL_SECONDS)

            return response, attempt

        except Exception as exc:
            last_error = repr(exc)
            time.sleep(min(2 ** attempt, 20))

    raise RuntimeError(
        f"{method} failed after {MAX_RETRIES} attempts for {url}: "
        f"{last_error}"
    )


def cached_get_bytes(
    url,
    *,
    params=None,
    cache_dir,
    suffix,
    headers=None,
):
    path = cache_dir / f"{cache_key(url, params=params)}{suffix}"

    if path.exists():
        return path.read_bytes(), {
            "status": "CACHE_HIT",
            "cache_path": str(path),
            "url": url,
            "http_status": 200,
            "attempt": 0,
        }

    response, attempt = request_with_retries(
        "GET",
        url,
        params=params,
        headers=headers,
    )

    path.write_bytes(response.content)

    return response.content, {
        "status": "DOWNLOADED",
        "cache_path": str(path),
        "url": response.url,
        "http_status": response.status_code,
        "attempt": attempt,
    }


def cached_post_text(
    url,
    *,
    data,
    cache_dir,
    suffix=".html",
    headers=None,
):
    path = cache_dir / f"{cache_key(url, data=data)}{suffix}"

    if path.exists():
        return path.read_text(
            encoding="utf-8",
            errors="replace",
        ), {
            "status": "CACHE_HIT",
            "cache_path": str(path),
            "url": url,
            "http_status": 200,
            "attempt": 0,
        }

    response, attempt = request_with_retries(
        "POST",
        url,
        data=data,
        headers=headers,
    )

    path.write_text(
        response.text,
        encoding="utf-8",
    )

    return response.text, {
        "status": "DOWNLOADED",
        "cache_path": str(path),
        "url": response.url,
        "http_status": response.status_code,
        "attempt": attempt,
    }

In [ ]:
# 5. RESOLVE ACTIVE AND HISTORICAL HKEX LISTINGS

active_content, active_log = cached_get_bytes(
    HKEX_ACTIVE_STOCKS_URL,
    cache_dir=HKEX_REFERENCE_CACHE_DIR,
    suffix=".json",
)

active_payload = json.loads(
    active_content.decode("utf-8", errors="replace")
)

if isinstance(active_payload, dict):
    for key in ["data", "stocks", "items", "results"]:
        if isinstance(active_payload.get(key), list):
            active_payload = active_payload[key]
            break

if not isinstance(active_payload, list):
    raise RuntimeError(
        "The HKEX active-stock endpoint did not return a list."
    )


def first_value(item, candidates):
    return next(
        (
            item.get(candidate)
            for candidate in candidates
            if item.get(candidate) is not None
        ),
        None,
    )


def normalise_company_name(value):
    if pd.isna(value):
        return ""

    text = html_lib.unescape(str(value)).upper()

    text = re.sub(
        r"\b(THE|LIMITED|LTD|HOLDINGS?|GROUP|CORPORATION|CORP|"
        r"COMPANY|CO|PLC|INCORPORATED|INC|HOLDING|INTL|"
        r"INTERNATIONAL|股份有限公司|有限公司|集團|控股)\b",
        " ",
        text,
    )

    text = re.sub(r"[^A-Z0-9\u3400-\u9FFF]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def meaningful_name_tokens(value):
    return {
        token
        for token in normalise_company_name(value).split()
        if len(token) >= 3
    }


def active_display_name(item):
    ignored = {
        "c",
        "i",
        "code",
        "id",
        "stockCode",
        "stockId",
    }

    values = []

    for key, value in item.items():
        if key in ignored or value is None:
            continue

        if isinstance(value, (str, int, float)):
            text = str(value).strip()

            if text and not re.fullmatch(r"\d+", text):
                values.append(text)

    return " | ".join(dict.fromkeys(values))


# ------------------------------------------------------------
# 5A. Current active-stock registry
# ------------------------------------------------------------

active_rows = []

for item in active_payload:
    if not isinstance(item, dict):
        continue

    stock_code = extract_numeric_listing_code(
        first_value(item, ["c", "code", "stockCode"])
    )

    stock_id = first_value(
        item,
        ["i", "id", "stockId"],
    )

    display_name = active_display_name(item)

    if pd.isna(stock_code) or stock_id is None:
        continue

    active_rows.append({
        "stock_code": stock_code,
        "hkex_stock_id": str(stock_id),
        "hkex_display_name": display_name,
        "hkex_name_normalised": normalise_company_name(display_name),
        "listing_start_date": pd.NaT,
        "listing_end_date": pd.NaT,
        "listing_status": "ACTIVE",
        "identifier_source": "HKEX_ACTIVE_STOCK_JSON",
        "listing_registry_reason": (
            "Present in current official HKEX active-stock JSON"
        ),
        "active_stock_raw_json": json.dumps(
            item,
            ensure_ascii=False,
        ),
    })


hkex_active_stock_list_df = pd.DataFrame(active_rows)

hkex_active_stock_duplicate_report_df = (
    hkex_active_stock_list_df[
        hkex_active_stock_list_df["stock_code"].duplicated(
            keep=False
        )
    ]
    .sort_values("stock_code")
    .reset_index(drop=True)
)

hkex_active_stock_unique_df = (
    hkex_active_stock_list_df
    .drop_duplicates("stock_code", keep="first")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5B. Persistent historical/delisted-listing registry
# ------------------------------------------------------------

historical_columns = [
    "stock_code",
    "hkex_stock_id",
    "hkex_display_name",
    "listing_start_date",
    "listing_end_date",
    "listing_status",
    "historical_reason",
]

if not HKEX_HISTORICAL_LISTINGS_PATH.exists():
    pd.DataFrame(
        HISTORICAL_LISTING_SEED,
        columns=historical_columns,
    ).to_csv(
        HKEX_HISTORICAL_LISTINGS_PATH,
        index=False,
    )

hkex_historical_listings_df = pd.read_csv(
    HKEX_HISTORICAL_LISTINGS_PATH,
    dtype="string",
)

for column in historical_columns:
    if column not in hkex_historical_listings_df.columns:
        hkex_historical_listings_df[column] = pd.NA

hkex_historical_listings_df["stock_code"] = (
    hkex_historical_listings_df["stock_code"]
    .map(extract_numeric_listing_code)
)

for column in [
    "listing_start_date",
    "listing_end_date",
]:
    hkex_historical_listings_df[column] = pd.to_datetime(
        hkex_historical_listings_df[column],
        errors="coerce",
    )

hkex_historical_listings_df["hkex_name_normalised"] = (
    hkex_historical_listings_df["hkex_display_name"]
    .map(normalise_company_name)
)

hkex_historical_listings_df["identifier_source"] = (
    "HISTORICAL_LISTING_REGISTRY"
)

hkex_historical_listings_df["listing_registry_reason"] = (
    hkex_historical_listings_df["historical_reason"]
)


# ------------------------------------------------------------
# 5C. Optional manual overrides
# ------------------------------------------------------------

override_columns = [
    "stock_code",
    "hkex_stock_id",
    "hkex_display_name",
    "listing_start_date",
    "listing_end_date",
    "listing_status",
    "override_reason",
]

if not HKEX_OVERRIDE_PATH.exists():
    pd.DataFrame(columns=override_columns).to_csv(
        HKEX_OVERRIDE_PATH,
        index=False,
    )

hkex_stock_id_overrides_df = pd.read_csv(
    HKEX_OVERRIDE_PATH,
    dtype="string",
)

for column in override_columns:
    if column not in hkex_stock_id_overrides_df.columns:
        hkex_stock_id_overrides_df[column] = pd.NA

hkex_stock_id_overrides_df["stock_code"] = (
    hkex_stock_id_overrides_df["stock_code"]
    .map(extract_numeric_listing_code)
)

for column in [
    "listing_start_date",
    "listing_end_date",
]:
    hkex_stock_id_overrides_df[column] = pd.to_datetime(
        hkex_stock_id_overrides_df[column],
        errors="coerce",
    )

hkex_stock_id_overrides_df["hkex_name_normalised"] = (
    hkex_stock_id_overrides_df["hkex_display_name"]
    .map(normalise_company_name)
)

hkex_stock_id_overrides_df["identifier_source"] = (
    "MANUAL_OVERRIDE"
)

hkex_stock_id_overrides_df["listing_registry_reason"] = (
    hkex_stock_id_overrides_df["override_reason"]
)

override_valid = (
    hkex_stock_id_overrides_df[
        hkex_stock_id_overrides_df["stock_code"].notna()
        & hkex_stock_id_overrides_df["hkex_stock_id"].notna()
    ]
    .copy()
)


# ------------------------------------------------------------
# 5D. Consolidated listing registry
#
# Precedence:
#   manual override
#   active official record
#   historical/delisted registry
# ------------------------------------------------------------

registry_columns = [
    "stock_code",
    "hkex_stock_id",
    "hkex_display_name",
    "hkex_name_normalised",
    "listing_start_date",
    "listing_end_date",
    "listing_status",
    "identifier_source",
    "listing_registry_reason",
]

hkex_stock_identifier_map_df = pd.concat(
    [
        override_valid[registry_columns],
        hkex_active_stock_unique_df[registry_columns],
        hkex_historical_listings_df[registry_columns],
    ],
    ignore_index=True,
)

hkex_stock_identifier_map_df = (
    hkex_stock_identifier_map_df
    .dropna(subset=["stock_code", "hkex_stock_id"])
    .drop_duplicates("stock_code", keep="first")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5E. Persistent issuer-name aliases
# ------------------------------------------------------------

alias_columns = [
    "stock_code",
    "security_master_name",
    "hkex_name_alias",
    "alias_reason",
]

if not HKEX_NAME_ALIASES_PATH.exists():
    pd.DataFrame(
        NAME_ALIAS_SEED,
        columns=alias_columns,
    ).to_csv(
        HKEX_NAME_ALIASES_PATH,
        index=False,
    )

hkex_name_aliases_df = pd.read_csv(
    HKEX_NAME_ALIASES_PATH,
    dtype="string",
)

for column in alias_columns:
    if column not in hkex_name_aliases_df.columns:
        hkex_name_aliases_df[column] = pd.NA

hkex_name_aliases_df["stock_code"] = (
    hkex_name_aliases_df["stock_code"]
    .map(extract_numeric_listing_code)
)

hkex_name_aliases_df["security_master_name_normalised"] = (
    hkex_name_aliases_df["security_master_name"]
    .map(normalise_company_name)
)

hkex_name_aliases_df["hkex_name_alias_normalised"] = (
    hkex_name_aliases_df["hkex_name_alias"]
    .map(normalise_company_name)
)

alias_by_code = (
    hkex_name_aliases_df[
        [
            "stock_code",
            "hkex_name_alias",
            "hkex_name_alias_normalised",
            "alias_reason",
        ]
    ]
    .dropna(subset=["stock_code"])
    .drop_duplicates("stock_code")
)


# ------------------------------------------------------------
# 5F. Validate broad candidates
# ------------------------------------------------------------

bridge = hong_kong_security_candidate_df.merge(
    hkex_stock_identifier_map_df,
    on="stock_code",
    how="left",
    validate="m:1",
)

bridge = bridge.merge(
    alias_by_code,
    on="stock_code",
    how="left",
    validate="m:1",
)

bridge["issuer_name_normalised"] = (
    bridge["issuer_name"].map(normalise_company_name)
)

# Alias-enhanced comparison name.
bridge["hkex_comparison_name"] = (
    bridge["hkex_name_alias"]
    .combine_first(bridge["hkex_display_name"])
)

bridge["hkex_comparison_name_normalised"] = (
    bridge["hkex_name_alias_normalised"]
    .combine_first(bridge["hkex_name_normalised"])
)

bridge["name_similarity_score"] = bridge.apply(
    lambda row: (
        fuzz.token_set_ratio(
            row["issuer_name_normalised"],
            row["hkex_comparison_name_normalised"],
        )
        if row["issuer_name_normalised"]
        and pd.notna(row["hkex_comparison_name_normalised"])
        and row["hkex_comparison_name_normalised"]
        else np.nan
    ),
    axis=1,
)

bridge["issuer_name_tokens"] = bridge["issuer_name"].map(
    meaningful_name_tokens
)

bridge["hkex_name_tokens"] = (
    bridge["hkex_comparison_name"].map(
        meaningful_name_tokens
    )
)

bridge["meaningful_token_overlap"] = bridge.apply(
    lambda row: len(
        row["issuer_name_tokens"].intersection(
            row["hkex_name_tokens"]
        )
    ),
    axis=1,
)

bridge["resolved_stock_code"] = (
    bridge["hkex_stock_id"].notna()
)

bridge["is_historical_listing"] = (
    bridge["listing_status"]
    .astype("string")
    .str.upper()
    .isin(
        [
            "DELISTED",
            "INACTIVE",
            "SUSPENDED",
            "HISTORICAL",
        ]
    )
    |
    bridge["listing_end_date"].notna()
)

bridge["accepted_hk_listing"] = (
    bridge["resolved_stock_code"]
    & (
        bridge["explicit_hk_signal"]
        | bridge["identifier_source"].isin(
            [
                "MANUAL_OVERRIDE",
                "HISTORICAL_LISTING_REGISTRY",
            ]
        )
        | bridge["name_similarity_score"].ge(72)
        | bridge["meaningful_token_overlap"].ge(1)
        | bridge["hkex_name_alias"].notna()
    )
)

bridge["universe_resolution_reason"] = np.select(
    [
        bridge["accepted_hk_listing"]
        & bridge["identifier_source"].eq(
            "MANUAL_OVERRIDE"
        ),

        bridge["accepted_hk_listing"]
        & bridge["identifier_source"].eq(
            "HISTORICAL_LISTING_REGISTRY"
        ),

        bridge["accepted_hk_listing"]
        & bridge["hkex_name_alias"].notna(),

        bridge["accepted_hk_listing"]
        & bridge["explicit_hk_signal"],

        bridge["accepted_hk_listing"]
        & bridge["name_similarity_score"].ge(72),

        bridge["accepted_hk_listing"]
        & bridge["meaningful_token_overlap"].ge(1),

        ~bridge["resolved_stock_code"],

        bridge["resolved_stock_code"]
        & ~bridge["accepted_hk_listing"],
    ],
    [
        "ACCEPTED_MANUAL_OVERRIDE",
        "ACCEPTED_HISTORICAL_LISTING",
        "ACCEPTED_NAME_ALIAS",
        "ACCEPTED_EXPLICIT_HK_SIGNAL",
        "ACCEPTED_NAME_SIMILARITY",
        "ACCEPTED_NAME_TOKEN_OVERLAP",
        "REJECTED_CODE_NOT_IN_LISTING_REGISTRY",
        "REJECTED_NAME_MISMATCH",
    ],
    default="REJECTED_OTHER",
)

hong_kong_security_bridge_df = bridge

hong_kong_security_universe_df = (
    bridge[bridge["accepted_hk_listing"]]
    .copy()
    .reset_index(drop=True)
)

hong_kong_universe_validation_rejections_df = (
    bridge[~bridge["accepted_hk_listing"]]
    .copy()
    .reset_index(drop=True)
)

hong_kong_issuer_universe_df = (
    hong_kong_security_universe_df[
        [
            "issuer_id",
            "issuer_name",
            "country",
            "stock_code",
            "listing_start_date",
            "listing_end_date",
            "listing_status",
            "is_historical_listing",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

hong_kong_resolved_securities_df = (
    hong_kong_security_universe_df[
        hong_kong_security_universe_df[
            "hkex_stock_id"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

hong_kong_unresolved_securities_df = (
    hong_kong_security_universe_df[
        hong_kong_security_universe_df[
            "hkex_stock_id"
        ].isna()
    ]
    .copy()
    .reset_index(drop=True)
)

hkex_stock_identifier_resolution_log_df = (
    bridge[
        [
            "security_id",
            "issuer_id",
            "issuer_name",
            "ticker",
            "country",
            "exchange",
            "stock_code",
            "explicit_hk_signal",
            "hkex_stock_id",
            "hkex_display_name",
            "hkex_name_alias",
            "identifier_source",
            "listing_start_date",
            "listing_end_date",
            "listing_status",
            "is_historical_listing",
            "name_similarity_score",
            "meaningful_token_overlap",
            "accepted_hk_listing",
            "universe_resolution_reason",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 5G. Point-in-time listing-validity helper
# ------------------------------------------------------------

def hkex_listing_is_active_as_of(
    dataframe,
    as_of_date,
):
    date_value = pd.Timestamp(as_of_date).normalize()

    listing_start = pd.to_datetime(
        dataframe["listing_start_date"],
        errors="coerce",
    )

    listing_end = pd.to_datetime(
        dataframe["listing_end_date"],
        errors="coerce",
    )

    started = (
        listing_start.isna()
        | listing_start.le(date_value)
    )

    not_ended = (
        listing_end.isna()
        | date_value.lt(listing_end)
    )

    return started & not_ended


def hong_kong_investable_universe_as_of(
    as_of_date,
):
    return (
        hong_kong_security_universe_df[
            hkex_listing_is_active_as_of(
                hong_kong_security_universe_df,
                as_of_date,
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


print("Official active-stock records:", len(hkex_active_stock_list_df))
print("Historical-listing records:", len(hkex_historical_listings_df))
print("Manual override records:", len(override_valid))
print("Broad listing candidates:", len(hong_kong_security_candidate_df))
print("Validated Hong Kong securities:", len(hong_kong_security_universe_df))
print(
    "Historical/delisted securities retained:",
    int(
        hong_kong_security_universe_df[
            "is_historical_listing"
        ].fillna(False).sum()
    ),
)
print(
    "Validated Hong Kong issuers:",
    hong_kong_issuer_universe_df[
        "issuer_id"
    ].nunique(),
)
print(
    "Resolved unique HKEX stock codes:",
    hong_kong_resolved_securities_df[
        "stock_code"
    ].nunique(),
)
print(
    "Rejected ambiguous/non-HK candidates:",
    len(hong_kong_universe_validation_rejections_df),
)

display(
    hong_kong_security_universe_df[
        [
            "security_id",
            "issuer_id",
            "issuer_name",
            "ticker",
            "country",
            "stock_code",
            "hkex_stock_id",
            "hkex_display_name",
            "hkex_name_alias",
            "listing_status",
            "listing_start_date",
            "listing_end_date",
            "is_historical_listing",
            "name_similarity_score",
            "universe_resolution_reason",
        ]
    ]
    .sort_values(
        [
            "is_historical_listing",
            "stock_code",
        ]
    )
    .head(150)
)

print()
print("Highest-scoring rejected candidates — review for false negatives:")

display(
    hong_kong_universe_validation_rejections_df[
        [
            "security_id",
            "issuer_name",
            "ticker",
            "country",
            "exchange",
            "stock_code",
            "hkex_display_name",
            "name_similarity_score",
            "meaningful_token_overlap",
            "universe_resolution_reason",
        ]
    ]
    .sort_values(
        "name_similarity_score",
        ascending=False,
        na_position="last",
    )
    .head(100)
)


# ------------------------------------------------------------
# 5H. Automotive regression test
# ------------------------------------------------------------

EXPECTED_HK_AUTOMOTIVE = {
    "00175": "Geely",
    "00489": "Dongfeng",
    "01114": "Brilliance",
    "01211": "BYD",
    "01958": "BAIC Motor",
    "02015": "Li Auto",
    "02238": "GAC",
    "02333": "Great Wall Motor",
    "09863": "Leapmotor",
    "09868": "XPeng",
}

candidate_codes = set(
    hong_kong_security_candidate_df[
        "stock_code"
    ].dropna()
)

accepted_codes = set(
    hong_kong_security_universe_df[
        "stock_code"
    ].dropna()
)

known_validation_rows = []

for code_value, expected_name in (
    EXPECTED_HK_AUTOMOTIVE.items()
):
    known_validation_rows.append({
        "stock_code": code_value,
        "expected_name": expected_name,
        "present_in_candidate_set": (
            code_value in candidate_codes
        ),
        "accepted_into_hk_universe": (
            code_value in accepted_codes
        ),
        "listing_status": (
            hong_kong_security_universe_df.loc[
                hong_kong_security_universe_df[
                    "stock_code"
                ].eq(code_value),
                "listing_status",
            ].iloc[0]
            if code_value in accepted_codes
            else pd.NA
        ),
    })

hkex_known_automotive_validation_df = pd.DataFrame(
    known_validation_rows
)

display(hkex_known_automotive_validation_df)

expected_present_codes = {
    code_value
    for code_value in EXPECTED_HK_AUTOMOTIVE
    if code_value in candidate_codes
}

missing_expected_codes = (
    expected_present_codes
    - accepted_codes
)

if missing_expected_codes:
    raise RuntimeError(
        "HK automotive universe regression test failed. "
        f"Expected candidate codes rejected: "
        f"{sorted(missing_expected_codes)}"
    )

print(
    "HK automotive universe regression test passed."
)

Official active-stock records: 18046
Historical-listing records: 1
Manual override records: 0
Broad listing candidates: 45
Validated Hong Kong securities: 20
Historical/delisted securities retained: 1
Validated Hong Kong issuers: 20
Resolved unique HKEX stock codes: 20
Rejected ambiguous/non-HK candidates: 25


,security_id,issuer_id,issuer_name,ticker,country,stock_code,hkex_stock_id,hkex_display_name,hkex_name_alias,listing_status,listing_start_date,listing_end_date,is_historical_listing,name_similarity_score,universe_resolution_reason
5,GAS_5720BEB2877CB1B8CA04,GAI_2943BFD9795D19C04833,GEELY AUTOMOBILE HOLDINGS LIMITED,175,HK,00175,312,GEELY AUTO,GEELY AUTO,ACTIVE,NaT,NaT,False,76.923077,ACCEPTED_NAME_ALIAS
12,GAS_8C2ADF81F06C1F2A1087,GAI_3A233B7635E27B018082,BRILLIANCE CHINA AUTOMOTIVE HOLDINGS LIMITED,1114,HK,01114,2506,BRILLIANCE CHI,<NA>,ACTIVE,NaT,NaT,False,83.333333,ACCEPTED_EXPLICIT_HK_SIGNAL
8,GAS_7BEEA4BAD5D4252C9825,GAI_F9BF134E7CCD96A4C594,BYD Co Ltd,1211,CN,01211,2696,BYD COMPANY,BYD COMPANY,ACTIVE,NaT,NaT,False,100.000000,ACCEPTED_NAME_ALIAS
3,GAS_2DE58F49778F72305AB8,GAI_A5C9694AC5A95EA3CF0B,Yadea Group Holdings Ltd,1585,CN,01585,140627,YADEA,<NA>,ACTIVE,NaT,NaT,False,100.000000,ACCEPTED_EXPLICIT_HK_SIGNAL
2,GAS_0F0802D8983EE892AD09,GAI_118982B3D7A3654914FD,Ganfeng Lithium Group Co Ltd,1772,CN,01772,191440,GANFENGLITHIUM,<NA>,ACTIVE,NaT,NaT,False,96.551724,ACCEPTED_EXPLICIT_HK_SIGNAL
19,GAS_F6F7D7DB70A9FA35B66C,GAI_342913484F832986D3FC,BAIC Motor Corp Ltd,1958,CN,01958,116122,BAIC MOTOR,<NA>,ACTIVE,NaT,NaT,False,100.000000,ACCEPTED_EXPLICIT_HK_SIGNAL
13,GAS_A2D1F31C004B5D309CB4,GAI_BDFE671E2612461D0CFC,Li Auto Inc.,2015,KY,02015,1000108505,LI AUTO-W,<NA>,ACTIVE,NaT,NaT,False,100.000000,ACCEPTED_EXPLICIT_HK_SIGNAL
15,GAS_B2C051C672FC07E5EA27,GAI_09DA7F36C135795A0916,"Guangzhou Automobile Group Co., Ltd",2238,HK,02238,49316,GAC GROUP,GAC GROUP,ACTIVE,NaT,NaT,False,17.391304,ACCEPTED_NAME_ALIAS
10,GAS_84B18AF96EAB297DAA9E,GAI_4FE2831532A093228A7A,Great Wall Motor Co Ltd,2333,CN,02333,6871,GWMOTOR,GWMOTOR,ACTIVE,NaT,NaT,False,52.173913,ACCEPTED_NAME_ALIAS
14,GAS_B09EC1A9AB2C70443E88,GAI_307F6D5D6C622C938686,Minieye Technology Co Ltd,2431,CN,02431,1000243516,MINIEYE,<NA>,ACTIVE,NaT,NaT,False,100.000000,ACCEPTED_EXPLICIT_HK_SIGNAL



Highest-scoring rejected candidates — review for false negatives:


,security_id,issuer_name,ticker,country,exchange,stock_code,hkex_display_name,name_similarity_score,meaningful_token_overlap,universe_resolution_reason
16,GAS_D9F2E731BF6EE5E8EEB9,Mazda Motor Corp,7261,JP,TSE,07261,FL2CAMNDQ100,17.391304,0,REJECTED_NAME_MISMATCH
0,GAS_034768D81B72F9474880,Nissan Motor Co Ltd,7201,JP,TSE,07201,NaN,NaN,0,REJECTED_CODE_NOT_IN_LISTING_REGISTRY
1,GAS_069EC9AC410A90F3CF02,TIANNENG POWER INTERNATIONAL LIMITED,None,HK,<NA>,<NA>,NaN,NaN,0,REJECTED_CODE_NOT_IN_LISTING_REGISTRY
2,GAS_1154405F1CBDE8821F51,Panasonic Holdings Corporation,6752,JP,TSE,06752,NaN,NaN,0,REJECTED_CODE_NOT_IN_LISTING_REGISTRY
3,GAS_13B2314857701B0279D9,Yulon Motor Co Ltd,2201,TW,TWSE,02201,NaN,NaN,0,REJECTED_CODE_NOT_IN_LISTING_REGISTRY
4,GAS_13C65F140E096987B8D9,Suzuki Motor Corp,7269,JP,TSE,07269,NaN,NaN,0,REJECTED_CODE_NOT_IN_LISTING_REGISTRY
5,GAS_14866004A8FB48938772,Jinchuan Group International Resources Co Ltd,None,HK,<NA>,<NA>,NaN,NaN,0,REJECTED_CODE_NOT_IN_LISTING_REGISTRY
6,GAS_1D88ED8C4181248BCD3E,Subaru Corp,7270,JP,TSE,07270,NaN,NaN,0,REJECTED_CODE_NOT_IN_LISTING_REGISTRY
7,GAS_20304E57A4FF15206535,Renesas Electronics Corp,6723,JP,TSE,06723,NaN,NaN,0,REJECTED_CODE_NOT_IN_LISTING_REGISTRY
8,GAS_52856086653E1412E32A,Isuzu Motors Ltd,7202,JP,TSE,07202,NaN,NaN,0,REJECTED_CODE_NOT_IN_LISTING_REGISTRY


,stock_code,expected_name,present_in_candidate_set,accepted_into_hk_universe,listing_status
0,00175,Geely,True,True,ACTIVE
1,00489,Dongfeng,True,True,DELISTED
2,01114,Brilliance,True,True,ACTIVE
3,01211,BYD,True,True,ACTIVE
4,01958,BAIC Motor,True,True,ACTIVE
5,02015,Li Auto,True,True,ACTIVE
6,02238,GAC,True,True,ACTIVE
7,02333,Great Wall Motor,True,True,ACTIVE
8,09863,Leapmotor,True,True,ACTIVE
9,09868,XPeng,True,True,ACTIVE


HK automotive universe regression test passed.


In [ ]:
# 6. ASSIGN HONG KONG ISSUER-LEVEL SOURCE PRECEDENCE

def first_present_table(table_dict, candidate_names):
    for name in candidate_names:
        dataframe = table_dict.get(name)

        if isinstance(dataframe, pd.DataFrame):
            return name, dataframe

    return None, pd.DataFrame()


def structured_issuer_coverage(
    source_system,
    table_dict,
    candidate_names,
):
    table_name, dataframe = first_present_table(
        table_dict,
        candidate_names,
    )

    output_columns = [
        "issuer_id",
        "structured_source_system",
        "structured_source_table",
        "structured_fact_rows",
        "structured_unique_concepts",
        "structured_earliest_available",
        "structured_latest_available",
    ]

    if dataframe.empty or "issuer_id" not in dataframe.columns:
        return pd.DataFrame(columns=output_columns)

    working = dataframe.copy()

    working = working[
        working["issuer_id"].notna()
    ]

    if working.empty:
        return pd.DataFrame(columns=output_columns)

    concept_column = first_existing_column(
        working,
        [
            "standard_concept",
            "canonical_concept",
        ],
    )

    availability_column = first_existing_column(
        working,
        [
            "available_datetime",
            "available_date",
            "filing_date",
        ],
    )

    grouped = working.groupby(
        "issuer_id",
        dropna=False,
    )

    coverage = grouped.size().rename(
        "structured_fact_rows"
    ).reset_index()

    if concept_column:
        concept_coverage = (
            grouped[concept_column]
            .nunique(dropna=True)
            .rename("structured_unique_concepts")
            .reset_index()
        )

        coverage = coverage.merge(
            concept_coverage,
            on="issuer_id",
            how="left",
            validate="1:1",
        )
    else:
        coverage["structured_unique_concepts"] = 0

    if availability_column:
        dates = pd.to_datetime(
            working[availability_column],
            errors="coerce",
            utc=True,
        )

        working = working.assign(
            _structured_available_datetime=dates
        )

        date_coverage = (
            working.groupby(
                "issuer_id",
                dropna=False,
            )["_structured_available_datetime"]
            .agg(
                structured_earliest_available="min",
                structured_latest_available="max",
            )
            .reset_index()
        )

        coverage = coverage.merge(
            date_coverage,
            on="issuer_id",
            how="left",
            validate="1:1",
        )
    else:
        coverage["structured_earliest_available"] = pd.NaT
        coverage["structured_latest_available"] = pd.NaT

    coverage["structured_source_system"] = source_system
    coverage["structured_source_table"] = table_name

    return coverage[output_columns]


structured_coverage_frames = [
    structured_issuer_coverage(
        "SEC",
        block_3_inputs,
        [
            "sec_fundamentals_security_linked_df",
            "sec_fundamentals_standardised_df",
        ],
    ),
    structured_issuer_coverage(
        "EDINET",
        block_5_inputs,
        [
            "japan_fundamentals_standardised_df",
        ],
    ),
    structured_issuer_coverage(
        "DART",
        block_6_inputs,
        [
            "korea_fundamentals_standardised_df",
        ],
    ),
]

upstream_structured_issuer_coverage_df = pd.concat(
    [
        frame
        for frame in structured_coverage_frames
        if not frame.empty
    ],
    ignore_index=True,
) if any(
    not frame.empty
    for frame in structured_coverage_frames
) else pd.DataFrame(
    columns=[
        "issuer_id",
        "structured_source_system",
        "structured_source_table",
        "structured_fact_rows",
        "structured_unique_concepts",
        "structured_earliest_available",
        "structured_latest_available",
    ]
)

# One issuer may be covered in more than one structured system. The precedence
# rank is deterministic and can be changed centrally if required.
STRUCTURED_SOURCE_PRECEDENCE = {
    "SEC": 1,
    "EDINET": 2,
    "DART": 3,
}

if upstream_structured_issuer_coverage_df.empty:
    preferred_structured_issuer_source_df = pd.DataFrame(
        columns=[
            "issuer_id",
            "structured_source_system",
            "structured_source_table",
            "structured_fact_rows",
            "structured_unique_concepts",
            "structured_earliest_available",
            "structured_latest_available",
            "structured_source_precedence",
        ]
    )
else:
    upstream_structured_issuer_coverage_df[
        "structured_source_precedence"
    ] = (
        upstream_structured_issuer_coverage_df[
            "structured_source_system"
        ]
        .map(STRUCTURED_SOURCE_PRECEDENCE)
        .fillna(99)
        .astype(int)
    )

    preferred_structured_issuer_source_df = (
        upstream_structured_issuer_coverage_df
        .sort_values(
            [
                "issuer_id",
                "structured_source_precedence",
                "structured_fact_rows",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
        .drop_duplicates("issuer_id", keep="first")
        .reset_index(drop=True)
    )

hong_kong_security_universe_df = (
    hong_kong_security_universe_df.merge(
        preferred_structured_issuer_source_df,
        on="issuer_id",
        how="left",
        validate="m:1",
    )
)

structured_fact_rows = pd.to_numeric(
    hong_kong_security_universe_df[
        "structured_fact_rows"
    ],
    errors="coerce",
).fillna(0)

structured_unique_concepts = pd.to_numeric(
    hong_kong_security_universe_df[
        "structured_unique_concepts"
    ],
    errors="coerce",
).fillna(0)

hong_kong_security_universe_df[
    "has_upstream_structured_fundamentals"
] = structured_fact_rows.gt(0)

hong_kong_security_universe_df[
    "has_usable_upstream_structured_fundamentals"
] = (
    structured_fact_rows.ge(
        MIN_STRUCTURED_FACT_ROWS
    )
    & structured_unique_concepts.ge(
        MIN_STRUCTURED_UNIQUE_CONCEPTS
    )
)

hong_kong_security_universe_df[
    "upstream_coverage_quality"
] = np.select(
    [
        hong_kong_security_universe_df[
            "has_usable_upstream_structured_fundamentals"
        ],
        hong_kong_security_universe_df[
            "has_upstream_structured_fundamentals"
        ],
    ],
    [
        "USABLE",
        "INSUFFICIENT",
    ],
    default="MISSING",
)

hong_kong_security_universe_df[
    "fundamental_ingestion_role"
] = np.where(
    SUPPRESS_DUPLICATE_STRUCTURED_FUNDAMENTALS
    & hong_kong_security_universe_df[
        "has_usable_upstream_structured_fundamentals"
    ],
    "SECONDARY_LISTING_ONLY",
    "PRIMARY_ISSUER_SOURCE",
)

hong_kong_security_universe_df[
    "include_in_hkex_fundamental_ingestion"
] = (
    hong_kong_security_universe_df[
        "fundamental_ingestion_role"
    ].eq("PRIMARY_ISSUER_SOURCE")
)

hong_kong_fundamental_ingestion_universe_df = (
    hong_kong_security_universe_df[
        hong_kong_security_universe_df[
            "include_in_hkex_fundamental_ingestion"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

hong_kong_secondary_listing_only_df = (
    hong_kong_security_universe_df[
        ~hong_kong_security_universe_df[
            "include_in_hkex_fundamental_ingestion"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

# Rebuild the resolved/unresolved subsets after adding source-precedence fields.
hong_kong_resolved_securities_df = (
    hong_kong_security_universe_df[
        hong_kong_security_universe_df[
            "hkex_stock_id"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

hong_kong_unresolved_securities_df = (
    hong_kong_security_universe_df[
        hong_kong_security_universe_df[
            "hkex_stock_id"
        ].isna()
    ]
    .copy()
    .reset_index(drop=True)
)

hong_kong_fundamental_source_precedence_df = (
    hong_kong_security_universe_df[
        [
            "issuer_id",
            "issuer_name",
            "security_id",
            "stock_code",
            "listing_status",
            "structured_source_system",
            "structured_source_table",
            "structured_fact_rows",
            "structured_unique_concepts",
            "has_upstream_structured_fundamentals",
            "has_usable_upstream_structured_fundamentals",
            "upstream_coverage_quality",
            "fundamental_ingestion_role",
            "include_in_hkex_fundamental_ingestion",
        ]
    ]
    .copy()
    .sort_values(
        [
            "fundamental_ingestion_role",
            "issuer_name",
            "stock_code",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Validated Hong Kong securities:",
    len(hong_kong_security_universe_df),
)

print(
    "HKEX primary fundamental-ingestion securities:",
    len(hong_kong_fundamental_ingestion_universe_df),
)

print(
    "Secondary listings using upstream structured fundamentals:",
    len(hong_kong_secondary_listing_only_df),
)

print(
    "Minimum upstream coverage required:",
    f"{MIN_STRUCTURED_FACT_ROWS} facts and "
    f"{MIN_STRUCTURED_UNIQUE_CONCEPTS} concepts",
)

print(
    "Issuers with insufficient upstream coverage retained for HKEX parsing:",
    int(
        hong_kong_security_universe_df[
            "upstream_coverage_quality"
        ].eq("INSUFFICIENT").sum()
    ),
)

display(
    hong_kong_fundamental_source_precedence_df
)

Validated Hong Kong securities: 20
HKEX primary fundamental-ingestion securities: 18
Secondary listings using upstream structured fundamentals: 2
Minimum upstream coverage required: 20 facts and 5 concepts
Issuers with insufficient upstream coverage retained for HKEX parsing: 0


,issuer_id,issuer_name,security_id,stock_code,listing_status,structured_source_system,structured_source_table,structured_fact_rows,structured_unique_concepts,has_upstream_structured_fundamentals,has_usable_upstream_structured_fundamentals,upstream_coverage_quality,fundamental_ingestion_role,include_in_hkex_fundamental_ingestion
0,GAI_342913484F832986D3FC,BAIC Motor Corp Ltd,GAS_F6F7D7DB70A9FA35B66C,01958,ACTIVE,NaN,NaN,NaN,NaN,False,False,MISSING,PRIMARY_ISSUER_SOURCE,True
1,GAI_3A233B7635E27B018082,BRILLIANCE CHINA AUTOMOTIVE HOLDINGS LIMITED,GAS_8C2ADF81F06C1F2A1087,01114,ACTIVE,NaN,NaN,NaN,NaN,False,False,MISSING,PRIMARY_ISSUER_SOURCE,True
2,GAI_F9BF134E7CCD96A4C594,BYD Co Ltd,GAS_7BEEA4BAD5D4252C9825,01211,ACTIVE,NaN,NaN,NaN,NaN,False,False,MISSING,PRIMARY_ISSUER_SOURCE,True
3,GAI_DFA494ABBAA0C6DE156F,Black Sesame International Hol,GAS_025E74C831EDA3481EFF,02533,ACTIVE,NaN,NaN,NaN,NaN,False,False,MISSING,PRIMARY_ISSUER_SOURCE,True
4,GAI_8D9BD38147455E2BD450,"CALB Group Co., Ltd.",GAS_69D90263C54B3B6679DE,03931,ACTIVE,NaN,NaN,NaN,NaN,False,False,MISSING,PRIMARY_ISSUER_SOURCE,True
5,GAI_A0E69150AAC34423F72C,CATL,GAS_33D68E0D48F42C33BF01,03750,ACTIVE,NaN,NaN,NaN,NaN,False,False,MISSING,PRIMARY_ISSUER_SOURCE,True
6,GAI_DAFED20C2A3AA91FFD2C,Dongfeng Motor Group Co Ltd,GAS_839B1CD6F82F16F4381B,00489,DELISTED,NaN,NaN,NaN,NaN,False,False,MISSING,PRIMARY_ISSUER_SOURCE,True
7,GAI_2943BFD9795D19C04833,GEELY AUTOMOBILE HOLDINGS LIMITED,GAS_5720BEB2877CB1B8CA04,00175,ACTIVE,NaN,NaN,NaN,NaN,False,False,MISSING,PRIMARY_ISSUER_SOURCE,True
8,GAI_118982B3D7A3654914FD,Ganfeng Lithium Group Co Ltd,GAS_0F0802D8983EE892AD09,01772,ACTIVE,NaN,NaN,NaN,NaN,False,False,MISSING,PRIMARY_ISSUER_SOURCE,True
9,GAI_4FE2831532A093228A7A,Great Wall Motor Co Ltd,GAS_84B18AF96EAB297DAA9E,02333,ACTIVE,NaN,NaN,NaN,NaN,False,False,MISSING,PRIMARY_ISSUER_SOURCE,True


In [ ]:
# 7. DISCOVER HKEXNEWS FILINGS

def parse_hk_release_datetime(text):
    cleaned = re.sub(r"\s+", " ", str(text)).strip()

    patterns = [
        (r"(\d{2}/\d{2}/\d{4})\s+(\d{2}:\d{2})", "%d/%m/%Y %H:%M"),
        (r"(\d{4}/\d{2}/\d{2})\s+(\d{2}:\d{2})", "%Y/%m/%d %H:%M"),
    ]

    for pattern, date_format in patterns:
        match = re.search(pattern, cleaned)

        if not match:
            continue

        value = pd.to_datetime(
            f"{match.group(1)} {match.group(2)}",
            format=date_format,
            errors="coerce",
        )

        if pd.notna(value):
            return (
                value
                .tz_localize("Asia/Hong_Kong")
                .tz_convert("UTC")
            )

    return pd.NaT


def clean_html_text(value):
    return re.sub(
        r"\s+",
        " ",
        BeautifulSoup(
            str(value),
            "lxml",
        ).get_text(" ", strip=True),
    ).strip()


def parse_hkex_result_rows(search_html, requested_code, stock_id):
    results = []
    soup = BeautifulSoup(search_html, "lxml")

    # Current HKEX result rows use these semantic classes.
    candidate_rows = soup.find_all("tr")

    for row in candidate_rows:
        document_container = row.select_one(".doc-link")

        if document_container is None:
            continue

        anchor = document_container.find("a", href=True)

        if anchor is None:
            continue

        href = anchor.get("href", "").strip()

        if not href:
            continue

        document_url = (
            href
            if href.startswith("http")
            else urljoin("https://www.hkexnews.hk/", href)
        )

        if not re.search(
            r"\.(pdf|htm|html)(?:\?|$)",
            document_url,
            flags=re.IGNORECASE,
        ):
            continue

        title = clean_html_text(anchor.get_text(" ", strip=True))

        headline_container = row.select_one(".headline")
        category_name = (
            clean_html_text(headline_container.get_text(" ", strip=True))
            if headline_container is not None
            else ""
        )

        release_container = row.select_one(".release-time")
        release_text = (
            clean_html_text(release_container.get_text(" ", strip=True))
            if release_container is not None
            else clean_html_text(row.get_text(" ", strip=True))
        )

        row_text = clean_html_text(row.get_text(" ", strip=True))
        release_datetime = parse_hk_release_datetime(row_text)

        stock_code_match = re.search(
            r"Stock Code:\s*(\d{1,5})",
            row_text,
            flags=re.IGNORECASE,
        )

        stock_code_reported = (
            stock_code_match.group(1).zfill(5)
            if stock_code_match
            else requested_code
        )

        combined_text = " ".join(
            [category_name, title, row_text]
        )

        if not TARGET_TITLE_PATTERN.search(combined_text):
            continue

        is_results = bool(
            RESULT_ANNOUNCEMENT_PATTERN.search(combined_text)
        )

        is_long_form = bool(
            LONG_FORM_REPORT_PATTERN.search(combined_text)
        )

        if is_results and not INCLUDE_RESULT_ANNOUNCEMENTS:
            continue

        if is_long_form and not INCLUDE_LONG_FORM_REPORTS:
            continue

        announcement_match = re.search(
            r"/(\d{10,})\.(?:pdf|htm|html)",
            document_url,
            flags=re.IGNORECASE,
        )

        announcement_id = (
            announcement_match.group(1)
            if announcement_match
            else hashlib.sha256(
                document_url.encode()
            ).hexdigest()[:24]
        )

        results.append({
            "announcement_id": announcement_id,
            "stock_code": requested_code,
            "stock_code_reported": stock_code_reported,
            "hkex_stock_id": str(stock_id),
            "release_datetime": release_datetime,
            "release_date": (
                release_datetime.normalize()
                if pd.notna(release_datetime)
                else pd.NaT
            ),
            "category_name": category_name,
            "document_title": title,
            "document_url": document_url,
            "document_format": (
                Path(document_url.split("?")[0])
                .suffix.lower()
                .lstrip(".")
            ),
            "is_result_announcement": is_results,
            "is_long_form_report": is_long_form,
            "search_row_text": row_text,
        })

    return pd.DataFrame(results)


def title_search_payload(stock_id, page_number):
    return {
        "stockId": str(stock_id),
        "sortDir": "desc",
        "sortByOptions": "DateTime",
        "market": "SEHK",
        "language": SEARCH_LANGUAGE,
        "category": "0",
        "from": pd.Timestamp(DISCOVERY_START_DATE).strftime("%Y%m%d"),
        "to": pd.Timestamp(DISCOVERY_END_DATE).strftime("%Y%m%d"),
        "page": str(page_number),
    }


search_headers = {
    "User-Agent": session.headers["User-Agent"],
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,*/*;q=0.8"
    ),
    "Accept-Language": session.headers["Accept-Language"],
    "Referer": HKEX_TITLE_SEARCH_URL,
    "Origin": HKEX_BASE,
}

search_frames = []
search_logs = []

search_universe = (
    hong_kong_fundamental_ingestion_universe_df[
        [
            "stock_code",
            "hkex_stock_id",
            "listing_start_date",
            "listing_end_date",
            "listing_status",
            "is_historical_listing",
            "issuer_id",
            "security_id",
            "fundamental_ingestion_role",
        ]
    ]
    .dropna(subset=["hkex_stock_id"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "HKEX securities searched for accounting disclosures:",
    len(search_universe),
)

print(
    "Genuine HKEX secondary listings retained but not reparsed:",
    len(hong_kong_secondary_listing_only_df),
)

for security_row in tqdm(
    search_universe.itertuples(index=False),
    total=len(search_universe),
    desc="Searching HKEXnews disclosures",
):
    total_security_rows = 0

    for page_number in range(1, MAX_SEARCH_PAGES + 1):
        payload = title_search_payload(
            security_row.hkex_stock_id,
            page_number,
        )

        # Restrict historical listings to their actual listing window.
        if pd.notna(security_row.listing_start_date):
            payload["from"] = max(
                pd.Timestamp(DISCOVERY_START_DATE),
                pd.Timestamp(security_row.listing_start_date),
            ).strftime("%Y%m%d")

        if pd.notna(security_row.listing_end_date):
            payload["to"] = min(
                pd.Timestamp(DISCOVERY_END_DATE),
                pd.Timestamp(security_row.listing_end_date),
            ).strftime("%Y%m%d")

        try:
            search_html, request_log = cached_post_text(
                HKEX_TITLE_SEARCH_URL,
                data=payload,
                cache_dir=HKEX_SEARCH_CACHE_DIR,
                headers=search_headers,
            )

            page_df = parse_hkex_result_rows(
                search_html,
                security_row.stock_code,
                security_row.hkex_stock_id,
            )

            if not page_df.empty:
                page_df["listing_start_date"] = (
                    security_row.listing_start_date
                )
                page_df["listing_end_date"] = (
                    security_row.listing_end_date
                )
                page_df["listing_status"] = (
                    security_row.listing_status
                )
                page_df["is_historical_listing"] = (
                    security_row.is_historical_listing
                )
                page_df["issuer_id"] = security_row.issuer_id
                page_df["security_id"] = security_row.security_id
                page_df["fundamental_ingestion_role"] = (
                    security_row.fundamental_ingestion_role
                )
                search_frames.append(page_df)
                total_security_rows += len(page_df)

            search_logs.append({
                "stock_code": security_row.stock_code,
                "hkex_stock_id": security_row.hkex_stock_id,
                "page_number": page_number,
                "status": request_log["status"],
                "http_status": request_log["http_status"],
                "target_rows": len(page_df),
                "cache_path": request_log["cache_path"],
                "error": pd.NA,
            })

            # HKEX pages normally contain around 100 result rows. If no document
            # rows are present on a later page, pagination is complete.
            soup = BeautifulSoup(search_html, "lxml")
            raw_doc_links = soup.select(".doc-link a[href]")

            if not raw_doc_links:
                break

            if len(raw_doc_links) < 100:
                break

        except Exception as exc:
            search_logs.append({
                "stock_code": security_row.stock_code,
                "hkex_stock_id": security_row.hkex_stock_id,
                "page_number": page_number,
                "status": "FAILED",
                "http_status": pd.NA,
                "target_rows": 0,
                "cache_path": pd.NA,
                "error": repr(exc),
            })
            break


hkex_disclosures_discovered_df = (
    pd.concat(
        search_frames,
        ignore_index=True,
    )
    .drop_duplicates(
        [
            "stock_code",
            "release_datetime",
            "document_url",
        ]
    )
    .reset_index(drop=True)
    if search_frames
    else pd.DataFrame(
        columns=[
            "announcement_id",
            "stock_code",
            "stock_code_reported",
            "hkex_stock_id",
            "release_datetime",
            "release_date",
            "category_name",
            "document_title",
            "document_url",
            "document_format",
            "is_result_announcement",
            "is_long_form_report",
            "search_row_text",
            "listing_start_date",
            "listing_end_date",
            "listing_status",
            "is_historical_listing",
        ]
    )
)

hkex_disclosure_search_log_df = pd.DataFrame(search_logs)

print(
    "Target HKEX disclosures discovered:",
    len(hkex_disclosures_discovered_df),
)

print(
    "Issuers with at least one target disclosure:",
    hkex_disclosures_discovered_df["stock_code"].nunique()
    if not hkex_disclosures_discovered_df.empty
    else 0,
)

display(
    hkex_disclosure_search_log_df.head(100)
)

display(
    hkex_disclosures_discovered_df.head(100)
)

if hong_kong_resolved_securities_df["stock_code"].eq("01211").any():
    byd_results = hkex_disclosures_discovered_df[
        hkex_disclosures_discovered_df["stock_code"].eq("01211")
    ]

    print("BYD target disclosures discovered:", len(byd_results))

    if byd_results.empty:
        raise RuntimeError(
            "BYD resolved successfully but the HKEX title-search POST returned "
            "no target disclosures. Inspect the cached HTML and search log "
            "before continuing."
        )

HKEX securities searched for accounting disclosures: 18
Genuine HKEX secondary listings retained but not reparsed: 2


Searching HKEXnews disclosures:   0%|          | 0/18 [00:00<?, ?it/s]

Target HKEX disclosures discovered: 90
Issuers with at least one target disclosure: 16


,stock_code,hkex_stock_id,page_number,status,http_status,target_rows,cache_path,error
0,02533,1000221013,1,DOWNLOADED,200,9,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,<NA>
1,02533,1000221013,2,DOWNLOADED,200,9,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,<NA>
2,02533,1000221013,3,DOWNLOADED,200,9,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,<NA>
3,02533,1000221013,4,DOWNLOADED,200,9,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,<NA>
4,02533,1000221013,5,DOWNLOADED,200,9,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,<NA>
5,02533,1000221013,6,DOWNLOADED,200,9,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,<NA>
6,02533,1000221013,7,DOWNLOADED,200,9,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,<NA>
7,02533,1000221013,8,DOWNLOADED,200,9,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,<NA>
8,02533,1000221013,9,DOWNLOADED,200,9,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,<NA>
9,02533,1000221013,10,DOWNLOADED,200,9,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,<NA>


,announcement_id,stock_code,stock_code_reported,hkex_stock_id,release_datetime,release_date,category_name,document_title,document_url,document_format,is_result_announcement,is_long_form_report,search_row_text,listing_start_date,listing_end_date,listing_status,is_historical_listing,issuer_id,security_id,fundamental_ingestion_role
0,2024082801875,02533,02533,1000221013,2024-08-28 13:32:00+00:00,2024-08-28 00:00:00+00:00,Announcements and Notices - [Interim Results],INTERIM RESULTS ANNOUNCEMENT FOR THE SIX MONTH...,https://www.hkexnews.hk/listedco/listconews/se...,pdf,True,False,Release Time: 28/08/2024 21:32 Stock Code: 025...,NaT,NaT,ACTIVE,False,GAI_DFA494ABBAA0C6DE156F,GAS_025E74C831EDA3481EFF,PRIMARY_ISSUER_SOURCE
1,2024092700749,02533,02533,1000221013,2024-09-27 08:37:00+00:00,2024-09-27 00:00:00+00:00,Financial Statements/ESG Information - [Interi...,2024 INTERIM REPORT,https://www.hkexnews.hk/listedco/listconews/se...,pdf,False,True,Release Time: 27/09/2024 16:37 Stock Code: 025...,NaT,NaT,ACTIVE,False,GAI_DFA494ABBAA0C6DE156F,GAS_025E74C831EDA3481EFF,PRIMARY_ISSUER_SOURCE
2,2025033100097,02533,02533,1000221013,2025-03-30 23:30:00+00:00,2025-03-30 00:00:00+00:00,Announcements and Notices - [Final Results],ANNUAL RESULTS ANNOUNCEMENT FOR THE YEAR ENDED...,https://www.hkexnews.hk/listedco/listconews/se...,pdf,True,False,Release Time: 31/03/2025 07:30 Stock Code: 025...,NaT,NaT,ACTIVE,False,GAI_DFA494ABBAA0C6DE156F,GAS_025E74C831EDA3481EFF,PRIMARY_ISSUER_SOURCE
3,2025042501583,02533,02533,1000221013,2025-04-25 09:13:00+00:00,2025-04-25 00:00:00+00:00,Financial Statements/ESG Information - [Annual...,2024 ANNUAL REPORT,https://www.hkexnews.hk/listedco/listconews/se...,pdf,False,True,Release Time: 25/04/2025 17:13 Stock Code: 025...,NaT,NaT,ACTIVE,False,GAI_DFA494ABBAA0C6DE156F,GAS_025E74C831EDA3481EFF,PRIMARY_ISSUER_SOURCE
4,2025082900315,02533,02533,1000221013,2025-08-29 00:20:00+00:00,2025-08-29 00:00:00+00:00,Announcements and Notices - [Interim Results],INTERIM RESULTS ANNOUNCEMENT FOR THE SIX MONTH...,https://www.hkexnews.hk/listedco/listconews/se...,pdf,True,False,Release Time: 29/08/2025 08:20 Stock Code: 025...,NaT,NaT,ACTIVE,False,GAI_DFA494ABBAA0C6DE156F,GAS_025E74C831EDA3481EFF,PRIMARY_ISSUER_SOURCE
5,2025091101246,02533,02533,1000221013,2025-09-11 12:00:00+00:00,2025-09-11 00:00:00+00:00,Financial Statements/ESG Information - [Interi...,2025 INTERIM REPORT,https://www.hkexnews.hk/listedco/listconews/se...,pdf,False,True,Release Time: 11/09/2025 20:00 Stock Code: 025...,NaT,NaT,ACTIVE,False,GAI_DFA494ABBAA0C6DE156F,GAS_025E74C831EDA3481EFF,PRIMARY_ISSUER_SOURCE
6,2025093002398,02533,02533,1000221013,2025-09-30 12:00:00+00:00,2025-09-30 00:00:00+00:00,Announcements and Notices - [Final Results / O...,SUPPLEMENTAL ANNOUNCEMENT IN RELATION TO THE A...,https://www.hkexnews.hk/listedco/listconews/se...,pdf,True,True,Release Time: 30/09/2025 20:00 Stock Code: 025...,NaT,NaT,ACTIVE,False,GAI_DFA494ABBAA0C6DE156F,GAS_025E74C831EDA3481EFF,PRIMARY_ISSUER_SOURCE
7,2026033100746,02533,02533,1000221013,2026-03-31 00:10:00+00:00,2026-03-31 00:00:00+00:00,Announcements and Notices - [Final Results],ANNUAL RESULTS ANNOUNCEMENT FOR THE YEAR ENDED...,https://www.hkexnews.hk/listedco/listconews/se...,pdf,True,False,Release Time: 31/03/2026 08:10 Stock Code: 025...,NaT,NaT,ACTIVE,False,GAI_DFA494ABBAA0C6DE156F,GAS_025E74C831EDA3481EFF,PRIMARY_ISSUER_SOURCE
8,2026042701016,02533,02533,1000221013,2026-04-27 08:48:00+00:00,2026-04-27 00:00:00+00:00,Financial Statements/ESG Information - [Annual...,2025 ANNUAL REPORT,https://www.hkexnews.hk/listedco/listconews/se...,pdf,False,True,Release Time: 27/04/2026 16:48 Stock Code: 025...,NaT,NaT,ACTIVE,False,GAI_DFA494ABBAA0C6DE156F,GAS_025E74C831EDA3481EFF,PRIMARY_ISSUER_SOURCE
9,2020033002457,01772,01772,191440,2020-03-30 14:14:00+00:00,2020-03-30 00:00:00+00:00,Announcements and Notices - [Final Results],UNAUDITED ANNUAL RESULTS ANNOUNCEMENT FOR THE ...,https://www.hkexnews.hk/listedco/listcon

BYD target disclosures discovered: 4


In [ ]:
# 8. LINK HKEX DISCLOSURES TO GLOBAL IDS

FILING_METADATA_COLUMNS = [
    "announcement_id",
    "document_id",
    "stock_code",
    "hkex_stock_id",
    "security_id",
    "issuer_id",
    "issuer_name",
    "ticker",
    "release_datetime",
    "release_date",
    "available_datetime",
    "available_date",
    "availability_basis",
    "source_system",
    "category_name",
    "document_title",
    "document_url",
    "document_format",
    "is_result_announcement",
    "is_long_form_report",
    "document_priority",
    "listing_start_date",
    "listing_end_date",
    "listing_status",
    "is_historical_listing",
    "fundamental_ingestion_role",
]

if hkex_disclosures_discovered_df.empty:
    hong_kong_filing_metadata_df = pd.DataFrame(
        columns=FILING_METADATA_COLUMNS
    )

else:
    disclosures = (
        hkex_disclosures_discovered_df
        .copy()
        .reset_index(drop=True)
    )

    merge_keys = [
        "stock_code",
        "hkex_stock_id",
    ]

    bridge_candidate_columns = [
        "stock_code",
        "hkex_stock_id",
        "security_id",
        "issuer_id",
        "issuer_name",
        "ticker",
        "listing_start_date",
        "listing_end_date",
        "listing_status",
        "is_historical_listing",
        "fundamental_ingestion_role",
    ]

    bridge_available_columns = [
        column
        for column in bridge_candidate_columns
        if column in hong_kong_resolved_securities_df.columns
    ]

    missing_merge_keys = [
        column
        for column in merge_keys
        if column not in bridge_available_columns
    ]

    if missing_merge_keys:
        raise RuntimeError(
            "Resolved HKEX security table is missing merge keys: "
            f"{missing_merge_keys}"
        )

    bridge = (
        hong_kong_resolved_securities_df[
            bridge_available_columns
        ]
        .drop_duplicates(
            subset=merge_keys,
            keep="first",
        )
        .reset_index(drop=True)
    )

    # The search stage may already carry security and listing fields.
    # Add only fields that are absent, preventing _x / _y suffixes.
    bridge_payload_columns = (
        merge_keys
        + [
            column
            for column in bridge.columns
            if column not in merge_keys
            and column not in disclosures.columns
        ]
    )

    bridge_payload = bridge[
        bridge_payload_columns
    ].copy()

    hong_kong_filing_metadata_df = (
        disclosures.merge(
            bridge_payload,
            on=merge_keys,
            how="left",
            validate="m:1",
        )
    )

    # Guarantee the stable schema expected by extraction and mapping.
    column_defaults = {
        "security_id": pd.NA,
        "issuer_id": pd.NA,
        "issuer_name": pd.NA,
        "ticker": pd.NA,
        "listing_start_date": pd.NaT,
        "listing_end_date": pd.NaT,
        "listing_status": pd.NA,
        "is_historical_listing": False,
        "fundamental_ingestion_role": (
            "PRIMARY_ISSUER_SOURCE"
        ),
    }

    for column, default_value in column_defaults.items():
        if column not in hong_kong_filing_metadata_df.columns:
            hong_kong_filing_metadata_df[
                column
            ] = default_value

    # Resolve any suffix columns left by an upstream cached table.
    coalesce_columns = [
        "security_id",
        "issuer_id",
        "issuer_name",
        "ticker",
        "listing_start_date",
        "listing_end_date",
        "listing_status",
        "is_historical_listing",
        "fundamental_ingestion_role",
    ]

    for column in coalesce_columns:
        suffix_candidates = [
            candidate
            for candidate in [
                f"{column}_x",
                f"{column}_y",
            ]
            if candidate
            in hong_kong_filing_metadata_df.columns
        ]

        if suffix_candidates:
            combined = (
                hong_kong_filing_metadata_df[column]
                if column
                in hong_kong_filing_metadata_df.columns
                else pd.Series(
                    pd.NA,
                    index=hong_kong_filing_metadata_df.index,
                )
            )

            for suffix_column in suffix_candidates:
                combined = combined.combine_first(
                    hong_kong_filing_metadata_df[
                        suffix_column
                    ]
                )

            hong_kong_filing_metadata_df[
                column
            ] = combined

            hong_kong_filing_metadata_df = (
                hong_kong_filing_metadata_df.drop(
                    columns=suffix_candidates,
                    errors="ignore",
                )
            )

    hong_kong_filing_metadata_df[
        "available_datetime"
    ] = pd.to_datetime(
        hong_kong_filing_metadata_df[
            "release_datetime"
        ],
        errors="coerce",
        utc=True,
    )

    hong_kong_filing_metadata_df[
        "available_date"
    ] = (
        hong_kong_filing_metadata_df[
            "available_datetime"
        ]
        .dt.normalize()
    )

    hong_kong_filing_metadata_df[
        "availability_basis"
    ] = "HKEXNEWS_RELEASE_TIMESTAMP"

    hong_kong_filing_metadata_df[
        "source_system"
    ] = "HKEXNEWS"

    hong_kong_filing_metadata_df[
        "document_id"
    ] = (
        hong_kong_filing_metadata_df[
            "announcement_id"
        ]
        .astype("string")
    )

    result_announcement_mask = (
        hong_kong_filing_metadata_df[
            "is_result_announcement"
        ]
        .fillna(False)
        .astype(bool)
    )

    long_form_report_mask = (
        hong_kong_filing_metadata_df[
            "is_long_form_report"
        ]
        .fillna(False)
        .astype(bool)
    )

    hong_kong_filing_metadata_df[
        "document_priority"
    ] = np.select(
        [
            result_announcement_mask,
            long_form_report_mask,
        ],
        [
            1,
            2,
        ],
        default=3,
    )

    duplicate_subset = [
        column
        for column in [
            "security_id",
            "announcement_id",
            "document_url",
        ]
        if column
        in hong_kong_filing_metadata_df.columns
    ]

    if duplicate_subset:
        hong_kong_filing_metadata_df = (
            hong_kong_filing_metadata_df
            .drop_duplicates(
                subset=duplicate_subset,
                keep="first",
            )
            .reset_index(drop=True)
        )

    for column in FILING_METADATA_COLUMNS:
        if column not in hong_kong_filing_metadata_df.columns:
            hong_kong_filing_metadata_df[
                column
            ] = pd.NA

    hong_kong_filing_metadata_df = (
        hong_kong_filing_metadata_df[
            FILING_METADATA_COLUMNS
        ]
        .copy()
    )


print(
    "Linked HKEX filing rows:",
    len(hong_kong_filing_metadata_df),
)

print(
    "Security link rate:",
    (
        hong_kong_filing_metadata_df[
            "security_id"
        ]
        .notna()
        .mean()
        if not hong_kong_filing_metadata_df.empty
        else np.nan
    ),
)

print(
    "Primary HKEX fundamental-source rows:",
    (
        hong_kong_filing_metadata_df[
            "fundamental_ingestion_role"
        ]
        .eq("PRIMARY_ISSUER_SOURCE")
        .sum()
        if not hong_kong_filing_metadata_df.empty
        else 0
    ),
)

Linked HKEX filing rows: 90
Security link rate: 1.0
Primary HKEX fundamental-source rows: 90


In [ ]:
# 9. DOWNLOAD HKEX DOCUMENTS AND EXTRACT TEXT

def local_document_path(document_id, document_format):
    suffix = "." + (document_format or "bin").lower().lstrip(".")
    return HKEX_DOCUMENT_CACHE_DIR / f"{document_id}{suffix}"


def download_hkex_document(row):
    path = local_document_path(
        row.document_id,
        row.document_format,
    )

    if path.exists():
        return path, {
            "document_id": row.document_id,
            "document_url": row.document_url,
            "status": "CACHE_HIT",
            "http_status": 200,
            "cache_path": str(path),
            "attempt": 0,
        }

    response, attempt = request_with_retries(
        "GET",
        row.document_url,
        headers={
            "User-Agent": session.headers["User-Agent"],
            "Referer": HKEX_TITLE_SEARCH_URL,
        },
    )

    path.write_bytes(response.content)

    return path, {
        "document_id": row.document_id,
        "document_url": row.document_url,
        "status": "DOWNLOADED",
        "http_status": response.status_code,
        "cache_path": str(path),
        "attempt": attempt,
    }


def extract_pdf_text_pymupdf(path):
    document = fitz.open(str(path))
    page_limit = len(document)

    if MAX_PDF_PAGES is not None:
        page_limit = min(page_limit, int(MAX_PDF_PAGES))

    pages = []

    for page_number in range(page_limit):
        try:
            pages.append(document[page_number].get_text("text") or "")
        except Exception:
            pages.append("")

    document.close()

    return "\n\n".join(pages), page_limit


def extract_pdf_text_pypdf(path):
    reader = PdfReader(str(path))
    page_limit = len(reader.pages)

    if MAX_PDF_PAGES is not None:
        page_limit = min(page_limit, int(MAX_PDF_PAGES))

    pages = []

    for page_number in range(page_limit):
        try:
            pages.append(reader.pages[page_number].extract_text() or "")
        except Exception:
            pages.append("")

    return "\n\n".join(pages), page_limit


def extract_document_text(document_id, path, document_format):
    text_path = HKEX_TEXT_CACHE_DIR / f"{document_id}.txt"

    if text_path.exists():
        return (
            text_path.read_text(
                encoding="utf-8",
                errors="replace",
            ),
            {
                "document_id": document_id,
                "status": "CACHE_HIT",
                "text_cache_path": str(text_path),
                "extraction_method": "CACHED_TEXT",
            },
        )

    document_format = str(document_format).lower()

    if document_format == "pdf":
        text, pages_extracted = extract_pdf_text_pymupdf(path)
        method = "PYMUPDF"

        if len(text.strip()) < 100:
            fallback_text, fallback_pages = extract_pdf_text_pypdf(path)

            if len(fallback_text.strip()) > len(text.strip()):
                text = fallback_text
                pages_extracted = fallback_pages
                method = "PYPDF_FALLBACK"

    else:
        soup = BeautifulSoup(
            path.read_bytes(),
            "lxml",
        )

        text = "\n".join(
            line.strip()
            for line in soup.get_text("\n").splitlines()
            if line.strip()
        )

        pages_extracted = pd.NA
        method = "HTML_TEXT"

    text_path.write_text(text, encoding="utf-8")

    return text, {
        "document_id": document_id,
        "status": "EXTRACTED",
        "text_cache_path": str(text_path),
        "extraction_method": method,
        "pages_extracted": pages_extracted,
        "character_count": len(text),
    }


download_rows = []
text_rows = []

documents_to_download = (
    hong_kong_filing_metadata_df
    .sort_values(
        ["document_priority", "available_datetime"],
        ascending=[True, True],
    )
    .drop_duplicates("document_id")
    .copy()
)

if MAX_DOCUMENTS_TO_DOWNLOAD is not None:
    documents_to_download = documents_to_download.head(
        int(MAX_DOCUMENTS_TO_DOWNLOAD)
    )

for row in tqdm(
    documents_to_download.itertuples(index=False),
    total=len(documents_to_download),
    desc="Downloading HKEX documents",
):
    try:
        path, download_log = download_hkex_document(row)
        download_rows.append(download_log)

        text, extraction_log = extract_document_text(
            row.document_id,
            path,
            row.document_format,
        )

        text_rows.append({
            "document_id": row.document_id,
            "document_text": text,
            **extraction_log,
        })

    except Exception as exc:
        download_rows.append({
            "document_id": row.document_id,
            "document_url": row.document_url,
            "status": "FAILED",
            "http_status": pd.NA,
            "cache_path": pd.NA,
            "attempt": pd.NA,
            "error": repr(exc),
        })


hkex_document_download_log_df = pd.DataFrame(
    download_rows,
    columns=[
        "document_id",
        "document_url",
        "status",
        "http_status",
        "cache_path",
        "attempt",
        "error",
    ],
)

hkex_document_text_df = pd.DataFrame(
    text_rows,
    columns=[
        "document_id",
        "document_text",
        "status",
        "text_cache_path",
        "extraction_method",
        "pages_extracted",
        "character_count",
    ],
)

print("Documents selected:", len(documents_to_download))
print("Documents with extracted text:", len(hkex_document_text_df))

if not hkex_document_download_log_df.empty:
    display(
        hkex_document_download_log_df["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="count")
    )

Documents selected: 90
Documents with extracted text: 90


,status,count
0,CACHE_HIT,90


In [ ]:
# 10. HONG KONG ACCOUNT DICTIONARY

# Multiple labels may map to the same canonical concept. Priority 1 is the
# strongest deterministic mapping; higher values are weaker alternatives.

HK_ACCOUNT_MAPPINGS = [
    # ------------------------------------------------------------
    # Income statement
    # ------------------------------------------------------------
    ("revenue", r"^(revenue|turnover|sales|sales revenue|total revenue|operating revenue|營業額|收入|收益|總收入)$", 1),
    ("cost_of_revenue", r"^(cost of sales|cost of revenue|cost of goods sold|營業成本|銷售成本|收益成本)$", 1),
    ("gross_profit", r"^(gross profit|gross loss|毛利|毛損)$", 1),

    ("operating_income", r"^(operating profit|operating loss|profit from operations|loss from operations|profit from operating activities|loss from operating activities|經營溢利|經營虧損|營業利潤|營業虧損)$", 1),
    ("profit_before_tax", r"^(profit before tax|loss before tax|profit before taxation|loss before taxation|除稅前溢利|除稅前虧損|稅前利潤|稅前虧損)$", 1),
    ("income_tax_expense", r"^(income tax expense|income tax credit|taxation|tax expense|tax credit|所得稅開支|所得稅抵免|稅項|稅項支出)$", 1),

    ("net_income", r"^(profit for the year|profit for the period|loss for the year|loss for the period|net profit|net loss|年內溢利|期內溢利|年內虧損|期內虧損|淨利潤|淨虧損)$", 1),
    ("net_income_attributable_to_owners", r"^(profit attributable to owners|profit attributable to shareholders|profit attributable to equity holders|loss attributable to owners|owners of the company|equity holders of the company|本公司擁有人應佔溢利|股東應佔溢利|本公司權益持有人應佔溢利)$", 1),
    ("net_income_attributable_to_nci", r"^(profit attributable to non-controlling interests|loss attributable to non-controlling interests|非控股權益應佔溢利|非控股權益應佔虧損)$", 1),

    ("basic_eps", r"^(basic earnings per share|basic loss per share|earnings per share basic|每股基本盈利|每股基本虧損)$", 1),
    ("diluted_eps", r"^(diluted earnings per share|diluted loss per share|earnings per share diluted|每股攤薄盈利|每股攤薄虧損)$", 1),

    ("research_and_development_expense", r"^(research and development expenses?|research and development costs?|r&d expenses?|r&d costs?|研發開支|研發成本|研究及開發開支|研究及開發成本)$", 1),
    ("selling_expense", r"^(selling expenses?|distribution expenses?|selling and distribution expenses?|銷售開支|分銷開支|銷售及分銷開支)$", 1),
    ("general_and_administrative_expense", r"^(administrative expenses?|general and administrative expenses?|administration expenses?|行政開支|一般及行政開支)$", 1),
    ("selling_general_and_administrative_expense", r"^(selling and administrative expenses?|selling general and administrative expenses?|selling distribution and administrative expenses?|銷售及行政開支|銷售一般及行政開支)$", 1),

    ("employee_benefit_expense", r"^(employee benefit expenses?|employee costs?|staff costs?|staff expenses?|僱員福利開支|員工成本|員工開支)$", 1),
    ("depreciation_expense", r"^(depreciation expense|depreciation expenses|depreciation|折舊|折舊開支)$", 1),
    ("amortisation_expense", r"^(amortisation expense|amortization expense|amortisation|amortization|攤銷|攤銷開支)$", 1),
    ("depreciation_and_amortisation", r"^(depreciation and amortisation|depreciation and amortization|depreciation amortisation and impairment|折舊及攤銷)$", 1),
    ("impairment_loss", r"^(impairment loss|impairment losses|impairment charge|減值虧損|減值損失)$", 1),

    ("finance_income", r"^(finance income|financial income|財務收入|融資收入)$", 1),
    ("finance_costs", r"^(finance costs?|financial costs?|borrowing costs?|財務成本|融資成本|借款成本)$", 1),
    ("interest_expense", r"^(interest expense|interest expenses|interest costs?|利息開支|利息支出)$", 1),
    ("interest_income", r"^(interest income|interest revenue|利息收入)$", 1),

    ("share_of_profit_of_associates", r"^(share of profit of associates|share of profits of associates|share of profit of joint ventures|應佔聯營公司溢利|應佔合營企業溢利)$", 1),
    ("share_of_loss_of_associates", r"^(share of loss of associates|share of losses of associates|share of loss of joint ventures|應佔聯營公司虧損|應佔合營企業虧損)$", 1),

    ("other_comprehensive_income", r"^(other comprehensive income|other comprehensive loss|其他全面收益|其他全面虧損)$", 1),
    ("comprehensive_income", r"^(total comprehensive income|total comprehensive loss|全面收益總額|全面虧損總額)$", 1),

    # ------------------------------------------------------------
    # Balance sheet — assets
    # ------------------------------------------------------------
    ("total_assets", r"^(total assets|assets total|資產總額|總資產)$", 1),
    ("current_assets", r"^(current assets|total current assets|流動資產|流動資產總額)$", 1),
    ("noncurrent_assets", r"^(non-current assets|noncurrent assets|total non-current assets|total noncurrent assets|非流動資產|非流動資產總額)$", 1),

    ("cash_and_cash_equivalents", r"^(cash and cash equivalents|cash and bank balances|bank balances and cash|cash at bank and on hand|cash and deposits|現金及現金等價物|現金及銀行結餘|銀行結餘及現金|銀行存款及現金)$", 1),
    ("restricted_cash", r"^(restricted cash|restricted bank deposits|pledged bank deposits|受限制現金|受限制銀行存款|已抵押銀行存款)$", 1),
    ("short_term_investments", r"^(short-term investments|short term investments|current investments|current financial assets|短期投資|流動投資|流動金融資產)$", 1),

    ("trade_receivables", r"^(trade receivables|trade debtors|accounts receivable|trade and bills receivables|bills and trade receivables|貿易應收款|應收賬款|應收帳款|應收票據及貿易應收款|貿易及票據應收款)$", 1),
    ("other_receivables", r"^(other receivables|trade and other receivables|prepayments deposits and other receivables|其他應收款|貿易及其他應收款|預付款項按金及其他應收款)$", 1),
    ("finance_receivables", r"^(finance receivables|financing receivables|lease receivables|loans and receivables|融資應收款|租賃應收款|貸款及應收款)$", 1),
    ("inventory", r"^(inventories|inventory|stocks|存貨|庫存)$", 1),
    ("raw_material_inventory", r"^(raw materials|raw material inventories|原材料|原料)$", 1),
    ("work_in_progress_inventory", r"^(work in progress|work-in-progress|work in process|在製品|在產品|半成品)$", 1),
    ("finished_goods_inventory", r"^(finished goods|finished products|製成品|產成品|成品)$", 1),

    ("property_plant_equipment", r"^(property plant and equipment|property, plant and equipment|property plant equipment|fixed assets|物業廠房及設備|物業、廠房及設備|固定資產)$", 1),
    ("right_of_use_assets", r"^(right-of-use assets|right of use assets|使用權資產)$", 1),
    ("investment_property", r"^(investment properties|investment property|投資物業)$", 1),
    ("goodwill", r"^(goodwill|商譽)$", 1),
    ("intangible_assets", r"^(intangible assets|other intangible assets|無形資產|其他無形資產)$", 1),
    ("capitalised_development_costs", r"^(capitalised development costs|capitalized development costs|development costs|資本化開發成本|開發成本)$", 1),

    ("investments_in_associates", r"^(investments in associates|interests in associates|investment in associates|於聯營公司的投資|於聯營公司之權益)$", 1),
    ("investments_in_joint_ventures", r"^(investments in joint ventures|interests in joint ventures|於合營企業的投資|於合營企業之權益)$", 1),
    ("deferred_tax_assets", r"^(deferred tax assets|遞延稅項資產)$", 1),

    # ------------------------------------------------------------
    # Balance sheet — liabilities
    # ------------------------------------------------------------
    ("total_liabilities", r"^(total liabilities|liabilities total|負債總額|總負債)$", 1),
    ("current_liabilities", r"^(current liabilities|total current liabilities|流動負債|流動負債總額)$", 1),
    ("noncurrent_liabilities", r"^(non-current liabilities|noncurrent liabilities|total non-current liabilities|total noncurrent liabilities|非流動負債|非流動負債總額)$", 1),

    ("trade_payables", r"^(trade payables|trade creditors|accounts payable|trade and bills payables|bills and trade payables|貿易應付款|應付賬款|應付帳款|應付票據及貿易應付款|貿易及票據應付款)$", 1),
    ("other_payables", r"^(other payables|trade and other payables|accruals and other payables|accrued expenses and other payables|其他應付款|貿易及其他應付款|應計費用及其他應付款)$", 1),
    ("contract_liabilities", r"^(contract liabilities|deferred revenue|advance receipts from customers|合約負債|遞延收入|客戶預付款)$", 1),

    ("short_term_debt", r"^(short-term borrowings|short term borrowings|current borrowings|current interest-bearing borrowings|bank borrowings current|短期借款|流動借款|流動銀行借款)$", 1),
    ("long_term_debt", r"^(long-term borrowings|long term borrowings|non-current borrowings|noncurrent borrowings|non-current interest-bearing borrowings|長期借款|非流動借款)$", 1),
    ("total_borrowings", r"^(total borrowings|borrowings|interest-bearing borrowings|bank loans and other borrowings|借款總額|借款|計息借款|銀行貸款及其他借款)$", 1),

    ("current_lease_liabilities", r"^(current lease liabilities|lease liabilities current|流動租賃負債)$", 1),
    ("noncurrent_lease_liabilities", r"^(non-current lease liabilities|noncurrent lease liabilities|lease liabilities non-current|非流動租賃負債)$", 1),

    ("warranty_provisions", r"^(warranty provisions?|product warranty provisions?|provision for warranties|保修撥備|產品保證撥備|保養撥備)$", 1),
    ("pension_liabilities", r"^(retirement benefit obligations|defined benefit liabilities|pension liabilities|retirement benefit liabilities|退休福利責任|界定福利負債)$", 1),
    ("deferred_tax_liabilities", r"^(deferred tax liabilities|遞延稅項負債)$", 1),

    # ------------------------------------------------------------
    # Equity
    # ------------------------------------------------------------
    ("total_equity", r"^(total equity|equity|shareholders equity|equity total|權益總額|權益|股東權益)$", 1),
    ("equity_attributable_to_owners", r"^(equity attributable to owners|equity attributable to shareholders|equity attributable to equity holders|本公司擁有人應佔權益|股東應佔權益|本公司權益持有人應佔權益)$", 1),
    ("noncontrolling_interests", r"^(non-controlling interests|noncontrolling interests|minority interests|非控股權益|少數股東權益)$", 1),
    ("share_capital", r"^(share capital|issued capital|股本|已發行股本)$", 1),
    ("share_premium", r"^(share premium|share premium account|股份溢價|股份溢價賬)$", 1),
    ("retained_earnings", r"^(retained earnings|retained profits|accumulated profits|保留盈利|保留溢利|累計溢利)$", 1),
    ("treasury_shares", r"^(treasury shares|treasury stock|own shares|庫存股份|自身股份)$", 1),

    # ------------------------------------------------------------
    # Cash flow
    # ------------------------------------------------------------
    ("operating_cash_flow", r"^(net cash generated from operating activities|net cash used in operating activities|cash generated from operations|net cash flows from operating activities|經營活動所得現金淨額|經營活動所用現金淨額|經營所得現金)$", 1),
    ("investing_cash_flow", r"^(net cash generated from investing activities|net cash used in investing activities|net cash flows from investing activities|投資活動所得現金淨額|投資活動所用現金淨額)$", 1),
    ("financing_cash_flow", r"^(net cash generated from financing activities|net cash used in financing activities|net cash flows from financing activities|融資活動所得現金淨額|融資活動所用現金淨額)$", 1),

    ("capital_expenditure", r"^(purchase of property plant and equipment|purchase of property, plant and equipment|payments for property plant and equipment|additions to property plant and equipment|購置物業廠房及設備|購買物業廠房及設備)$", 1),
    ("intangible_asset_purchases", r"^(purchase of intangible assets|payments for intangible assets|additions to intangible assets|購置無形資產|購買無形資產)$", 1),
    ("business_acquisition_cash_outflow", r"^(acquisition of subsidiaries|purchase of subsidiaries|acquisition of businesses|收購附屬公司|收購業務)$", 1),
    ("business_disposal_cash_inflow", r"^(disposal of subsidiaries|proceeds from disposal of subsidiaries|disposal of businesses|出售附屬公司所得款項|出售業務所得款項)$", 1),

    ("debt_issuance", r"^(proceeds from borrowings|proceeds from bank borrowings|proceeds from issue of debt|new bank loans|借款所得款項|銀行借款所得款項|發行債務所得款項)$", 1),
    ("debt_repayment", r"^(repayment of borrowings|repayments of borrowings|repayment of bank loans|償還借款|償還銀行貸款)$", 1),
    ("lease_payments", r"^(repayment of lease liabilities|lease payments|principal portion of lease payments|償還租賃負債|租賃付款)$", 1),

    ("dividends_paid", r"^(dividends paid|payment of dividends|dividend paid|已付股息|支付股息)$", 1),
    ("share_repurchases", r"^(repurchase of shares|purchase of own shares|share buybacks|購回股份|回購股份)$", 1),
    ("share_issuance_proceeds", r"^(proceeds from issue of shares|issue of shares|發行股份所得款項)$", 1),

    ("interest_paid", r"^(interest paid|finance costs paid|已付利息)$", 1),
    ("interest_received", r"^(interest received|已收利息)$", 1),
    ("income_taxes_paid", r"^(income taxes paid|tax paid|taxes paid|已付所得稅|已付稅項)$", 1),
    ("cash_change", r"^(net increase in cash and cash equivalents|net decrease in cash and cash equivalents|increase in cash and cash equivalents|decrease in cash and cash equivalents|現金及現金等價物增加淨額|現金及現金等價物減少淨額)$", 1),

    # ------------------------------------------------------------
    # Per-share / capital structure
    # ------------------------------------------------------------
    ("weighted_average_shares_basic", r"^(weighted average number of ordinary shares|weighted average number of shares basic|普通股加權平均數|基本加權平均股數)$", 1),
    ("weighted_average_shares_diluted", r"^(weighted average number of diluted shares|weighted average number of shares diluted|攤薄加權平均股數)$", 1),

    # ------------------------------------------------------------
    # Automotive / sector extensions
    # ------------------------------------------------------------
    ("vehicle_sales_volume", r"^(vehicle sales|vehicles sold|vehicle sales volume|sales volume|汽車銷量|車輛銷量|銷售量)$", 1),
    ("vehicle_production_volume", r"^(vehicle production|vehicles produced|vehicle production volume|production volume|汽車產量|車輛產量|產量)$", 1),
    ("automotive_revenue", r"^(automotive revenue|automobile revenue|automotive segment revenue|vehicle revenue|汽車業務收入|汽車收入)$", 1),
    ("financial_services_revenue", r"^(financial services revenue|finance segment revenue|金融服務收入)$", 1),
    ("automotive_debt", r"^(automotive debt|automotive borrowings|汽車業務債務|汽車業務借款)$", 1),
    ("financial_services_debt", r"^(financial services debt|financial services borrowings|金融服務債務|金融服務借款)$", 1),
]


hong_kong_source_account_mapping_proposed_df = pd.DataFrame(
    HK_ACCOUNT_MAPPINGS,
    columns=[
        "standard_concept",
        "account_label_regex",
        "priority",
    ],
)

canonical_concepts = set(
    global_canonical_schema_df[
        "standard_concept"
    ]
    .dropna()
    .astype("string")
)

hong_kong_unsupported_concept_mappings_df = (
    hong_kong_source_account_mapping_proposed_df[
        ~hong_kong_source_account_mapping_proposed_df[
            "standard_concept"
        ].isin(canonical_concepts)
    ]
    .copy()
    .reset_index(drop=True)
)

hong_kong_source_account_mapping_df = (
    hong_kong_source_account_mapping_proposed_df[
        hong_kong_source_account_mapping_proposed_df[
            "standard_concept"
        ].isin(canonical_concepts)
    ]
    .copy()
    .reset_index(drop=True)
)

hong_kong_standard_concept_dictionary_df = (
    hong_kong_source_account_mapping_df.merge(
        global_canonical_schema_df,
        on="standard_concept",
        how="left",
        validate="m:1",
    )
)

unresolved_after_filter = (
    hong_kong_standard_concept_dictionary_df[
        hong_kong_standard_concept_dictionary_df[
            "statement_type"
        ].isna()
    ]["standard_concept"]
    .drop_duplicates()
    .tolist()
)

if unresolved_after_filter:
    raise RuntimeError(
        "Canonical merge failed after filtering supported concepts: "
        f"{unresolved_after_filter}"
    )

print(
    "Proposed Hong Kong mapping rows:",
    len(hong_kong_source_account_mapping_proposed_df),
)

print(
    "Supported Hong Kong mapping rows:",
    len(hong_kong_source_account_mapping_df),
)

print(
    "Canonical concepts represented:",
    hong_kong_standard_concept_dictionary_df[
        "standard_concept"
    ].nunique(),
)

print(
    "Unsupported proposed canonical concepts:",
    hong_kong_unsupported_concept_mappings_df[
        "standard_concept"
    ].nunique(),
)

if not hong_kong_unsupported_concept_mappings_df.empty:
    display(
        hong_kong_unsupported_concept_mappings_df[
            [
                "standard_concept",
                "account_label_regex",
                "priority",
            ]
        ]
        .sort_values(
            [
                "standard_concept",
                "priority",
            ]
        )
        .reset_index(drop=True)
    )

Proposed Hong Kong mapping rows: 96
Supported Hong Kong mapping rows: 90
Canonical concepts represented: 90
Unsupported proposed canonical concepts: 6


,standard_concept,account_label_regex,priority
0,investment_property,^(investment properties|investment property|投資...,1
1,investments_in_joint_ventures,^(investments in joint ventures|interests in j...,1
2,share_of_loss_of_associates,^(share of loss of associates|share of losses ...,1
3,share_of_profit_of_associates,^(share of profit of associates|share of profi...,1
4,weighted_average_shares_basic,^(weighted average number of ordinary shares|w...,1
5,weighted_average_shares_diluted,^(weighted average number of diluted shares|we...,1


In [ ]:
# 11. EXTRACT HONG KONG ACCOUNT/VALUE CANDIDATES

NUMBER_PATTERN = re.compile(
    r"(?P<value>"
    r"\(?-?\d{1,3}(?:,\d{3})*(?:\.\d+)?\)?"
    r"|"
    r"\(?-?\d+(?:\.\d+)?\)?"
    r")"
    r"(?:\s*(?P<unit>"
    r"thousand|million|billion|"
    r"千|百萬|十億"
    r"))?",
    flags=re.IGNORECASE,
)

CURRENCY_PATTERN = re.compile(
    r"(HK\$|HKD|RMB|CNY|USD|US\$|EUR|GBP|"
    r"港幣|港元|人民幣)",
    flags=re.IGNORECASE,
)

PERIOD_HEADER_PATTERN = re.compile(
    r"^(year ended|six months ended|period ended|"
    r"as at|at|截至|於)\b",
    flags=re.IGNORECASE,
)

NOTE_PREFIX_PATTERN = re.compile(
    r"^(?:note\s*)?\(?\d{1,3}[a-z]?\)?[\.\-:\s]+",
    flags=re.IGNORECASE,
)

TRAILING_NOTE_PATTERN = re.compile(
    r"\s+(?:note\s*)?\(?\d{1,3}[a-z]?\)?$",
    flags=re.IGNORECASE,
)


def normalise_label(value):
    text = html_lib.unescape(str(value))

    text = text.replace("\u00a0", " ")
    text = text.replace("，", ",")
    text = text.replace("：", ":")
    text = text.replace("（", "(")
    text = text.replace("）", ")")

    text = re.sub(r"\s+", " ", text).strip()

    # Remove bullets, table numbering and common note references.
    text = re.sub(
        r"^[\-–—•·*#\s]+",
        "",
        text,
    )

    text = NOTE_PREFIX_PATTERN.sub("", text)
    text = TRAILING_NOTE_PATTERN.sub("", text)

    # Remove trailing punctuation without removing internal hyphens.
    text = re.sub(
        r"[\s:;,\.\-–—]+$",
        "",
        text,
    )

    return text.strip()


def canonicalise_label_for_matching(value):
    text = normalise_label(value).lower()

    # Remove common display-only fragments.
    text = re.sub(
        r"\b(note|notes|restated|audited|unaudited|"
        r"continuing operations|discontinued operations)\b",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\b(hk\$|rmb|cny|usd|eur|gbp|"
        r"million|billion|thousand)\b",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"[^\w\u3400-\u9fff&\- ]+",
        " ",
        text,
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


def parse_number(value_text, unit_text=None):
    if value_text is None:
        return np.nan

    text = str(value_text).replace(",", "").strip()

    negative = (
        text.startswith("(")
        and text.endswith(")")
    )

    text = text.strip("()")

    value = pd.to_numeric(
        text,
        errors="coerce",
    )

    if pd.isna(value):
        return np.nan

    if negative:
        value = -value

    unit_key = (
        str(unit_text).lower()
        if unit_text
        else ""
    )

    multiplier = {
        "thousand": 1_000,
        "千": 1_000,
        "million": 1_000_000,
        "百萬": 1_000_000,
        "billion": 1_000_000_000,
        "十億": 1_000_000_000,
    }.get(unit_key, 1)

    return float(value) * multiplier


def infer_document_currency(text):
    matches = CURRENCY_PATTERN.findall(
        str(text)[:30_000]
    )

    if not matches:
        return pd.NA

    mapping = {
        "HK$": "HKD",
        "HKD": "HKD",
        "港幣": "HKD",
        "港元": "HKD",
        "RMB": "CNY",
        "CNY": "CNY",
        "人民幣": "CNY",
        "USD": "USD",
        "US$": "USD",
        "EUR": "EUR",
        "GBP": "GBP",
    }

    normalised = [
        mapping.get(
            str(match).upper(),
            mapping.get(
                str(match),
                str(match),
            ),
        )
        for match in matches
    ]

    counts = pd.Series(normalised).value_counts()

    return (
        counts.index[0]
        if not counts.empty
        else pd.NA
    )


def infer_statement_type_from_context(
    lines,
    line_number,
):
    start = max(0, line_number - 12)

    context = " ".join(
        lines[start : line_number + 1]
    ).lower()

    if re.search(
        r"(statement of financial position|balance sheet|"
        r"consolidated statement of financial position|"
        r"財務狀況表|資產負債表)",
        context,
    ):
        return "BALANCE_SHEET"

    if re.search(
        r"(statement of profit or loss|income statement|"
        r"statement of comprehensive income|"
        r"損益表|綜合收益表|全面收益表)",
        context,
    ):
        return "INCOME_STATEMENT"

    if re.search(
        r"(statement of cash flows|cash flow statement|"
        r"現金流量表)",
        context,
    ):
        return "CASH_FLOW"

    if re.search(
        r"(statement of changes in equity|"
        r"權益變動表)",
        context,
    ):
        return "EQUITY_STATEMENT"

    return pd.NA


def extract_account_candidates(
    document_id,
    document_text,
    extraction_method,
):
    lines = [
        normalise_label(line)
        for line in str(document_text).splitlines()
    ]

    lines = [
        line
        for line in lines
        if line
    ]

    document_currency = infer_document_currency(
        document_text
    )

    rows = []

    for line_number, line in enumerate(lines):
        # Skip pure headers and date lines.
        if PERIOD_HEADER_PATTERN.search(line):
            continue

        number_matches = list(
            NUMBER_PATTERN.finditer(line)
        )

        if not number_matches:
            continue

        first_number = number_matches[0]

        raw_label = normalise_label(
            line[: first_number.start()]
        )

        account_label_clean = canonicalise_label_for_matching(
            raw_label
        )

        if len(account_label_clean) < 2:
            continue

        if len(account_label_clean) > 220:
            continue

        if re.fullmatch(
            r"[\d\W_]+",
            account_label_clean,
        ):
            continue

        # Exclude lines whose "label" is effectively a date or percentage.
        if re.fullmatch(
            r"(20\d{2}|19\d{2}|percentage|percent|%)",
            account_label_clean,
            flags=re.IGNORECASE,
        ):
            continue

        statement_type_inferred = (
            infer_statement_type_from_context(
                lines,
                line_number,
            )
        )

        for value_rank, match in enumerate(
            number_matches[:6],
            start=1,
        ):
            rows.append({
                "document_id": document_id,
                "line_number": line_number,
                "source_line": line,
                "account_label_raw": raw_label,
                "account_label": account_label_clean,
                "value_rank_in_line": value_rank,
                "reported_value_text": match.group("value"),
                "reported_value": parse_number(
                    match.group("value"),
                    match.group("unit"),
                ),
                "scale_label": match.group("unit"),
                "currency": document_currency,
                "statement_type_inferred": (
                    statement_type_inferred
                ),
                "extraction_method": extraction_method,
            })

    return pd.DataFrame(
        rows,
        columns=[
            "document_id",
            "line_number",
            "source_line",
            "account_label_raw",
            "account_label",
            "value_rank_in_line",
            "reported_value_text",
            "reported_value",
            "scale_label",
            "currency",
            "statement_type_inferred",
            "extraction_method",
        ],
    )


candidate_frames = []

if hkex_document_text_df.empty:
    print(
        "No extracted HKEX document text is available. "
        "An empty candidate table will be created."
    )

else:
    valid_text = (
        hkex_document_text_df[
            [
                "document_id",
                "document_text",
                "extraction_method",
            ]
        ]
        .dropna(
            subset=[
                "document_id",
                "document_text",
            ]
        )
        .copy()
    )

    valid_text = valid_text[
        valid_text["document_text"]
        .astype("string")
        .str.strip()
        .ne("")
    ]

    for row in tqdm(
        valid_text.itertuples(index=False),
        total=len(valid_text),
        desc="Extracting HKEX account candidates",
    ):
        frame = extract_account_candidates(
            row.document_id,
            row.document_text,
            row.extraction_method,
        )

        if not frame.empty:
            candidate_frames.append(frame)


hong_kong_account_candidates_raw_df = (
    pd.concat(
        candidate_frames,
        ignore_index=True,
    )
    if candidate_frames
    else pd.DataFrame(
        columns=[
            "document_id",
            "line_number",
            "source_line",
            "account_label_raw",
            "account_label",
            "value_rank_in_line",
            "reported_value_text",
            "reported_value",
            "scale_label",
            "currency",
            "statement_type_inferred",
            "extraction_method",
        ]
    )
)

print(
    "Raw account/value candidate rows:",
    len(hong_kong_account_candidates_raw_df),
)

print(
    "Unique cleaned account labels:",
    hong_kong_account_candidates_raw_df[
        "account_label"
    ].nunique(),
)

Extracting HKEX account candidates:   0%|          | 0/90 [00:00<?, ?it/s]

Raw account/value candidate rows: 90253
Unique cleaned account labels: 15093


In [ ]:
# 12. MAP AND STANDARDISE HONG KONG FACTS

FUZZY_ACCEPT_SCORE = 91
FUZZY_REVIEW_SCORE = 82

# Canonical anchor phrases used only after deterministic mappings fail.
# These are not direct mappings by themselves; they generate candidates.
CANONICAL_ANCHORS = {
    "revenue": [
        "revenue",
        "turnover",
        "sales revenue",
        "operating revenue",
        "營業額",
        "收入",
        "收益",
    ],
    "cost_of_revenue": [
        "cost of sales",
        "cost of revenue",
        "cost of goods sold",
        "銷售成本",
        "營業成本",
    ],
    "gross_profit": [
        "gross profit",
        "毛利",
    ],
    "operating_income": [
        "operating profit",
        "profit from operations",
        "經營溢利",
        "營業利潤",
    ],
    "profit_before_tax": [
        "profit before tax",
        "profit before taxation",
        "除稅前溢利",
        "稅前利潤",
    ],
    "net_income": [
        "profit for the year",
        "profit for the period",
        "net profit",
        "年內溢利",
        "期內溢利",
        "淨利潤",
    ],
    "cash_and_cash_equivalents": [
        "cash and cash equivalents",
        "cash and bank balances",
        "bank balances and cash",
        "現金及現金等價物",
        "現金及銀行結餘",
    ],
    "trade_receivables": [
        "trade receivables",
        "accounts receivable",
        "trade and bills receivables",
        "貿易應收款",
        "應收賬款",
    ],
    "inventory": [
        "inventories",
        "inventory",
        "存貨",
    ],
    "property_plant_equipment": [
        "property plant and equipment",
        "fixed assets",
        "物業廠房及設備",
        "固定資產",
    ],
    "total_assets": [
        "total assets",
        "assets total",
        "資產總額",
    ],
    "trade_payables": [
        "trade payables",
        "accounts payable",
        "trade and bills payables",
        "貿易應付款",
        "應付賬款",
    ],
    "short_term_debt": [
        "short term borrowings",
        "current borrowings",
        "短期借款",
    ],
    "long_term_debt": [
        "long term borrowings",
        "non current borrowings",
        "長期借款",
        "非流動借款",
    ],
    "total_liabilities": [
        "total liabilities",
        "liabilities total",
        "負債總額",
    ],
    "total_equity": [
        "total equity",
        "shareholders equity",
        "權益總額",
        "股東權益",
    ],
    "operating_cash_flow": [
        "net cash generated from operating activities",
        "cash generated from operations",
        "經營活動所得現金淨額",
    ],
    "investing_cash_flow": [
        "net cash used in investing activities",
        "net cash generated from investing activities",
        "投資活動所用現金淨額",
    ],
    "financing_cash_flow": [
        "net cash used in financing activities",
        "net cash generated from financing activities",
        "融資活動所用現金淨額",
    ],
    "capital_expenditure": [
        "purchase of property plant and equipment",
        "payments for property plant and equipment",
        "購置物業廠房及設備",
    ],
    "vehicle_sales_volume": [
        "vehicle sales",
        "vehicle sales volume",
        "汽車銷量",
        "車輛銷量",
    ],
    "vehicle_production_volume": [
        "vehicle production",
        "vehicle production volume",
        "汽車產量",
        "車輛產量",
    ],
}

canonical_anchor_rows = []

for standard_concept, anchors in CANONICAL_ANCHORS.items():
    for anchor in anchors:
        canonical_anchor_rows.append({
            "standard_concept": standard_concept,
            "anchor_label": canonicalise_label_for_matching(
                anchor
            ),
        })

hong_kong_canonical_anchor_df = pd.DataFrame(
    canonical_anchor_rows
)


def expected_observed_unit_family(standard_concept):
    if standard_concept in {
        "basic_eps",
        "diluted_eps",
    }:
        return "PER_SHARE"

    if standard_concept in {
        "weighted_average_shares_basic",
        "weighted_average_shares_diluted",
        "vehicle_sales_volume",
        "vehicle_production_volume",
    }:
        return "COUNT"

    return "MONETARY"


def deterministic_mapping_candidates(
    account_label,
):
    dictionary = (
        hong_kong_standard_concept_dictionary_df
    )

    matched = dictionary[
        dictionary["account_label_regex"].map(
            lambda pattern: bool(
                re.search(
                    pattern,
                    str(account_label).strip(),
                    flags=re.IGNORECASE,
                )
            )
        )
    ].copy()

    if matched.empty:
        return matched

    matched["mapping_method"] = (
        "DETERMINISTIC_REGEX"
    )

    matched["mapping_score"] = (
        100.0 - matched["priority"].astype(float)
    )

    return matched


def hong_kong_block9_registry_mapping_candidates(account_label):
    """Return one unambiguous, previously accepted Block 9 synonym."""
    normalised_label = normalise_block9_synonym_label(account_label)

    if pd.isna(normalised_label):
        return pd.DataFrame()

    registry_row = block9_synonym_lookup.get(normalised_label)

    if not registry_row:
        return pd.DataFrame()

    standard_concept = registry_row.get("standard_concept")

    if pd.isna(standard_concept):
        return pd.DataFrame()

    result = pd.DataFrame([
        {
            "standard_concept": standard_concept,
            "anchor_label": normalised_label,
            "mapping_method": "BLOCK9_ACCEPTED_SYNONYM",
            "mapping_score": 100.0,
            "priority": 2,
            "block9_registry_accepted_observations": (
                registry_row.get("accepted_observations")
            ),
            "block9_registry_unique_issuers": (
                registry_row.get("unique_issuers")
            ),
            "block9_registry_unique_filings": (
                registry_row.get("unique_filings")
            ),
            "block9_registry_generation": (
                registry_row.get("registry_generation")
            ),
        }
    ])

    return result.merge(
        global_canonical_schema_df,
        on="standard_concept",
        how="left",
        validate="m:1",
    )


def fuzzy_mapping_candidates(
    account_label,
):
    anchors = (
        hong_kong_canonical_anchor_df[
            "anchor_label"
        ].tolist()
    )

    if not account_label or not anchors:
        return pd.DataFrame()

    matches = process.extract(
        account_label,
        anchors,
        scorer=fuzz.token_set_ratio,
        limit=5,
    )

    rows = []

    for matched_anchor, score, anchor_index in matches:
        standard_concept = (
            hong_kong_canonical_anchor_df
            .iloc[anchor_index][
                "standard_concept"
            ]
        )

        rows.append({
            "standard_concept": standard_concept,
            "anchor_label": matched_anchor,
            "mapping_method": "FUZZY_ANCHOR",
            "mapping_score": float(score),
            "priority": 99,
        })

    if not rows:
        return pd.DataFrame()

    result = pd.DataFrame(rows)

    result = result.merge(
        global_canonical_schema_df,
        on="standard_concept",
        how="left",
        validate="m:1",
    )

    return result



GENERIC_FUZZY_LABEL_STOPLIST = {
    "a",
    "an",
    "and",
    "at",
    "by",
    "for",
    "from",
    "in",
    "note",
    "notes",
    "of",
    "on",
    "or",
    "the",
    "to",
    "total",
    "with",
    "year",
    "period",
    "group",
    "company",
}

FUZZY_SINGLE_TOKEN_WHITELIST = {
    "amortisation",
    "amortization",
    "borrowings",
    "cash",
    "depreciation",
    "equity",
    "goodwill",
    "inventories",
    "inventory",
    "revenue",
    "sales",
    "taxation",
    "turnover",
}

def fuzzy_label_quality(account_label):
    """
    Decide whether a cleaned label contains enough semantic information for
    fuzzy mapping. Deterministic mappings are not restricted by this function.
    """
    if pd.isna(account_label):
        return {
            "eligible": False,
            "reason": "MISSING_LABEL",
            "character_count": 0,
            "token_count": 0,
        }

    text = str(account_label).strip().lower()

    if not text:
        return {
            "eligible": False,
            "reason": "EMPTY_LABEL",
            "character_count": 0,
            "token_count": 0,
        }

    tokens = [
        token
        for token in re.findall(
            r"[a-z0-9]+|[\u3400-\u9fff]+",
            text,
            flags=re.IGNORECASE,
        )
        if token
    ]

    character_count = len(
        re.sub(r"\s+", "", text)
    )
    token_count = len(tokens)
    contains_cjk = bool(
        re.search(r"[\u3400-\u9fff]", text)
    )

    if text in GENERIC_FUZZY_LABEL_STOPLIST:
        reason = "GENERIC_STOPWORD"

    elif re.fullmatch(r"[\W\d_]+", text):
        reason = "NON_SEMANTIC_LABEL"

    elif contains_cjk and character_count < 2:
        reason = "CJK_LABEL_TOO_SHORT"

    elif not contains_cjk and character_count < 4:
        reason = "LATIN_LABEL_TOO_SHORT"

    elif (
        not contains_cjk
        and token_count == 1
        and text not in FUZZY_SINGLE_TOKEN_WHITELIST
    ):
        reason = "UNAPPROVED_SINGLE_TOKEN"

    else:
        reason = "ELIGIBLE"

    return {
        "eligible": reason == "ELIGIBLE",
        "reason": reason,
        "character_count": character_count,
        "token_count": token_count,
    }


label_quality_rows = []

for label in (
    hong_kong_account_candidates_raw_df[
        "account_label"
    ]
    .dropna()
    .astype("string")
    .drop_duplicates()
):
    quality = fuzzy_label_quality(label)

    label_quality_rows.append({
        "account_label": label,
        "fuzzy_label_is_eligible": quality["eligible"],
        "fuzzy_label_rejection_reason": quality["reason"],
        "fuzzy_label_character_count": quality["character_count"],
        "fuzzy_label_token_count": quality["token_count"],
    })

hong_kong_fuzzy_label_quality_df = pd.DataFrame(
    label_quality_rows
)

fuzzy_eligible_labels = set(
    hong_kong_fuzzy_label_quality_df.loc[
        hong_kong_fuzzy_label_quality_df[
            "fuzzy_label_is_eligible"
        ],
        "account_label",
    ].astype("string")
)

unique_labels = (
    hong_kong_account_candidates_raw_df[
        "account_label"
    ]
    .dropna()
    .astype("string")
    .drop_duplicates()
    .tolist()
)

observed_mapping_rows = []

for account_label in tqdm(
    unique_labels,
    desc="Mapping unique HKEX account labels",
):
    deterministic = deterministic_mapping_candidates(
        account_label
    )

    if not deterministic.empty:
        deterministic["account_label"] = account_label
        observed_mapping_rows.append(
            deterministic
        )
        continue

    registry_mapping = (
        hong_kong_block9_registry_mapping_candidates(
            account_label
        )
    )

    if not registry_mapping.empty:
        registry_mapping["account_label"] = account_label
        observed_mapping_rows.append(
            registry_mapping
        )
        continue

    if str(account_label) not in fuzzy_eligible_labels:
        continue

    fuzzy = fuzzy_mapping_candidates(
        account_label
    )

    if not fuzzy.empty:
        fuzzy["account_label"] = account_label
        observed_mapping_rows.append(fuzzy)


hong_kong_observed_account_mapping_df = (
    pd.concat(
        observed_mapping_rows,
        ignore_index=True,
    )
    if observed_mapping_rows
    else pd.DataFrame()
)


hong_kong_block9_registry_matches_df = (
    hong_kong_observed_account_mapping_df.loc[
        hong_kong_observed_account_mapping_df[
            "mapping_method"
        ]
        .astype("string")
        .eq("BLOCK9_ACCEPTED_SYNONYM")
    ]
    .copy()
    .reset_index(drop=True)
    if (
        not hong_kong_observed_account_mapping_df.empty
        and "mapping_method"
        in hong_kong_observed_account_mapping_df.columns
    )
    else pd.DataFrame()
)

if hong_kong_observed_account_mapping_df.empty:
    hong_kong_block9_registry_matches_df = pd.DataFrame()
    hong_kong_fundamentals_mapped_df = pd.DataFrame()
    hong_kong_fundamentals_standardised_df = pd.DataFrame()
    hong_kong_mapping_alternatives_df = pd.DataFrame()
    hong_kong_mapping_review_queue_df = pd.DataFrame()
    hong_kong_low_quality_fuzzy_rejections_df = pd.DataFrame()
    hong_kong_fuzzy_label_quality_df = pd.DataFrame()
    hong_kong_unmapped_account_inventory_df = pd.DataFrame()
    hong_kong_automotive_extension_candidates_df = pd.DataFrame()

else:
    mapped = (
        hong_kong_account_candidates_raw_df.merge(
            hong_kong_observed_account_mapping_df,
            on="account_label",
            how="left",
            validate="m:m",
        )
    )

    mapped = mapped.merge(
        hong_kong_fuzzy_label_quality_df,
        on="account_label",
        how="left",
        validate="m:1",
    )

    mapped["fuzzy_label_is_eligible"] = (
        mapped["fuzzy_label_is_eligible"]
        .fillna(False)
        .astype(bool)
    )

    filing_bridge_columns = [
        "document_id",
        "announcement_id",
        "stock_code",
        "document_title",
        "category_name",
        "document_url",
        "available_datetime",
        "available_date",
        "availability_basis",
        "is_result_announcement",
        "is_long_form_report",
        "document_priority",
        "security_id",
        "issuer_id",
        "issuer_name",
        "ticker",
    ]

    # Select only columns that genuinely exist.
    filing_bridge = (
        hong_kong_filing_metadata_df[
            [
                column
                for column in filing_bridge_columns
                if column in hong_kong_filing_metadata_df.columns
            ]
        ]
        .copy()
    )

    listing_metadata_columns = [
        "security_id",
        "listing_start_date",
        "listing_end_date",
        "listing_status",
        "is_historical_listing",
    ]

    available_listing_columns = [
        column
        for column in listing_metadata_columns
        if column in hong_kong_security_universe_df.columns
    ]

    if "security_id" not in available_listing_columns:
        raise RuntimeError(
            "hong_kong_security_universe_df does not contain security_id."
        )

    listing_metadata_bridge = (
        hong_kong_security_universe_df[
            available_listing_columns
        ]
        .drop_duplicates("security_id")
        .copy()
    )

    # Remove any partial listing columns already present so the merge does not
    # create _x and _y variants.
    filing_bridge = filing_bridge.drop(
        columns=[
            column
            for column in [
                "listing_start_date",
                "listing_end_date",
                "listing_status",
                "is_historical_listing",
            ]
            if column in filing_bridge.columns
        ],
        errors="ignore",
    )

    filing_bridge = (
        filing_bridge.merge(
            listing_metadata_bridge,
            on="security_id",
            how="left",
            validate="m:1",
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Guarantee a stable downstream schema.
    for column, default_value in {
        "listing_start_date": pd.NaT,
        "listing_end_date": pd.NaT,
        "listing_status": pd.NA,
        "is_historical_listing": False,
    }.items():
        if column not in filing_bridge.columns:
            filing_bridge[column] = default_value

    filing_bridge["listing_start_date"] = pd.to_datetime(
        filing_bridge["listing_start_date"],
        errors="coerce",
    )

    filing_bridge["listing_end_date"] = pd.to_datetime(
        filing_bridge["listing_end_date"],
        errors="coerce",
    )

    filing_bridge["is_historical_listing"] = (
        filing_bridge["is_historical_listing"]
        .fillna(False)
        .astype("boolean")
    )

    print(
        "Filing bridge rows:",
        len(filing_bridge),
    )

    print(
        "Historical filing bridge rows:",
        int(
            filing_bridge[
                "is_historical_listing"
            ].fillna(False).sum()
        ),
    )

    mapped = mapped.merge(
        filing_bridge,
        on="document_id",
        how="left",
        validate="m:m",
    )

    mapped["observed_unit_family"] = (
        mapped["standard_concept"]
        .map(expected_observed_unit_family)
    )

    mapped["period_type_match"] = True

    statement_type_inferred = (
        mapped["statement_type_inferred"]
        .astype("string")
    )

    statement_type_expected = (
        mapped["statement_type"]
        .astype("string")
    )

    mapped["statement_type_match"] = (
        statement_type_inferred.isna()
        | statement_type_expected.isna()
        | statement_type_inferred.fillna("").eq(
            statement_type_expected.fillna("")
        )
    ).fillna(True).astype("boolean")

    observed_unit_family = (
        mapped["observed_unit_family"]
        .astype("string")
    )

    expected_unit_family = (
        mapped["expected_unit_family"]
        .astype("string")
    )

    mapped["unit_family_match"] = (
        observed_unit_family.isna()
        | expected_unit_family.isna()
        | observed_unit_family.fillna("").eq(
            expected_unit_family.fillna("")
        )
    ).fillna(True).astype("boolean")

    mapped["is_numeric_fact"] = (
        mapped["reported_value"]
        .notna()
        .astype("boolean")
    )

    deterministic_mask = (
        mapped["mapping_method"]
        .astype("string")
        .eq("DETERMINISTIC_REGEX")
        .fillna(False)
    )

    block9_registry_mask = (
        mapped["mapping_method"]
        .astype("string")
        .eq("BLOCK9_ACCEPTED_SYNONYM")
        .fillna(False)
    )

    fuzzy_score_accept_mask = (
        pd.to_numeric(
            mapped["mapping_score"],
            errors="coerce",
        )
        .ge(FUZZY_ACCEPT_SCORE)
        .fillna(False)
    )


    # ------------------------------------------------------------
    # Block 9 registry diagnostic
    # ------------------------------------------------------------

    registry_debug_columns = [
        "account_label",
        "standard_concept",
        "mapping_method",
        "mapping_score",

        "statement_type",
        "statement_type_inferred",
        "expected_statement_type_canonical",
        "statement_type_match",

        "reported_unit_family",
        "expected_unit_family",
        "expected_unit_family_canonical",
        "unit_family_match",

        "reported_value",
        "reported_value_numeric",
        "is_numeric_fact",
    ]

    available_registry_debug_columns = [
        column
        for column in registry_debug_columns
        if column in mapped.columns
    ]

    registry_debug = (
        mapped.loc[
            mapped[
                "mapping_method"
            ]
            .astype("string")
            .eq(
                "BLOCK9_ACCEPTED_SYNONYM"
            ),
            available_registry_debug_columns,
        ]
        .copy()
    )

    print(
        "Available diagnostic columns:",
        available_registry_debug_columns,
    )

    print(
        "Block 9 registry rows:",
        f"{len(registry_debug):,}",
    )

    display(
        registry_debug.head(50)
    )

    print(
        sorted(
            mapped.columns.tolist()
        )
    )

    # Deterministic mappings are accepted when numeric.
    # Fuzzy mappings require a high score plus compatible statement and unit
    # context. Every component is converted to an ordinary NA-safe Boolean.
    mapped["mapping_is_accepted"] = (
        deterministic_mask
        & mapped["is_numeric_fact"].fillna(False)
    ) | (
        block9_registry_mask
        & mapped["statement_type_match"].fillna(False)
        & mapped["unit_family_match"].fillna(False)
        & mapped["is_numeric_fact"].fillna(False)
    ) | (
        ~deterministic_mask
        & ~block9_registry_mask
        & fuzzy_score_accept_mask
        & mapped["fuzzy_label_is_eligible"].fillna(False)
        & mapped["statement_type_match"].fillna(False)
        & mapped["unit_family_match"].fillna(False)
        & mapped["is_numeric_fact"].fillna(False)
    )

    mapped["mapping_is_accepted"] = (
        mapped["mapping_is_accepted"]
        .fillna(False)
        .astype(bool)
    )

    fuzzy_method_mask = (
        mapped["mapping_method"]
        .astype("string")
        .eq("FUZZY_ANCHOR")
        .fillna(False)
    )

    fuzzy_review_score_mask = (
        pd.to_numeric(
            mapped["mapping_score"],
            errors="coerce",
        )
        .between(
            FUZZY_REVIEW_SCORE,
            FUZZY_ACCEPT_SCORE - 0.0001,
            inclusive="both",
        )
        .fillna(False)
    )

    mapped["mapping_requires_review"] = (
        fuzzy_method_mask
        & fuzzy_review_score_mask
        & mapped["fuzzy_label_is_eligible"].fillna(False)
    ).astype(bool)

    hong_kong_low_quality_fuzzy_rejections_df = (
        mapped[
            fuzzy_method_mask
            & ~mapped[
                "fuzzy_label_is_eligible"
            ].fillna(False)
        ]
        .copy()
        .sort_values(
            [
                "fuzzy_label_rejection_reason",
                "account_label",
                "mapping_score",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    hong_kong_mapping_review_queue_df = (
        mapped[
            mapped["mapping_requires_review"]
        ]
        .copy()
        .sort_values(
            [
                "mapping_score",
                "account_label",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    accepted = mapped[
        mapped["mapping_is_accepted"]
    ].copy()

    accepted_mapping_score = pd.to_numeric(
        accepted["mapping_score"],
        errors="coerce",
    ).fillna(0.0)

    accepted_priority = pd.to_numeric(
        accepted["priority"],
        errors="coerce",
    ).fillna(99.0)

    accepted_value_rank = pd.to_numeric(
        accepted["value_rank_in_line"],
        errors="coerce",
    ).fillna(1.0)

    accepted_document_priority = pd.to_numeric(
        accepted["document_priority"],
        errors="coerce",
    ).fillna(99.0)

    accepted_deterministic_mask = (
        accepted["mapping_method"]
        .astype("string")
        .eq("DETERMINISTIC_REGEX")
        .fillna(False)
    )

    statement_mismatch_penalty = (
        ~accepted["statement_type_match"]
        .fillna(False)
        .astype(bool)
    ).astype(int)

    unit_mismatch_penalty = (
        ~accepted["unit_family_match"]
        .fillna(False)
        .astype(bool)
    ).astype(int)

    non_numeric_penalty = (
        ~accepted["is_numeric_fact"]
        .fillna(False)
        .astype(bool)
    ).astype(int)

    accepted["selection_score"] = (
        np.where(
            accepted_deterministic_mask,
            0.0,
            200.0 - accepted_mapping_score,
        )
        + accepted_priority * 2.0
        + (accepted_value_rank - 1.0) * 10.0
        + statement_mismatch_penalty * 25.0
        + unit_mismatch_penalty * 20.0
        + non_numeric_penalty * 100.0
        + accepted_document_priority * 3.0
    )

    duplicate_key = [
        "issuer_id",
        "security_id",
        "document_id",
        "standard_concept",
        "available_datetime",
        "currency",
    ]

    for column in duplicate_key:
        if column not in accepted.columns:
            accepted[column] = pd.NA

    accepted = (
        accepted
        .sort_values(
            duplicate_key
            + [
                "selection_score",
                "line_number",
                "value_rank_in_line",
            ]
        )
        .reset_index(drop=True)
    )

    accepted["concept_selection_rank"] = (
        accepted
        .groupby(
            duplicate_key,
            dropna=False,
        )
        .cumcount()
        + 1
    )

    accepted["is_selected_standard_fact"] = (
        accepted[
            "concept_selection_rank"
        ].eq(1)
    )

    hong_kong_fundamentals_mapped_df = (
        accepted.copy()
    )

    hong_kong_mapping_alternatives_df = (
        accepted[
            ~accepted[
                "is_selected_standard_fact"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    hong_kong_fundamentals_standardised_df = (
        accepted[
            accepted[
                "is_selected_standard_fact"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    mapped_label_set = set(
        mapped.loc[
            mapped["mapping_is_accepted"],
            "account_label",
        ]
        .dropna()
        .astype("string")
    )

    unmapped = (
        hong_kong_account_candidates_raw_df[
            ~hong_kong_account_candidates_raw_df[
                "account_label"
            ]
            .astype("string")
            .isin(mapped_label_set)
        ]
        .copy()
    )

    hong_kong_unmapped_account_inventory_df = (
        unmapped
        .groupby(
            "account_label",
            dropna=False,
        )
        .agg(
            candidate_rows=(
                "reported_value",
                "size",
            ),
            document_count=(
                "document_id",
                "nunique",
            ),
            numeric_share=(
                "reported_value",
                lambda series: (
                    series.notna().mean()
                ),
            ),
            example_raw_label=(
                "account_label_raw",
                "first",
            ),
            example_line=(
                "source_line",
                "first",
            ),
            statement_type_example=(
                "statement_type_inferred",
                "first",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "document_count",
                "candidate_rows",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    automotive_pattern = re.compile(
        r"(vehicle|automotive|automobile|production|"
        r"deliver|warranty|dealer|battery|electric vehicle|"
        r"charging|mobility|汽車|車輛|產量|銷量|交付|"
        r"保修|售後|電池|新能源|電動車|充電)",
        flags=re.IGNORECASE,
    )

    hong_kong_automotive_extension_candidates_df = (
        hong_kong_unmapped_account_inventory_df[
            hong_kong_unmapped_account_inventory_df[
                "account_label"
            ]
            .astype("string")
            .str.contains(
                automotive_pattern,
                na=False,
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


print(
    "Observed mapping rows:",
    len(hong_kong_observed_account_mapping_df),
)

print(
    "Accepted mapped HKEX facts:",
    len(hong_kong_fundamentals_mapped_df),
)

print(
    "Selected HKEX standardised facts:",
    len(hong_kong_fundamentals_standardised_df),
)

print(
    "Unique standard concepts observed:",
    (
        hong_kong_fundamentals_standardised_df[
            "standard_concept"
        ].nunique()
        if not hong_kong_fundamentals_standardised_df.empty
        else 0
    ),
)

print(
    "Mapping-review queue:",
    len(hong_kong_mapping_review_queue_df),
)

print(
    "Fuzzy-eligible unique labels:",
    int(
        hong_kong_fuzzy_label_quality_df[
            "fuzzy_label_is_eligible"
        ].fillna(False).sum()
    )
    if not hong_kong_fuzzy_label_quality_df.empty
    else 0,
)

print(
    "Low-quality fuzzy candidates rejected:",
    len(hong_kong_low_quality_fuzzy_rejections_df),
)

print(
    "Unmapped labels:",
    len(hong_kong_unmapped_account_inventory_df),
)

print(
    "Automotive extensions:",
    len(hong_kong_automotive_extension_candidates_df),
)

if not hong_kong_fundamentals_standardised_df.empty:
    display(
        hong_kong_fundamentals_standardised_df[
            [
                "issuer_name",
                "document_title",
                "account_label_raw",
                "account_label",
                "standard_concept",
                "reported_value",
                "currency",
                "mapping_method",
                "mapping_score",
                "statement_type_inferred",
                "available_datetime",
            ]
        ]
        .head(100)
    )

Mapping unique HKEX account labels:   0%|          | 0/15093 [00:00<?, ?it/s]

Filing bridge rows: 90
Historical filing bridge rows: 0
Available diagnostic columns: ['account_label', 'standard_concept', 'mapping_method', 'mapping_score', 'statement_type', 'statement_type_inferred', 'statement_type_match', 'expected_unit_family', 'unit_family_match', 'reported_value', 'is_numeric_fact']
Block 9 registry rows: 226


,account_label,standard_concept,mapping_method,mapping_score,statement_type,statement_type_inferred,statement_type_match,expected_unit_family,unit_family_match,reported_value,is_numeric_fact
545,property plant and equipment increase by,capital_expenditure,BLOCK9_ACCEPTED_SYNONYM,100.0,CASH_FLOW,BALANCE_SHEET,False,MONETARY,True,1.314400e+07,True
1121,wholesale volume of passenger vehicles reached,vehicle_sales_volume,BLOCK9_ACCEPTED_SYNONYM,100.0,OPERATING_METRIC,<NA>,True,COUNT,True,2.144400e+07,True
3920,cost of sales decreased by,cost_of_revenue,BLOCK9_ACCEPTED_SYNONYM,100.0,INCOME_STATEMENT,<NA>,True,MONETARY,True,7.400000e+00,True
3921,cost of sales decreased by,cost_of_revenue,BLOCK9_ACCEPTED_SYNONYM,100.0,INCOME_STATEMENT,<NA>,True,MONETARY,True,4.090700e+09,True
3922,cost of sales decreased by,cost_of_revenue,BLOCK9_ACCEPTED_SYNONYM,100.0,INCOME_STATEMENT,<NA>,True,MONETARY,True,2.010000e+02,True
3923,cost of sales decreased by,cost_of_revenue,BLOCK9_ACCEPTED_SYNONYM,100.0,INCOME_STATEMENT,<NA>,True,MONETARY,True,8.000000e+00,True
3924,cost of sales decreased by,cost_of_revenue,BLOCK9_ACCEPTED_SYNONYM,100.0,INCOME_STATEMENT,<NA>,True,MONETARY,True,3.787600e+09,True
3925,cost of sales decreased by,cost_of_revenue,BLOCK9_ACCEPTED_SYNONYM,100.0,INCOME_STATEMENT,<NA>,True,MONETARY,True,2.010000e+02,True
6270,income tax expense increased by approximately,income_tax_expense,BLOCK9_ACCEPTED_SYNONYM,100.0,INCOME_STATEMENT,<NA>,True,MONETARY,True,7.900000e+01,True
6271,income tax expense increased by approximately,income_tax_expense,BLOCK9_ACCEPTED_SYNONYM,100.0,INCOME_STATEMENT,<NA>,True,MONETARY,True,4.640000e+07,True


['account_label', 'account_label_raw', 'account_label_regex', 'aggregation_policy', 'anchor_label', 'announcement_id', 'availability_basis', 'available_date', 'available_datetime', 'block9_registry_accepted_observations', 'block9_registry_generation', 'block9_registry_unique_filings', 'block9_registry_unique_issuers', 'category_name', 'core_tier', 'currency', 'document_id', 'document_priority', 'document_title', 'document_url', 'expected_period_type', 'expected_unit_family', 'extraction_method', 'fuzzy_label_character_count', 'fuzzy_label_is_eligible', 'fuzzy_label_rejection_reason', 'fuzzy_label_token_count', 'is_core', 'is_historical_listing', 'is_long_form_report', 'is_numeric_fact', 'is_result_announcement', 'issuer_id', 'issuer_name', 'line_number', 'listing_end_date', 'listing_start_date', 'listing_status', 'mapping_method', 'mapping_score', 'observed_unit_family', 'period_type_match', 'priority', 'reported_value', 'reported_value_text', 'scale_label', 'security_id', 'source_line

/tmp/ipykernel_1172/1881648265.py:1148: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,issuer_name,document_title,account_label_raw,account_label,standard_concept,reported_value,currency,mapping_method,mapping_score,statement_type_inferred,available_datetime
0,"Guangzhou Automobile Group Co., Ltd",2019 ANNUAL RESULTS ANNOUNCEMENT,corresponding period last year. Total gross pr...,corresponding period last year total gross pro...,gross_profit,2.523000e+00,CNY,FUZZY_ANCHOR,100.000000,<NA>,2020-03-31 14:03:00+00:00
1,"Guangzhou Automobile Group Co., Ltd",2019 ANNUAL RESULTS ANNOUNCEMENT,for the,for the,net_income,7.000000e+00,CNY,FUZZY_ANCHOR,100.000000,<NA>,2020-03-31 14:03:00+00:00
2,"Guangzhou Automobile Group Co., Ltd",2019 ANNUAL RESULTS ANNOUNCEMENT,last year. The sales revenue of the Group amou...,last year the sales revenue of the group amoun...,revenue,5.970400e+10,CNY,BLOCK9_ACCEPTED_SYNONYM,100.000000,<NA>,2020-03-31 14:03:00+00:00
3,"Guangzhou Automobile Group Co., Ltd",2019 ANNUAL RESULTS ANNOUNCEMENT,Borrowings – current: mainly due to the combin...,borrowings current mainly due to the combined ...,short_term_debt,2.000000e+09,CNY,FUZZY_ANCHOR,100.000000,<NA>,2020-03-31 14:03:00+00:00
4,"Guangzhou Automobile Group Co., Ltd",2019 ANNUAL RESULTS ANNOUNCEMENT,"the reporting period, the total vehicle produc...",the reporting period the total vehicle product...,vehicle_production_volume,2.613000e+06,CNY,FUZZY_ANCHOR,100.000000,<NA>,2020-03-31 14:03:00+00:00
5,"Guangzhou Automobile Group Co., Ltd",Annual Report 2019,"During the reporting period, the Group recorde...",during the reporting period the group recorded...,cost_of_revenue,5.718100e+10,CNY,FUZZY_ANCHOR,100.000000,<NA>,2020-04-27 10:11:00+00:00
6,"Guangzhou Automobile Group Co., Ltd",Annual Report 2019,Depreciation and amortisation (Notes,depreciation and amortisation,depreciation_and_amortisation,7.000000e+00,CNY,DETERMINISTIC_REGEX,99.000000,<NA>,2020-04-27 10:11:00+00:00
7,"Guangzhou Automobile Group Co., Ltd",Annual Report 2019,Depreciation (Notes,depreciation,depreciation_expense,7.000000e+00,CNY,DETERMINISTIC_REGEX,99.000000,<NA>,2020-04-27 10:11:00+00:00
8,"Guangzhou Automobile Group Co., Ltd",Annual Report 2019,used in note,used in,financing_cash_flow,2.300000e+01,CNY,FUZZY_ANCHOR,100.000000,<NA>,2020-04-27 10:11:00+00:00
9,"Guangzhou Automobile Group Co., Ltd",Annual Report 2019,Total gross profit amounted to approximately RMB,total gross profit amounted to approximately,gross_profit,2.523000e+09,CNY,BLOCK9_ACCEPTED_SYNONYM,100.000000,<NA>,2020-04-27 10:11:00+00:00


In [ ]:
# 13. HONG KONG POINT-IN-TIME ACCESS HELPERS

def hong_kong_fundamentals_as_of(
    dataframe,
    as_of_date,
    issuer_ids=None,
    security_ids=None,
    standard_concepts=None,
):
    if dataframe.empty:
        return dataframe.copy()

    cutoff = pd.Timestamp(as_of_date)

    cutoff = (
        cutoff.tz_localize("UTC")
        if cutoff.tzinfo is None
        else cutoff.tz_convert("UTC")
    )

    result = dataframe[
        pd.to_datetime(
            dataframe["available_datetime"],
            errors="coerce",
            utc=True,
        ) <= cutoff
    ].copy()

    if issuer_ids is not None:
        result = result[
            result["issuer_id"].isin(set(issuer_ids))
        ]

    if security_ids is not None:
        result = result[
            result["security_id"].isin(set(security_ids))
        ]

    if standard_concepts is not None:
        result = result[
            result["standard_concept"].isin(
                set(standard_concepts)
            )
        ]

    return result


def latest_hong_kong_fact_as_of(dataframe, as_of_date):
    result = hong_kong_fundamentals_as_of(
        dataframe,
        as_of_date,
    )

    if result.empty:
        return result

    return (
        result
        .sort_values(
            [
                "available_datetime",
                "document_priority",
                "document_id",
            ]
        )
        .drop_duplicates(
            [
                "security_id",
                "issuer_id",
                "standard_concept",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )

In [ ]:
# 14. HONG KONG COVERAGE AND QUALITY REPORTS

hong_kong_filing_coverage_report_df = pd.DataFrame({
    "metric": [
        "hong_kong_securities",
        "hong_kong_issuers",
        "resolved_hkex_securities",
        "resolved_unique_stock_codes",
        "historical_listings_retained",
        "unresolved_hkex_securities",
        "disclosures_discovered",
        "issuers_with_target_disclosures",
        "documents_with_text",
        "raw_account_candidates",
        "unique_cleaned_account_labels",
        "accepted_mapped_fact_rows",
        "standardised_fact_rows",
        "security_link_rate",
    ],
    "value": [
        len(hong_kong_security_universe_df),
        hong_kong_issuer_universe_df["issuer_id"].nunique(),
        len(hong_kong_resolved_securities_df),
        hong_kong_resolved_securities_df["stock_code"].nunique(),
        int(
            hong_kong_security_universe_df[
                "is_historical_listing"
            ].fillna(False).sum()
        ),
        len(hong_kong_unresolved_securities_df),
        len(hong_kong_filing_metadata_df),
        (
            hong_kong_filing_metadata_df["issuer_id"].nunique()
            if not hong_kong_filing_metadata_df.empty
            else 0
        ),
        len(hkex_document_text_df),
        len(hong_kong_account_candidates_raw_df),
        (
            hong_kong_account_candidates_raw_df[
                "account_label"
            ].nunique()
            if not hong_kong_account_candidates_raw_df.empty
            else 0
        ),
        len(hong_kong_fundamentals_mapped_df),
        len(hong_kong_fundamentals_standardised_df),
        (
            hong_kong_fundamentals_standardised_df[
                "security_id"
            ].notna().mean()
            if not hong_kong_fundamentals_standardised_df.empty
            else np.nan
        ),
    ],
})

hong_kong_standard_concept_coverage_df = (
    hong_kong_fundamentals_standardised_df
    .groupby(
        [
            "standard_concept",
            "statement_type",
            "core_tier",
        ],
        dropna=False,
    )
    .agg(
        fact_rows=("reported_value", "size"),
        issuer_count=("issuer_id", "nunique"),
        security_count=("security_id", "nunique"),
        document_count=("document_id", "nunique"),
        earliest_available=("available_datetime", "min"),
        latest_available=("available_datetime", "max"),
        numeric_fact_share=(
            "reported_value",
            lambda series: series.notna().mean(),
        ),
        unit_match_share=("unit_family_match", "mean"),
        deterministic_share=(
            "mapping_method",
            lambda series: (
                series.eq(
                    "DETERMINISTIC_REGEX"
                ).mean()
            ),
        ),
        average_mapping_score=("mapping_score", "mean"),
    )
    .reset_index()
    if not hong_kong_fundamentals_standardised_df.empty
    else pd.DataFrame()
)

observed = (
    hong_kong_fundamentals_standardised_df
    .groupby("standard_concept", dropna=False)
    .agg(
        issuer_coverage=("issuer_id", "nunique"),
        security_coverage=("security_id", "nunique"),
        filing_coverage=("document_id", "nunique"),
        fact_rows=("reported_value", "size"),
        first_available_datetime=("available_datetime", "min"),
        last_available_datetime=("available_datetime", "max"),
        numeric_fact_share=(
            "reported_value",
            lambda series: series.notna().mean(),
        ),
        deterministic_fact_share=(
            "mapping_method",
            lambda series: (
                series.eq(
                    "DETERMINISTIC_REGEX"
                ).mean()
            ),
        ),
        average_mapping_score=("mapping_score", "mean"),
    )
    .reset_index()
    if not hong_kong_fundamentals_standardised_df.empty
    else pd.DataFrame(
        columns=["standard_concept"]
    )
)

hong_kong_standard_concept_availability_df = (
    global_canonical_schema_df.merge(
        observed,
        on="standard_concept",
        how="left",
    )
)

for column in [
    "issuer_coverage",
    "security_coverage",
    "filing_coverage",
    "fact_rows",
]:
    hong_kong_standard_concept_availability_df[
        column
    ] = (
        hong_kong_standard_concept_availability_df[
            column
        ]
        .fillna(0)
        .astype(int)
    )

total_issuers = max(
    hong_kong_issuer_universe_df[
        "issuer_id"
    ].dropna().nunique(),
    1,
)

hong_kong_standard_concept_availability_df[
    "issuer_coverage_rate"
] = (
    hong_kong_standard_concept_availability_df[
        "issuer_coverage"
    ]
    / total_issuers
)

hong_kong_standard_concept_availability_df[
    "coverage_class"
] = pd.cut(
    hong_kong_standard_concept_availability_df[
        "issuer_coverage_rate"
    ],
    bins=[
        -0.001,
        0.10,
        0.30,
        0.60,
        0.80,
        1.00,
    ],
    labels=[
        "VERY_SPARSE",
        "SPARSE",
        "MODERATE",
        "HIGH",
        "VERY_HIGH",
    ],
)

mapping_method_counts = (
    hong_kong_fundamentals_standardised_df[
        "mapping_method"
    ]
    .value_counts(dropna=False)
    .rename_axis("mapping_method")
    .reset_index(name="fact_rows")
    if not hong_kong_fundamentals_standardised_df.empty
    else pd.DataFrame(
        columns=[
            "mapping_method",
            "fact_rows",
        ]
    )
)

hong_kong_mapping_method_report_df = (
    mapping_method_counts
)

hong_kong_mapping_quality_df = pd.DataFrame({
    "metric": [
        "canonical_standard_concepts",
        "hong_kong_source_mapping_rows",
        "unique_cleaned_account_labels",
        "observed_mapping_rows",
        "accepted_mapped_fact_rows",
        "selected_standardised_fact_rows",
        "unique_standard_concepts_observed",
        "deterministic_selected_fact_share",
        "average_selected_mapping_score",
        "mapping_review_queue_rows",
        "fuzzy_eligible_unique_labels",
        "low_quality_fuzzy_candidates_rejected",
        "unmapped_accounts_in_inventory",
        "automotive_extension_candidates",
    ],
    "value": [
        global_canonical_schema_df[
            "standard_concept"
        ].nunique(),
        len(hong_kong_standard_concept_dictionary_df),
        (
            hong_kong_account_candidates_raw_df[
                "account_label"
            ].nunique()
            if not hong_kong_account_candidates_raw_df.empty
            else 0
        ),
        len(hong_kong_observed_account_mapping_df),
        len(hong_kong_fundamentals_mapped_df),
        len(hong_kong_fundamentals_standardised_df),
        (
            hong_kong_fundamentals_standardised_df[
                "standard_concept"
            ].nunique()
            if not hong_kong_fundamentals_standardised_df.empty
            else 0
        ),
        (
            hong_kong_fundamentals_standardised_df[
                "mapping_method"
            ]
            .eq("DETERMINISTIC_REGEX")
            .mean()
            if not hong_kong_fundamentals_standardised_df.empty
            else np.nan
        ),
        (
            hong_kong_fundamentals_standardised_df[
                "mapping_score"
            ].mean()
            if not hong_kong_fundamentals_standardised_df.empty
            else np.nan
        ),
        len(hong_kong_mapping_review_queue_df),
        (
            int(
                hong_kong_fuzzy_label_quality_df[
                    "fuzzy_label_is_eligible"
                ].fillna(False).sum()
            )
            if not hong_kong_fuzzy_label_quality_df.empty
            else 0
        ),
        len(hong_kong_low_quality_fuzzy_rejections_df),
        len(hong_kong_unmapped_account_inventory_df),
        len(hong_kong_automotive_extension_candidates_df),
    ],
})


hong_kong_fuzzy_label_quality_report_df = (
    hong_kong_fuzzy_label_quality_df[
        "fuzzy_label_rejection_reason"
    ]
    .value_counts(dropna=False)
    .rename_axis("label_quality_class")
    .reset_index(name="unique_label_count")
    if not hong_kong_fuzzy_label_quality_df.empty
    else pd.DataFrame(
        columns=[
            "label_quality_class",
            "unique_label_count",
        ]
    )
)

display(hong_kong_filing_coverage_report_df)
display(hong_kong_mapping_quality_df)
display(hong_kong_mapping_method_report_df)
display(hong_kong_fuzzy_label_quality_report_df)
display(
    hong_kong_standard_concept_availability_df
    .sort_values(
        [
            "issuer_coverage",
            "fact_rows",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(100)
)

,metric,value
0,hong_kong_securities,20.0
1,hong_kong_issuers,20.0
2,resolved_hkex_securities,20.0
3,resolved_unique_stock_codes,20.0
4,historical_listings_retained,1.0
5,unresolved_hkex_securities,0.0
6,disclosures_discovered,90.0
7,issuers_with_target_disclosures,16.0
8,documents_with_text,90.0
9,raw_account_candidates,90253.0


,metric,value
0,canonical_standard_concepts,100.000000
1,hong_kong_source_mapping_rows,90.000000
2,unique_cleaned_account_labels,15093.000000
3,observed_mapping_rows,73084.000000
4,accepted_mapped_fact_rows,2832.000000
5,selected_standardised_fact_rows,534.000000
6,unique_standard_concepts_observed,39.000000
7,deterministic_selected_fact_share,0.052434
8,average_selected_mapping_score,99.786975
9,mapping_review_queue_rows,642.000000


,mapping_method,fact_rows
0,FUZZY_ANCHOR,408
1,BLOCK9_ACCEPTED_SYNONYM,98
2,DETERMINISTIC_REGEX,28


,label_quality_class,unique_label_count
0,ELIGIBLE,14675
1,UNAPPROVED_SINGLE_TOKEN,314
2,LATIN_LABEL_TOO_SHORT,87
3,GENERIC_STOPWORD,17


,standard_concept,statement_type,expected_period_type,expected_unit_family,core_tier,is_core,aggregation_policy,issuer_coverage,security_coverage,filing_coverage,fact_rows,first_available_datetime,last_available_datetime,numeric_fact_share,deterministic_fact_share,average_mapping_score,issuer_coverage_rate,coverage_class
0,revenue,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,16,16,70,70,2020-03-26 13:06:00+00:00,2026-04-30 09:10:00+00:00,1.0,0.042857,99.957143,0.80,HIGH
6,net_income,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,15,15,37,37,2020-03-26 13:06:00+00:00,2026-04-30 09:10:00+00:00,1.0,0.000000,100.000000,0.75,HIGH
37,trade_receivables,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,14,14,30,30,2020-03-27 13:25:00+00:00,2026-04-30 09:10:00+00:00,1.0,0.066667,99.933333,0.70,HIGH
2,gross_profit,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,13,13,54,54,2020-03-26 13:06:00+00:00,2026-04-30 09:10:00+00:00,1.0,0.000000,100.000000,0.65,HIGH
40,inventory,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,12,12,39,39,2020-03-27 08:31:00+00:00,2026-04-27 08:48:00+00:00,1.0,0.000000,100.000000,0.60,MODERATE
34,cash_and_cash_equivalents,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,10,10,29,29,2020-03-26 13:06:00+00:00,2026-04-30 09:10:00+00:00,1.0,0.000000,100.000000,0.50,MODERATE
52,total_liabilities,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,10,10,23,23,2020-03-26 13:06:00+00:00,2025-09-22 11:38:00+00:00,1.0,0.000000,100.000000,0.50,MODERATE
1,cost_of_revenue,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,9,9,45,45,2020-03-27 08:31:00+00:00,2026-04-30 09:10:00+00:00,1.0,0.000000,100.000000,0.45,MODERATE
31,total_assets,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,8,8,26,26,2020-03-26 13:06:00+00:00,2026-04-30 09:10:00+00:00,1.0,0.038462,99.961538,0.40,MODERATE
44,property_plant_equipment,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,8,8,24,24,2020-04-20 09:15:00+00:00,2025-04-28 10:19:00+00:00,1.0,0.000000,100.000000,0.40,MODERATE


In [ ]:
# 15. HONG KONG IN-MEMORY OUTPUT CONTRACT

hong_kong_block_data = {
    "hong_kong_security_candidate_df": hong_kong_security_candidate_df,
    "hong_kong_security_universe_df": hong_kong_security_universe_df,
    "hong_kong_fundamental_ingestion_universe_df": hong_kong_fundamental_ingestion_universe_df,
    "hong_kong_secondary_listing_only_df": hong_kong_secondary_listing_only_df,
    "hong_kong_fundamental_source_precedence_df": hong_kong_fundamental_source_precedence_df,
    "upstream_structured_issuer_coverage_df": upstream_structured_issuer_coverage_df,
    "preferred_structured_issuer_source_df": preferred_structured_issuer_source_df,
    "hong_kong_universe_validation_rejections_df": hong_kong_universe_validation_rejections_df,
    "hkex_known_automotive_validation_df": hkex_known_automotive_validation_df,
    "hong_kong_issuer_universe_df": hong_kong_issuer_universe_df,
    "hong_kong_universe_exclusions_df": hong_kong_universe_exclusions_df,

    "hkex_active_stock_list_df": hkex_active_stock_list_df,
    "hkex_active_stock_duplicate_report_df": hkex_active_stock_duplicate_report_df,
    "hkex_historical_listings_df": hkex_historical_listings_df,
    "hkex_name_aliases_df": hkex_name_aliases_df,
    "hkex_stock_id_overrides_df": hkex_stock_id_overrides_df,
    "hkex_stock_identifier_map_df": hkex_stock_identifier_map_df,
    "hkex_stock_identifier_resolution_log_df": hkex_stock_identifier_resolution_log_df,
    "hong_kong_security_bridge_df": hong_kong_security_bridge_df,
    "hong_kong_resolved_securities_df": hong_kong_resolved_securities_df,
    "hong_kong_unresolved_securities_df": hong_kong_unresolved_securities_df,

    "hkex_disclosures_discovered_df": hkex_disclosures_discovered_df,
    "hkex_disclosure_search_log_df": hkex_disclosure_search_log_df,
    "hong_kong_filing_metadata_df": hong_kong_filing_metadata_df,

    "hkex_document_download_log_df": hkex_document_download_log_df,
    "hkex_document_text_df": hkex_document_text_df,

    "hong_kong_source_account_mapping_proposed_df": hong_kong_source_account_mapping_proposed_df,
    "hong_kong_unsupported_concept_mappings_df": hong_kong_unsupported_concept_mappings_df,
    "hong_kong_source_account_mapping_df": hong_kong_source_account_mapping_df,
    "hong_kong_canonical_anchor_df": hong_kong_canonical_anchor_df,
    "hong_kong_standard_concept_dictionary_df": hong_kong_standard_concept_dictionary_df,
    "hong_kong_account_candidates_raw_df": hong_kong_account_candidates_raw_df,
    "hong_kong_observed_account_mapping_df": hong_kong_observed_account_mapping_df,
    "hong_kong_block9_registry_matches_df": hong_kong_block9_registry_matches_df,
    "block9_accepted_synonym_registry_df": block9_accepted_synonym_registry_df,
    "block9_synonym_registry_conflicts_df": block9_synonym_registry_conflicts_df,
    "block9_synonym_registry_status_df": block9_synonym_registry_status_df,
    "hong_kong_fuzzy_label_quality_df": hong_kong_fuzzy_label_quality_df,
    "hong_kong_fuzzy_label_quality_report_df": hong_kong_fuzzy_label_quality_report_df,
    "hong_kong_mapping_review_queue_df": hong_kong_mapping_review_queue_df,
    "hong_kong_low_quality_fuzzy_rejections_df": hong_kong_low_quality_fuzzy_rejections_df,

    "hong_kong_fundamentals_mapped_df": hong_kong_fundamentals_mapped_df,
    "hong_kong_mapping_alternatives_df": hong_kong_mapping_alternatives_df,
    "hong_kong_fundamentals_standardised_df": hong_kong_fundamentals_standardised_df,

    "hong_kong_unmapped_account_inventory_df": hong_kong_unmapped_account_inventory_df,
    "hong_kong_automotive_extension_candidates_df": hong_kong_automotive_extension_candidates_df,

    "hong_kong_filing_coverage_report_df": hong_kong_filing_coverage_report_df,
    "hong_kong_standard_concept_coverage_df": hong_kong_standard_concept_coverage_df,
    "hong_kong_standard_concept_availability_df": hong_kong_standard_concept_availability_df,
    "hong_kong_mapping_quality_df": hong_kong_mapping_quality_df,
    "hong_kong_mapping_method_report_df": hong_kong_mapping_method_report_df,
}

print("Hong Kong transformations complete.")

for name in [
    "hong_kong_filing_metadata_df",
    "hkex_document_text_df",
    "hong_kong_account_candidates_raw_df",
    "hong_kong_fundamentals_standardised_df",
    "hong_kong_standard_concept_availability_df",
]:
    print(f"  {name}: {len(hong_kong_block_data[name]):,} rows")

Hong Kong transformations complete.
  hong_kong_filing_metadata_df: 90 rows
  hkex_document_text_df: 90 rows
  hong_kong_account_candidates_raw_df: 90,253 rows
  hong_kong_fundamentals_standardised_df: 534 rows
  hong_kong_standard_concept_availability_df: 100 rows


In [ ]:
# 16. CHINA SETTINGS, DIRECTORIES AND UPSTREAM INPUTS

# Reuse PROJECT_ROOT and DATA_ROOT initialised above.


BLOCK_7_OUTPUT_DIR = DATA_ROOT / "interim" / "block_7"
CNINFO_CACHE_DIR = DATA_ROOT / "external" / "cninfo"
CNINFO_SEARCH_CACHE_DIR = CNINFO_CACHE_DIR / "search"
CNINFO_METADATA_CACHE_DIR = CNINFO_CACHE_DIR / "metadata"
CNINFO_ANNOUNCEMENT_CACHE_DIR = CNINFO_CACHE_DIR / "announcements"
CNINFO_DOCUMENT_CACHE_DIR = CNINFO_CACHE_DIR / "documents"
CNINFO_PDF_CACHE_DIR = CNINFO_DOCUMENT_CACHE_DIR
CNINFO_TEXT_CACHE_DIR = CNINFO_CACHE_DIR / "text"

for directory in [
    BLOCK_7_OUTPUT_DIR,
    CNINFO_CACHE_DIR,
    CNINFO_SEARCH_CACHE_DIR,
    CNINFO_METADATA_CACHE_DIR,
    CNINFO_ANNOUNCEMENT_CACHE_DIR,
    CNINFO_DOCUMENT_CACHE_DIR,
    CNINFO_TEXT_CACHE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

BLOCK_2_MANIFEST_PATH = DATA_ROOT / "interim" / "block_2" / "block_2_manifest.json"
BLOCK_3_MANIFEST_PATH = DATA_ROOT / "interim" / "block_3" / "block_3_manifest.json"
BLOCK_4_MANIFEST_PATH = DATA_ROOT / "interim" / "block_4" / "block_4_manifest.json"
BLOCK_5_MANIFEST_PATH = DATA_ROOT / "interim" / "block_5" / "block_5_manifest.json"
BLOCK_6_MANIFEST_PATH = DATA_ROOT / "interim" / "block_6" / "block_6_manifest.json"
BLOCK_7_MANIFEST_PATH = DATA_ROOT / "interim" / "block_7" / "block_7_manifest.json"
BLOCK_7_MANIFEST_PATH = BLOCK_7_OUTPUT_DIR / "block_7_manifest.json"

TARGET_ETFS = {"DRIV", "CARZ", "IDRV", "KARS"}
MAINLAND_EXCHANGES = {"SSE", "SZSE", "BSE"}

DISCOVERY_START_DATE = "2019-01-01"
DISCOVERY_END_DATE = datetime.now(timezone.utc).date().isoformat()

MAX_SECURITIES = None
PERSIST_BLOCK_7_OUTPUTS = True
OVERWRITE_PERSISTED_OUTPUTS = True

MIN_CNINFO_FILINGS = 1
MIN_CNINFO_STANDARD_CONCEPTS = 5
MIN_UPSTREAM_FACT_ROWS = 20
MIN_UPSTREAM_UNIQUE_CONCEPTS = 5

FUZZY_ACCEPT_SCORE = 93.0
FUZZY_REVIEW_SCORE = 85.0


# CNINFO entity matching is deliberately conservative. Automatic fuzzy-name
# matches are accepted only where the candidate and CNINFO names use compatible
# scripts and the match is unique. The editable override table below handles
# known dual-listed automotive groups and other validated relationships.
CNINFO_ENTITY_NAME_ACCEPT_SCORE = 96.0
CNINFO_ENTITY_NAME_REVIEW_SCORE = 88.0

# Consolidated source precedence. CNINFO is normally supplementary when SEC or
# HKEX already provides usable consolidated fundamentals.
CHINA_PRIMARY_SOURCE_PRECEDENCE = {
    "SEC": 1,
    "HKEX": 2,
    "CNINFO": 3,
    "EDINET": 4,
    "DART": 5,
}

# Curated relationship seeds. These are not used unless the ETF issuer matches
# the supplied issuer-name regex or ticker. Each row should identify a verified
# Mainland listed entity and describe its relationship to the ETF economic
# issuer. Users can extend this table without changing the pipeline.
CNINFO_ENTITY_OVERRIDES = [
    {
        "issuer_name_regex": r"\bBYD\b",
        "security_ticker_regex": r"^(1211|BYDDF|BYDDY|002594)$",
        "cninfo_stock_code": "002594",
        "cninfo_exchange": "SZSE",
        "cninfo_entity_relationship": "SAME_ISSUER_DUAL_LISTING",
        "override_confidence": 1.00,
    },
    {
        "issuer_name_regex": r"GREAT WALL|GWMOTOR",
        "security_ticker_regex": r"^(2333|GWLLF|601633)$",
        "cninfo_stock_code": "601633",
        "cninfo_exchange": "SSE",
        "cninfo_entity_relationship": "SAME_ISSUER_DUAL_LISTING",
        "override_confidence": 1.00,
    },
    {
        "issuer_name_regex": r"GUANGZHOU AUTOMOBILE|\bGAC\b",
        "security_ticker_regex": r"^(2238|GNZUF|601238)$",
        "cninfo_stock_code": "601238",
        "cninfo_exchange": "SSE",
        "cninfo_entity_relationship": "SAME_ISSUER_DUAL_LISTING",
        "override_confidence": 1.00,
    },
    {
        "issuer_name_regex": r"WEICHAI",
        "security_ticker_regex": r"^(2338|WEICY|000338)$",
        "cninfo_stock_code": "000338",
        "cninfo_exchange": "SZSE",
        "cninfo_entity_relationship": "SAME_ISSUER_DUAL_LISTING",
        "override_confidence": 1.00,
    },
    {
        "issuer_name_regex": r"FUYAO",
        "security_ticker_regex": r"^(3606|FYAAF|600660)$",
        "cninfo_stock_code": "600660",
        "cninfo_exchange": "SSE",
        "cninfo_entity_relationship": "SAME_ISSUER_DUAL_LISTING",
        "override_confidence": 1.00,
    },
    {
        "issuer_name_regex": r"CHANGAN",
        "security_ticker_regex": r"^(000625|200625)$",
        "cninfo_stock_code": "000625",
        "cninfo_exchange": "SZSE",
        "cninfo_entity_relationship": "SAME_ISSUER_MULTIPLE_SHARE_CLASS",
        "override_confidence": 1.00,
    },
]

# Concepts and labels likely to provide China-specific operating enrichment.
CHINA_LOCAL_ONLY_STANDARD_CONCEPTS = {
    "vehicle_sales_volume",
    "vehicle_production_volume",
    "government_grants",
    "government_subsidies",
    "construction_in_progress",
    "restricted_cash",
    "related_party_receivables",
    "related_party_payables",
}


# Optional user-maintained relationship catalogue. When present, this CSV is
# appended to the built-in validated seeds. It must use the schema documented
# below and may contain several CNINFO entities for one issuer.
CHINA_RELATIONSHIP_CATALOGUE_PATH = (
    PROJECT_ROOT
    / "config"
    / "china_issuer_relationships.csv"
)

CHINA_RELATIONSHIP_CATALOGUE_COLUMNS = [
    "issuer_id",
    "issuer_name_regex",
    "security_ticker_regex",
    "cninfo_stock_code",
    "cninfo_exchange",
    "cninfo_entity_relationship",
    "cninfo_entity_role",
    "relationship_valid_from",
    "relationship_valid_to",
    "relationship_source",
    "relationship_notes",
    "override_confidence",
]

VALID_CNINFO_ENTITY_RELATIONSHIPS = {
    "SAME_ISSUER_DUAL_LISTING",
    "SAME_ISSUER_MAINLAND_LISTING",
    "SAME_ISSUER_MULTIPLE_SHARE_CLASS",
    "MAINLAND_PARENT",
    "MAINLAND_SUBSIDIARY",
    "OPERATING_COMPANY",
    "ASSOCIATE_OR_JV",
    "LISTED_SUBSIDIARY",
    "SPINOFF",
}

VALID_CNINFO_ENTITY_ROLES = {
    "PRIMARY_OPERATING_ENTITY",
    "PARENT_ENTITY",
    "MANUFACTURING_ENTITY",
    "BATTERY_ENTITY",
    "COMPONENT_ENTITY",
    "COMMERCIAL_VEHICLE_ENTITY",
    "FINANCE_ENTITY",
    "TECHNOLOGY_ENTITY",
    "LISTED_SUBSIDIARY",
    "ASSOCIATE_OR_JV",
    "OTHER_CONFIRMED_ENTITY",
}


CHINA_FACT_CATEGORY_RULES = {
    "OPERATING_STATISTICS": {
        "vehicle_sales_volume",
        "vehicle_production_volume",
    },
    "GOVERNMENT_SUPPORT": {
        "government_grants",
        "government_subsidies",
    },
    "CAPITAL_PROJECTS": {
        "capital_expenditure",
        "construction_in_progress",
    },
    "LIQUIDITY_RESTRICTIONS": {
        "restricted_cash",
    },
    "RELATED_PARTIES": {
        "related_party_receivables",
        "related_party_payables",
    },
}

CHINA_LOCAL_DISCLOSURE_LABEL_PATTERN = re.compile(
    r"(政府补助|补贴|产量|销量|新能源汽车|车型|产能|在建工程|"
    r"受限资金|受限货币资金|关联方|担保|质押|研发项目|募投项目|"
    r"government grant|subsid|production volume|sales volume|capacity|"
    r"restricted cash|related party|guarantee|pledge)",
    flags=re.IGNORECASE,
)

MAX_DOCUMENTS_PER_SECURITY = 20
MAX_ANNOUNCEMENTS_PER_SECURITY = None
MAX_DOCUMENTS_TO_DOWNLOAD = None
MAX_PDF_PAGES = None
MAX_RETRIES = 5
REQUEST_INTERVAL_SECONDS = 0.35
REQUEST_TIMEOUT_SECONDS = 120
SEARCH_PAGE_SIZE = 30

CNINFO_BASE_URL = "https://www.cninfo.com.cn"
CNINFO_SEARCH_URL = f"{CNINFO_BASE_URL}/new/hisAnnouncement/query"
CNINFO_ANNOUNCEMENT_ENDPOINT = CNINFO_SEARCH_URL

CNINFO_STOCK_METADATA_ENDPOINTS = {
    "SZSE": f"{CNINFO_BASE_URL}/new/data/szse_stock.json",
    "SSE": f"{CNINFO_BASE_URL}/new/data/shse_stock.json",
    "BSE": f"{CNINFO_BASE_URL}/new/data/bjse_stock.json",
}

CNINFO_STATIC_BASE_URL = "https://static.cninfo.com.cn/"

TARGET_CATEGORIES = {
    "category_ndbg_szsh": "ANNUAL_REPORT",
    "category_bndbg_szsh": "SEMIANNUAL_REPORT",
    "category_yjdbg_szsh": "Q1_REPORT",
    "category_sjdbg_szsh": "Q3_REPORT",
}

TARGET_TITLE_PATTERN = re.compile(
    r"(年度报告|年报|半年度报告|半年报|第一季度报告|第三季度报告|"
    r"annual report|semi.?annual report|first quarter report|third quarter report)",
    flags=re.IGNORECASE,
)

SUMMARY_TITLE_PATTERN = re.compile(
    r"(摘要|summary)",
    flags=re.IGNORECASE,
)


def load_manifest_tables(manifest_path, required_names):
    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Missing upstream manifest: {manifest_path}"
        )

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    records = {
        item["table_name"]: item
        for item in manifest.get("tables", [])
    }

    missing = set(required_names).difference(records)

    if missing:
        raise RuntimeError(
            f"{manifest_path.name} is missing required tables: "
            f"{sorted(missing)}"
        )

    loaded = {}

    for name in required_names:
        path = Path(records[name]["path"])

        if not path.exists():
            raise FileNotFoundError(path)

        loaded[name] = pd.read_parquet(path)

    return loaded, manifest


def load_optional_manifest_tables(manifest_path, requested_names):
    if not manifest_path.exists():
        return {}, {
            "manifest_path": str(manifest_path),
            "status": "MANIFEST_NOT_FOUND",
            "tables": [],
        }

    with manifest_path.open("r", encoding="utf-8") as file:
        manifest = json.load(file)

    records = {
        item["table_name"]: item
        for item in manifest.get("tables", [])
    }

    loaded = {}

    for name in requested_names:
        record = records.get(name)

        if record is None:
            continue

        path = Path(record["path"])

        if path.exists():
            loaded[name] = pd.read_parquet(path)

    manifest["manifest_path"] = str(manifest_path)
    manifest["status"] = "LOADED"

    return loaded, manifest


block_2_inputs, block_2_manifest = load_manifest_tables(
    BLOCK_2_MANIFEST_PATH,
    {
        "security_master_df",
        "issuer_master_df",
        "security_identifier_history_df",
        "security_listing_history_df",
        "security_etf_membership_intervals_df",
    },
)

security_master_df = block_2_inputs["security_master_df"]
issuer_master_df = block_2_inputs["issuer_master_df"]
security_identifier_history_df = block_2_inputs["security_identifier_history_df"]
security_listing_history_df = block_2_inputs["security_listing_history_df"]
security_etf_membership_intervals_df = block_2_inputs[
    "security_etf_membership_intervals_df"
]

required_security_master_columns = {
    "security_id",
    "issuer_id",
    "issuer_name",
    "ticker",
    "listing_ticker",
    "exchange",
    "listing_country",
    "share_class",
}

missing_security_master_columns = required_security_master_columns.difference(
    security_master_df.columns
)

if missing_security_master_columns:
    raise RuntimeError(
        "Block 2 security_master_df is missing listing-identity fields: "
        f"{sorted(missing_security_master_columns)}. "
        "Run the regenerated Block 2 schema version 1.2.0 first."
    )

block_4_inputs, block_4_manifest = load_manifest_tables(
    BLOCK_4_MANIFEST_PATH,
    {"europe_standard_concept_dictionary_df"},
)

global_canonical_schema_df = (
    block_4_inputs["europe_standard_concept_dictionary_df"][
        [
            "standard_concept",
            "statement_type",
            "expected_period_type",
            "expected_unit_family",
            "core_tier",
            "is_core",
            "aggregation_policy",
        ]
    ]
    .drop_duplicates("standard_concept")
    .reset_index(drop=True)
)

block_3_inputs, block_3_manifest = load_optional_manifest_tables(
    BLOCK_3_MANIFEST_PATH,
    {
        "sec_fundamentals_security_linked_df",
        "sec_fundamentals_standardised_df",
    },
)

block_5_inputs, block_5_manifest = load_optional_manifest_tables(
    BLOCK_5_MANIFEST_PATH,
    {"japan_fundamentals_standardised_df"},
)

block_6_inputs, block_6_manifest = load_optional_manifest_tables(
    BLOCK_6_MANIFEST_PATH,
    {"korea_fundamentals_standardised_df"},
)

block_7_inputs = {
    "hong_kong_fundamentals_standardised_df": (
        hong_kong_fundamentals_standardised_df
    )
}
block_7_manifest = {
    "block": 7,
    "status": "IN_MEMORY_HKEX_COMPONENT",
}

print("Block 2 schema version:", block_2_manifest.get("schema_version"))
print("Security Master rows:", len(security_master_df))
print("ETF interval rows:", len(security_etf_membership_intervals_df))
print(
    "Canonical concepts:",
    global_canonical_schema_df["standard_concept"].nunique(),
)
print("Block 7 output directory:", BLOCK_7_OUTPUT_DIR)

Block 2 schema version: 1.2.0
Security Master rows: 512
ETF interval rows: 8028
Canonical concepts: 100
Block 7 output directory: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_7


In [ ]:
# 17. BUILD THE CROSS-LISTING CHINESE ECONOMIC-ISSUER UNIVERSE

def first_existing_column(dataframe, candidates):
    return next(
        (
            column
            for column in candidates
            if column in dataframe.columns
        ),
        None,
    )


def clean_text(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    return text if text else pd.NA


def normalise_company_name(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).upper()

    text = re.sub(
        r"\b(HOLDINGS?|LIMITED|LTD|CORPORATION|CORP|COMPANY|CO|"
        r"INCORPORATED|INC|PLC|GROUP)\b",
        " ",
        text,
    )

    text = re.sub(
        r"[^A-Z0-9\u4e00-\u9fff]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text if text else pd.NA


def extract_six_digit_stock_code(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip().upper()

    if re.fullmatch(r"\d{6}", text):
        return text

    match = re.search(
        r"(?<!\d)(\d{6})(?!\d)",
        text,
    )

    return match.group(1) if match else pd.NA


def isin_country_prefix(value):
    if pd.isna(value):
        return pd.NA

    text = re.sub(
        r"[^A-Z0-9]",
        "",
        str(value).upper(),
    )

    if re.fullmatch(
        r"[A-Z]{2}[A-Z0-9]{9}\d",
        text,
    ):
        return text[:2]

    return pd.NA


def infer_primary_listing_source(exchange, listing_country):
    if pd.notna(exchange):
        exchange = str(exchange)

        if exchange in {"NYSE", "NASDAQ", "NYSE_ARCA"}:
            return "SEC"

        if exchange == "HKEX":
            return "HKEX"

        if exchange in MAINLAND_EXCHANGES:
            return "CNINFO"

        if exchange == "TSE":
            return "EDINET"

        if exchange in {"KRX", "KOSDAQ"}:
            return "DART"

    if pd.notna(listing_country):
        country = str(listing_country)

        return {
            "US": "SEC",
            "HK": "HKEX",
            "CN": "CNINFO",
            "JP": "EDINET",
            "KR": "DART",
        }.get(country, "UNRESOLVED")

    return "UNRESOLVED"


def join_unique(values):
    cleaned = sorted(
        set(
            str(value)
            for value in values.dropna()
            if str(value).strip()
        )
    )

    return " | ".join(cleaned) if cleaned else pd.NA


etf_column = first_existing_column(
    security_etf_membership_intervals_df,
    ["etf", "etf_ticker", "fund", "fund_ticker"],
)

if etf_column is None:
    raise RuntimeError(
        "Block 2 security_etf_membership_intervals_df has no ETF column."
    )

intervals = security_etf_membership_intervals_df.copy()

intervals["etf_ticker"] = (
    intervals[etf_column]
    .astype("string")
    .str.upper()
    .str.strip()
)

target_etf_intervals_df = (
    intervals[
        intervals["etf_ticker"].isin(TARGET_ETFS)
    ]
    .copy()
    .reset_index(drop=True)
)

if target_etf_intervals_df.empty:
    raise RuntimeError(
        "No point-in-time memberships were found for DRIV, CARZ, IDRV or KARS."
    )

required_interval_columns = {
    "security_id",
    "issuer_id",
    "etf_ticker",
}

missing_interval_columns = required_interval_columns.difference(
    target_etf_intervals_df.columns
)

if missing_interval_columns:
    raise RuntimeError(
        "Block 2 ETF intervals are missing required identity columns: "
        f"{sorted(missing_interval_columns)}"
    )

target_security_ids = set(
    target_etf_intervals_df["security_id"].dropna()
)

target_etf_security_master_df = (
    security_master_df[
        security_master_df["security_id"].isin(target_security_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

for column in [
    "issuer_country",
    "country",
    "listing_country",
    "exchange",
    "ticker",
    "listing_ticker",
    "isin",
    "issuer_name",
    "security_name",
]:
    if column not in target_etf_security_master_df.columns:
        target_etf_security_master_df[column] = pd.NA

target_etf_security_master_df["isin_country"] = (
    target_etf_security_master_df["isin"]
    .map(isin_country_prefix)
)

target_etf_security_master_df["candidate_stock_code"] = (
    target_etf_security_master_df["listing_ticker"]
    .map(extract_six_digit_stock_code)
    .combine_first(
        target_etf_security_master_df["ticker"]
        .map(extract_six_digit_stock_code)
    )
)

target_etf_security_master_df["economic_issuer_name_normalised"] = (
    target_etf_security_master_df["issuer_name"]
    .map(normalise_company_name)
    .combine_first(
        target_etf_security_master_df["security_name"]
        .map(normalise_company_name)
    )
)

target_etf_security_master_df["china_economic_issuer_evidence"] = np.select(
    [
        target_etf_security_master_df["issuer_country"]
        .eq("CN")
        .fillna(False),

        target_etf_security_master_df["country"]
        .eq("CN")
        .fillna(False),

        (
            target_etf_security_master_df["listing_country"]
            .isin({"CN", "HK"})
            .fillna(False)
            & target_etf_security_master_df["isin_country"]
            .isin({"CN", "HK", "KY"})
            .fillna(False)
        ),

        target_etf_security_master_df["isin_country"]
        .eq("CN")
        .fillna(False),

        target_etf_security_master_df["exchange"]
        .isin(MAINLAND_EXCHANGES)
        .fillna(False),
    ],
    [
        "ISSUER_COUNTRY_CN",
        "SECURITY_COUNTRY_CN",
        "CHINESE_OFFSHORE_OR_HK_LISTING",
        "CN_ISIN",
        "MAINLAND_EXCHANGE",
    ],
    default="NO_CHINA_EVIDENCE",
)

china_issuer_security_universe_df = (
    target_etf_security_master_df[
        target_etf_security_master_df[
            "china_economic_issuer_evidence"
        ].ne("NO_CHINA_EVIDENCE")
    ]
    .copy()
    .reset_index(drop=True)
)

china_issuer_security_universe_df["primary_listing_source_system"] = (
    china_issuer_security_universe_df.apply(
        lambda row: infer_primary_listing_source(
            row.get("exchange"),
            row.get("listing_country"),
        ),
        axis=1,
    )
)

china_security_ids = set(
    china_issuer_security_universe_df["security_id"].dropna()
)

china_issuer_etf_membership_intervals_df = (
    target_etf_intervals_df[
        target_etf_intervals_df["security_id"].isin(china_security_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

# Build point-in-time security membership summaries from the interval table.
# Prefix all derived fields to avoid collisions with similarly named columns
# already present in security_master_df.
security_membership_summary_df = (
    china_issuer_etf_membership_intervals_df
    .groupby("security_id", dropna=False)
    .agg(
        membership_etf_count=("etf_ticker", "nunique"),
        membership_etf_tickers=("etf_ticker", join_unique),
        membership_first_date=("membership_valid_from", "min"),
        membership_last_date=("membership_valid_to", "max"),
        membership_row_count=("etf_ticker", "size"),
    )
    .reset_index()
)

china_issuer_security_universe_df = (
    china_issuer_security_universe_df
    .drop(
        columns=[
            "membership_etf_count",
            "membership_etf_tickers",
            "membership_first_date",
            "membership_last_date",
            "membership_row_count",
        ],
        errors="ignore",
    )
    .merge(
        security_membership_summary_df,
        on="security_id",
        how="left",
        validate="1:1",
    )
)

required_security_membership_columns = {
    "membership_etf_count",
    "membership_etf_tickers",
    "membership_first_date",
    "membership_last_date",
    "membership_row_count",
}

missing_security_membership_columns = (
    required_security_membership_columns.difference(
        china_issuer_security_universe_df.columns
    )
)

if missing_security_membership_columns:
    raise RuntimeError(
        "Step 3 failed to attach ETF membership summaries to the Chinese "
        "security universe: "
        f"{sorted(missing_security_membership_columns)}"
    )

# Build the issuer-level universe directly from the authoritative membership
# intervals, then overlay descriptive security-master information. This avoids
# relying on potentially suffixed or pre-existing security-level aggregation
# columns.
issuer_membership_summary_df = (
    china_issuer_etf_membership_intervals_df
    .groupby("issuer_id", dropna=False)
    .agg(
        issuer_membership_security_count=("security_id", "nunique"),
        issuer_membership_security_ids=("security_id", join_unique),
        issuer_etf_count=("etf_ticker", "nunique"),
        issuer_etf_tickers=("etf_ticker", join_unique),
        issuer_first_membership_date=("membership_valid_from", "min"),
        issuer_last_membership_date=("membership_valid_to", "max"),
        issuer_membership_row_count=("etf_ticker", "size"),
    )
    .reset_index()
)

issuer_security_attributes_df = (
    china_issuer_security_universe_df
    .groupby("issuer_id", dropna=False)
    .agg(
        issuer_name=("issuer_name", "first"),
        issuer_country=("issuer_country", "first"),
        listed_security_count=("security_id", "nunique"),
        listed_security_ids=("security_id", join_unique),
        listing_exchanges=("exchange", join_unique),
        listing_countries=("listing_country", join_unique),
        primary_listing_sources=(
            "primary_listing_source_system",
            join_unique,
        ),
        china_economic_issuer_evidence=(
            "china_economic_issuer_evidence",
            join_unique,
        ),
    )
    .reset_index()
)

china_economic_issuer_universe_df = (
    issuer_membership_summary_df.merge(
        issuer_security_attributes_df,
        on="issuer_id",
        how="left",
        validate="1:1",
    )
)

china_economic_issuer_universe_df["security_count"] = (
    china_economic_issuer_universe_df[
        "issuer_membership_security_count"
    ]
)

china_economic_issuer_universe_df["security_ids"] = (
    china_economic_issuer_universe_df[
        "issuer_membership_security_ids"
    ]
)

china_economic_issuer_universe_df["etf_count"] = (
    china_economic_issuer_universe_df[
        "issuer_etf_count"
    ]
)

china_economic_issuer_universe_df["etf_tickers"] = (
    china_economic_issuer_universe_df[
        "issuer_etf_tickers"
    ]
)

china_economic_issuer_universe_df["first_membership_date"] = (
    china_economic_issuer_universe_df[
        "issuer_first_membership_date"
    ]
)

china_economic_issuer_universe_df["last_membership_date"] = (
    china_economic_issuer_universe_df[
        "issuer_last_membership_date"
    ]
)

duplicate_security_columns = (
    china_issuer_security_universe_df.columns[
        china_issuer_security_universe_df.columns.duplicated()
    ]
    .tolist()
)

duplicate_issuer_columns = (
    china_economic_issuer_universe_df.columns[
        china_economic_issuer_universe_df.columns.duplicated()
    ]
    .tolist()
)

if duplicate_security_columns or duplicate_issuer_columns:
    raise RuntimeError(
        "Duplicate columns remain in the Chinese universe tables. "
        f"Security duplicates: {duplicate_security_columns}; "
        f"issuer duplicates: {duplicate_issuer_columns}"
    )

china_issuer_universe_quality_df = pd.DataFrame({
    "metric": [
        "target_etfs",
        "target_etf_membership_rows",
        "target_etf_unique_securities",
        "china_economic_issuers",
        "china_listed_securities",
        "china_etf_membership_rows",
        "us_listed_chinese_securities",
        "hong_kong_listed_chinese_securities",
        "mainland_listed_chinese_securities",
        "unresolved_listing_source_securities",
        "issuer_security_count_reconciliation",
    ],
    "value": [
        len(TARGET_ETFS),
        len(target_etf_intervals_df),
        target_etf_security_master_df["security_id"].nunique(),
        china_economic_issuer_universe_df["issuer_id"].nunique(),
        china_issuer_security_universe_df["security_id"].nunique(),
        len(china_issuer_etf_membership_intervals_df),
        china_issuer_security_universe_df[
            "primary_listing_source_system"
        ].eq("SEC").sum(),
        china_issuer_security_universe_df[
            "primary_listing_source_system"
        ].eq("HKEX").sum(),
        china_issuer_security_universe_df[
            "primary_listing_source_system"
        ].eq("CNINFO").sum(),
        china_issuer_security_universe_df[
            "primary_listing_source_system"
        ].eq("UNRESOLVED").sum(),
        int(
            china_economic_issuer_universe_df[
                "security_count"
            ].sum()
        ),
    ],
})

if china_economic_issuer_universe_df.empty:
    raise RuntimeError(
        "No Chinese economic issuers were identified in the four target ETFs."
    )

print(
    "Chinese economic issuers:",
    china_economic_issuer_universe_df["issuer_id"].nunique(),
)

print(
    "Chinese listed securities across all venues:",
    china_issuer_security_universe_df["security_id"].nunique(),
)

print(
    "ETF membership rows linked to Chinese issuers:",
    len(china_issuer_etf_membership_intervals_df),
)

print(
    "Duplicate columns in security universe:",
    len(duplicate_security_columns),
)

print(
    "Duplicate columns in issuer universe:",
    len(duplicate_issuer_columns),
)

display(china_issuer_universe_quality_df)
display(china_issuer_security_universe_df)
display(china_economic_issuer_universe_df)

Chinese economic issuers: 103
Chinese listed securities across all venues: 79
ETF membership rows linked to Chinese issuers: 1035
Duplicate columns in security universe: 0
Duplicate columns in issuer universe: 0


,metric,value
0,target_etfs,4
1,target_etf_membership_rows,8028
2,target_etf_unique_securities,512
3,china_economic_issuers,103
4,china_listed_securities,79
5,china_etf_membership_rows,1035
6,us_listed_chinese_securities,4
7,hong_kong_listed_chinese_securities,17
8,mainland_listed_chinese_securities,1
9,unresolved_listing_source_securities,57


,security_id,issuer_id,security_name,issuer_name,ticker,listing_ticker,isin,cusip,lei,issuer_country,country,listing_country,exchange,mic,currency,share_class,security_type,listing_status,mapping_status,listing_mapping_status,security_identity_type,security_identity_value,listing_resolution_method,listing_resolution_confidence,listing_evidence_count,listing_exchange_variant_count,first_observed_snapshot_date,last_observed_snapshot_date,first_public_available_date,last_public_available_date,observation_count,etf_count,source_system,schema_version,isin_country,candidate_stock_code,economic_issuer_name_normalised,china_economic_issuer_evidence,primary_listing_source_system,membership_etf_count,membership_etf_tickers,membership_first_date,membership_last_date,membership_row_count
0,GAS_000EFF8BBFFA00B37D9E,GAI_D8F28DDEBA5977F9D69A,XIAMEN TUNGSTEN CO LTD-A COMMON STOCK,"Xiamen Tungsten Co., Ltd.",None,None,CNE000001D15,None,300300SEC2FOC4PL5N49,CN,CN,None,None,None,CNY,None,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,UNRESOLVED,ISIN,CNE000001D15,UNRESOLVED,0.00,0,0,2021-06-30,2026-03-31,2021-08-30,2026-05-29,20,1,SEC_NPORT,1.2.0,CN,<NA>,XIAMEN TUNGSTEN,ISSUER_COUNTRY_CN,UNRESOLVED,1,KARS,2021-08-30,2026-05-29,20
1,GAS_025E74C831EDA3481EFF,GAI_DFA494ABBAA0C6DE156F,Black Sesame International Holding Ltd,Black Sesame International Hol,2533,02533,KYG129301068,000000000,None,CN,CN,HK,HKEX,XHKG,HKD,ORDINARY_OR_H_SHARE,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,HIGH_CONFIDENCE,ISIN,KYG129301068,HKD_CONTEXT_NUMERIC_TICKER,0.92,4,1,2025-06-30,2026-03-31,2025-08-25,2026-05-21,4,1,SEC_NPORT,1.2.0,KY,<NA>,BLACK SESAME INTERNATIONAL HOL,ISSUER_COUNTRY_CN,HKEX,1,CARZ,2025-08-25,2026-05-21,4
2,GAS_02F70E8F8EF8D010360A,GAI_EF3155D5B48653A2BD33,SHANGHAI BELLING CO LTD-A COMMON STOCK,"Shanghai Belling Corp., Ltd",None,None,CNE000000XB6,None,None,CN,CN,None,None,None,CNY,None,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,UNRESOLVED,ISIN,CNE000000XB6,UNRESOLVED,0.00,0,0,2019-09-30,2021-03-31,2019-11-27,2021-08-30,7,1,SEC_NPORT,1.2.0,CN,<NA>,SHANGHAI BELLING,ISSUER_COUNTRY_CN,UNRESOLVED,1,KARS,2019-11-27,2021-08-30,7
3,GAS_0AD6D2407B9614C625F3,GAI_5586FB58001D48BE17BA,XPeng Inc,XPENG INC.,9868,09868,KYG982AW1003,000000000,549300TZNMIREMWQU857,KY,KY,HK,HKEX,XHKG,HKD,ORDINARY_OR_H_SHARE,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,HIGH_CONFIDENCE,ISIN,KYG982AW1003,HKD_CONTEXT_NUMERIC_TICKER,0.92,2,1,2022-01-31,2026-04-30,2022-03-29,2026-06-25,35,3,SEC_NPORT,1.2.0,KY,<NA>,XPENG,CHINESE_OFFSHORE_OR_HK_LISTING,HKEX,3,CARZ | IDRV | KARS,2022-03-29,2026-06-25,35
4,GAS_0B0421EEBE72643EFEDF,GAI_7C516DB5B777B30FA5F4,NINGBO JOYSON ELECTRONIC -A COMMON STOCK,NINGBO JOYSON ELECTRONIC CORP.,None,None,CNE000000DJ1,None,300300XPY07RUN7V4863,CN,CN,None,None,None,CNY,None,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,UNRESOLVED,ISIN,CNE000000DJ1,UNRESOLVED,0.00,0,0,2019-09-30,2020-06-30,2019-11-27,2020-08-27,4,1,SEC_NPORT,1.2.0,CN,<NA>,NINGBO JOYSON ELECTRONIC,ISSUER_COUNTRY_CN,UNRESOLVED,1,KARS,2019-11-27,2020-11-24,4
5,GAS_0D9FFC7DEC8471103C4F,GAI_993FF914B41A16496DF8,BEIJING EASPRING MATERIAL-A COMMON STOCK,"Beijing Easpring Material Technology CO., LTD.",None,None,CNE100000NN1,None,None,CN,CN,None,None,None,CNY,None,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,UNRESOLVED,ISIN,CNE100000NN1,UNRESOLVED,0.00,0,0,2021-06-30,2026-03-31,2021-08-30,2026-05-29,20,1,SEC_NPORT,1.2.0,CN,<NA>,BEIJING EASPRING MATERIAL TECHNOLOGY,ISSUER_COUNTRY_CN,UNRESOLVED,1,KARS,2021-08-30,2026-05-29,20
6,GAS_0F04944896B3684D09B0,GAI_1E14A8DB4121ABAFC4B4,GUANGDONG DONGFANG PRECISI-A COMMON STOCK,Guangdong Dongfang Precision Science & Technol...,None,None,CNE1000016L3,None,None,CN,CN,None,None,None,CNY,None,EQUITY_OR_EQUITY_LINKED,OBSERVED_IN_ETF_HOLDINGS,IDENTIFIER_BASED,UNRESOLVED,ISIN,CNE1000016L3,UNRESOLVED,0.00,0,0,2019-09-30,2020-09-30,2019-11-27,2020-11-24,5,1,SE

,issuer_id,issuer_membership_security_count,issuer_membership_security_ids,issuer_etf_count,issuer_etf_tickers,issuer_first_membership_date,issuer_last_membership_date,issuer_membership_row_count,issuer_name,issuer_country,listed_security_count,listed_security_ids,listing_exchanges,listing_countries,primary_listing_sources,china_economic_issuer_evidence,security_count,security_ids,etf_count,etf_tickers,first_membership_date,last_membership_date
0,GAI_008BF1916FA3C09BB36B,1,GAS_FB0365ACB408D3970591,1,KARS,2022-05-27,2024-05-30,9,Guangzhou Great Power Energy and Technology Co...,CN,1.0,GAS_FB0365ACB408D3970591,<NA>,<NA>,UNRESOLVED,ISSUER_COUNTRY_CN,1,GAS_FB0365ACB408D3970591,1,KARS,2022-05-27,2024-05-30
1,GAI_01CAE5E9EA608E2482D2,1,GAS_B0B0614B82FE2565480D,1,KARS,2023-11-29,2026-05-29,10,Hunan Yuneng New Energy Battery Material Co Ltd,CN,1.0,GAS_B0B0614B82FE2565480D,<NA>,<NA>,UNRESOLVED,ISSUER_COUNTRY_CN,1,GAS_B0B0614B82FE2565480D,1,KARS,2023-11-29,2026-05-29
2,GAI_0787D230076902C0E857,1,GAS_9013A32826453D14D079,1,KARS,2021-08-30,2026-05-29,20,"ZHEJIANG HUAYOU COBALT CO., LTD",CN,1.0,GAS_9013A32826453D14D079,<NA>,<NA>,UNRESOLVED,ISSUER_COUNTRY_CN,1,GAS_9013A32826453D14D079,1,KARS,2021-08-30,2026-05-29
3,GAI_098ABF62DAB5DA55A65D,1,GAS_D3A4BEB9E49457BA8C37,1,KARS,2026-02-27,2026-05-29,2,"NavInfo Co., Ltd.",CN,1.0,GAS_D3A4BEB9E49457BA8C37,<NA>,<NA>,UNRESOLVED,ISSUER_COUNTRY_CN,1,GAS_D3A4BEB9E49457BA8C37,1,KARS,2026-02-27,2026-05-29
4,GAI_09DA7F36C135795A0916,1,GAS_B2C051C672FC07E5EA27,2,CARZ | KARS,2019-11-19,2026-05-29,37,"Guangzhou Automobile Group Co., Ltd",HK,1.0,GAS_B2C051C672FC07E5EA27,HKEX,HK,HKEX,CHINESE_OFFSHORE_OR_HK_LISTING,1,GAS_B2C051C672FC07E5EA27,2,CARZ | KARS,2019-11-19,2026-05-29
5,GAI_0BCA9270FF1670736E5A,1,GAS_67453941547E8CFCA7F1,1,KARS,2026-05-29,NaT,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,GAS_67453941547E8CFCA7F1,1,KARS,2026-05-29,NaT
6,GAI_10DE387103D01591CDFE,1,GAS_CE5A3C713E42281904FD,1,CARZ,2025-05-22,2025-08-25,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,GAS_CE5A3C713E42281904FD,1,CARZ,2025-05-22,2025-08-25
7,GAI_118982B3D7A3654914FD,2,GAS_0F0802D8983EE892AD09 | GAS_376976AF42716CB...,4,CARZ | DRIV | IDRV | KARS,2020-04-28,2026-06-25,84,Ganfeng Lithium Group Co Ltd,CN,2.0,GAS_0F0802D8983EE892AD09 | GAS_376976AF42716CB...,HKEX,HK,HKEX | UNRESOLVED,ISSUER_COUNTRY_CN,2,GAS_0F0802D8983EE892AD09 | GAS_376976AF42716CB...,4,CARZ | DRIV | IDRV | KARS,2020-04-28,2026-06-25
8,GAI_11D73C9A6BB9EA6924F2,1,GAS_3BDDC08D09DEB3CF277D,1,KARS,2021-08-30,2026-05-29,20,"GEM Co., Ltd.",CN,1.0,GAS_3BDDC08D09DEB3CF277D,<NA>,<NA>,UNRESOLVED,ISSUER_COUNTRY_CN,1,GAS_3BDDC08D09DEB3CF277D,1,KARS,2021-08-30,2026-05-29
9,GAI_1230A0A328AB78758D3A,1,GAS_281A00DF016838ECFF9C,2,CARZ | KARS,2026-02-25,2026-05-29,4,WeRide Inc,CN,1.0,GAS_281A00DF016838ECFF9C,<NA>,US,SEC,ISSUER_COUNTRY_CN,1,GAS_281A00DF016838ECFF9C,2,CARZ | KARS,2026-02-25,2026-05-29


In [ ]:
# 18. CNINFO CLIENT AND PERSISTENT CACHE

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/142 Safari/537.36"
    ),
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
    "Referer": f"{CNINFO_BASE_URL}/new/index",
    "Origin": CNINFO_BASE_URL,
})


def cache_key(url, params=None, data=None):
    payload = json.dumps(
        {"url": url, "params": params or {}, "data": data or {}},
        sort_keys=True,
        default=str,
        ensure_ascii=False,
    )

    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def cached_get(url, params=None, cache_dir=CNINFO_SEARCH_CACHE_DIR, suffix=".bin"):
    path = cache_dir / f"{cache_key(url, params=params)}{suffix}"

    if path.exists():
        return path.read_bytes(), {
            "status": "CACHE_HIT",
            "cache_path": str(path),
            "url": url,
        }

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                url,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            path.write_bytes(response.content)
            time.sleep(REQUEST_INTERVAL_SECONDS)

            return response.content, {
                "status": "DOWNLOADED",
                "http_status": response.status_code,
                "cache_path": str(path),
                "url": response.url,
                "attempt": attempt,
            }

        except Exception as exc:
            last_error = repr(exc)
            time.sleep(min(2 ** attempt, 20))

    raise RuntimeError(f"GET failed for {url}: {last_error}")


def cached_post_json(url, data, cache_dir=CNINFO_SEARCH_CACHE_DIR):
    path = cache_dir / f"{cache_key(url, data=data)}.json"

    if path.exists():
        with path.open("r", encoding="utf-8") as file:
            return json.load(file), {
                "status": "CACHE_HIT",
                "cache_path": str(path),
                "url": url,
            }

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.post(
                url,
                data=data,
                timeout=REQUEST_TIMEOUT_SECONDS,
                headers={
                    "X-Requested-With": "XMLHttpRequest",
                    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
                },
            )
            response.raise_for_status()
            payload = response.json()

            with path.open("w", encoding="utf-8") as file:
                json.dump(payload, file, ensure_ascii=False)

            time.sleep(REQUEST_INTERVAL_SECONDS)

            return payload, {
                "status": "DOWNLOADED",
                "http_status": response.status_code,
                "cache_path": str(path),
                "url": response.url,
                "attempt": attempt,
            }

        except Exception as exc:
            last_error = repr(exc)
            time.sleep(min(2 ** attempt, 20))

    raise RuntimeError(f"POST failed for {url}: {last_error}")

In [ ]:
# 19. DOWNLOAD CNINFO STOCK METADATA AND MAP CHINESE ISSUERS TO CNINFO ENTITIES

def parse_stock_metadata_payload(payload, exchange):
    if isinstance(payload, dict):
        rows = []

        for key in ["stockList", "stocks", "data", "records"]:
            value = payload.get(key)

            if isinstance(value, list):
                rows = value
                break

    elif isinstance(payload, list):
        rows = payload

    else:
        rows = []

    normalised = []

    for item in rows:
        if not isinstance(item, dict):
            continue

        code = (
            item.get("code")
            or item.get("stockCode")
            or item.get("secCode")
            or item.get("scode")
        )

        org_id = (
            item.get("orgId")
            or item.get("orgid")
            or item.get("orgID")
            or item.get("secid")
            or item.get("id")
        )

        short_name = (
            item.get("zwjc")
            or item.get("shortName")
            or item.get("name")
            or item.get("secName")
        )

        full_name = (
            item.get("zwqm")
            or item.get("fullName")
            or item.get("companyName")
        )

        match = re.search(
            r"(?<!\d)(\d{6})(?!\d)",
            str(code),
        )

        if not match:
            continue

        normalised.append({
            "stock_code": match.group(1),
            "cninfo_org_id": org_id,
            "cninfo_short_name": short_name,
            "cninfo_full_name": full_name,
            "cninfo_name_normalised": normalise_company_name(
                full_name if full_name else short_name
            ),
            "exchange": exchange,
            "metadata_raw_json": json.dumps(
                item,
                ensure_ascii=False,
            ),
        })

    return pd.DataFrame(normalised)


metadata_frames = []
metadata_logs = []

for exchange, endpoint in CNINFO_STOCK_METADATA_ENDPOINTS.items():
    try:
        content, log = cached_get(
            endpoint,
            cache_dir=CNINFO_METADATA_CACHE_DIR,
            suffix=".json",
        )

        payload = json.loads(
            content.decode(
                "utf-8",
                errors="replace",
            )
        )

        frame = parse_stock_metadata_payload(
            payload,
            exchange,
        )

        if not frame.empty:
            metadata_frames.append(frame)

        log.update({
            "exchange": exchange,
            "row_count": len(frame),
        })

        metadata_logs.append(log)

    except Exception as exc:
        metadata_logs.append({
            "exchange": exchange,
            "status": "FAILED",
            "row_count": 0,
            "error": repr(exc),
        })


CNINFO_STOCK_METADATA_COLUMNS = [
    "stock_code",
    "cninfo_org_id",
    "cninfo_short_name",
    "cninfo_full_name",
    "cninfo_name_normalised",
    "exchange",
    "metadata_raw_json",
]

cninfo_stock_metadata_df = (
    pd.concat(
        metadata_frames,
        ignore_index=True,
        sort=False,
    )
    if metadata_frames
    else pd.DataFrame(
        columns=CNINFO_STOCK_METADATA_COLUMNS
    )
)

for column in CNINFO_STOCK_METADATA_COLUMNS:
    if column not in cninfo_stock_metadata_df.columns:
        cninfo_stock_metadata_df[column] = pd.NA

cninfo_stock_metadata_df = (
    cninfo_stock_metadata_df[
        CNINFO_STOCK_METADATA_COLUMNS
    ]
    .dropna(
        subset=[
            "stock_code",
            "exchange",
        ]
    )
    .drop_duplicates(
        [
            "stock_code",
            "exchange",
        ]
    )
    .reset_index(drop=True)
)

cninfo_stock_metadata_log_df = pd.DataFrame(metadata_logs)


def load_optional_relationship_catalogue(path):
    if not path.exists():
        return pd.DataFrame(
            columns=CHINA_RELATIONSHIP_CATALOGUE_COLUMNS
        )

    catalogue = pd.read_csv(path)

    for column in CHINA_RELATIONSHIP_CATALOGUE_COLUMNS:
        if column not in catalogue.columns:
            catalogue[column] = pd.NA

    catalogue = catalogue[
        CHINA_RELATIONSHIP_CATALOGUE_COLUMNS
    ].copy()

    catalogue["cninfo_stock_code"] = (
        catalogue["cninfo_stock_code"]
        .astype("string")
        .str.extract(r"(\d{6})", expand=False)
    )

    catalogue["override_confidence"] = pd.to_numeric(
        catalogue["override_confidence"],
        errors="coerce",
    )

    return catalogue


china_relationship_catalogue_df = (
    load_optional_relationship_catalogue(
        CHINA_RELATIONSHIP_CATALOGUE_PATH
    )
)


override_rows = []

for override in CNINFO_ENTITY_OVERRIDES:
    matches = china_issuer_security_universe_df[
        china_issuer_security_universe_df["issuer_name"]
        .astype("string")
        .str.contains(
            override["issuer_name_regex"],
            case=False,
            regex=True,
            na=False,
        )
        & china_issuer_security_universe_df["ticker"]
        .astype("string")
        .str.contains(
            override["security_ticker_regex"],
            case=False,
            regex=True,
            na=False,
        )
    ]

    for row in matches.itertuples(index=False):
        override_rows.append({
            "issuer_id": row.issuer_id,
            "security_id": row.security_id,
            "economic_issuer_name": row.issuer_name,
            "source_security_ticker": row.ticker,
            "source_exchange": row.exchange,
            "primary_listing_source_system": row.primary_listing_source_system,
            "cninfo_stock_code": override["cninfo_stock_code"],
            "cninfo_exchange_expected": override["cninfo_exchange"],
            "cninfo_entity_relationship": override[
                "cninfo_entity_relationship"
            ],
            "cninfo_entity_role": (
                "PRIMARY_OPERATING_ENTITY"
                if str(
                    override["cninfo_entity_relationship"]
                ).startswith("SAME_ISSUER")
                else "OTHER_CONFIRMED_ENTITY"
            ),
            "relationship_valid_from": pd.NaT,
            "relationship_valid_to": pd.NaT,
            "relationship_source": "BUILT_IN_CURATED_OVERRIDE",
            "relationship_notes": pd.NA,
            "entity_mapping_method": "CURATED_OVERRIDE",
            "entity_mapping_confidence": override["override_confidence"],
        })

china_cninfo_entity_override_df = pd.DataFrame(override_rows)


external_override_rows = []

if not china_relationship_catalogue_df.empty:
    for catalogue_row in china_relationship_catalogue_df.itertuples(
        index=False
    ):
        candidates = china_issuer_security_universe_df.copy()

        if pd.notna(catalogue_row.issuer_id):
            candidates = candidates[
                candidates["issuer_id"].eq(
                    str(catalogue_row.issuer_id)
                )
            ]
        else:
            issuer_pattern = (
                str(catalogue_row.issuer_name_regex)
                if pd.notna(catalogue_row.issuer_name_regex)
                else r"$^"
            )

            ticker_pattern = (
                str(catalogue_row.security_ticker_regex)
                if pd.notna(catalogue_row.security_ticker_regex)
                else r".*"
            )

            candidates = candidates[
                candidates["issuer_name"]
                .astype("string")
                .str.contains(
                    issuer_pattern,
                    case=False,
                    regex=True,
                    na=False,
                )
                & candidates["ticker"]
                .astype("string")
                .str.contains(
                    ticker_pattern,
                    case=False,
                    regex=True,
                    na=False,
                )
            ]

        for candidate in candidates.itertuples(index=False):
            external_override_rows.append({
                "issuer_id": candidate.issuer_id,
                "security_id": candidate.security_id,
                "economic_issuer_name": candidate.issuer_name,
                "source_security_ticker": candidate.ticker,
                "source_exchange": candidate.exchange,
                "primary_listing_source_system": (
                    candidate.primary_listing_source_system
                ),
                "cninfo_stock_code": (
                    catalogue_row.cninfo_stock_code
                ),
                "cninfo_exchange_expected": (
                    catalogue_row.cninfo_exchange
                ),
                "cninfo_entity_relationship": (
                    catalogue_row.cninfo_entity_relationship
                ),
                "cninfo_entity_role": (
                    catalogue_row.cninfo_entity_role
                ),
                "relationship_valid_from": pd.to_datetime(
                    catalogue_row.relationship_valid_from,
                    errors="coerce",
                ),
                "relationship_valid_to": pd.to_datetime(
                    catalogue_row.relationship_valid_to,
                    errors="coerce",
                ),
                "relationship_source": (
                    catalogue_row.relationship_source
                    if pd.notna(catalogue_row.relationship_source)
                    else "USER_RELATIONSHIP_CATALOGUE"
                ),
                "relationship_notes": (
                    catalogue_row.relationship_notes
                ),
                "entity_mapping_method": (
                    "USER_RELATIONSHIP_CATALOGUE"
                ),
                "entity_mapping_confidence": pd.to_numeric(
                    catalogue_row.override_confidence,
                    errors="coerce",
                ),
            })

china_cninfo_external_override_df = pd.DataFrame(
    external_override_rows
)


direct_mainland_rows = (
    china_issuer_security_universe_df[
        china_issuer_security_universe_df["exchange"].isin(
            MAINLAND_EXCHANGES
        )
        & china_issuer_security_universe_df[
            "candidate_stock_code"
        ].notna()
    ]
    .copy()
)

direct_mainland_entity_df = pd.DataFrame({
    "issuer_id": direct_mainland_rows["issuer_id"],
    "security_id": direct_mainland_rows["security_id"],
    "economic_issuer_name": direct_mainland_rows["issuer_name"],
    "source_security_ticker": direct_mainland_rows["ticker"],
    "source_exchange": direct_mainland_rows["exchange"],
    "primary_listing_source_system": (
        direct_mainland_rows["primary_listing_source_system"]
    ),
    "cninfo_stock_code": direct_mainland_rows["candidate_stock_code"],
    "cninfo_exchange_expected": direct_mainland_rows["exchange"],
    "cninfo_entity_relationship": "SAME_ISSUER_MAINLAND_LISTING",
    "cninfo_entity_role": "PRIMARY_OPERATING_ENTITY",
    "relationship_valid_from": pd.NaT,
    "relationship_valid_to": pd.NaT,
    "relationship_source": "BLOCK2_CANONICAL_LISTING_IDENTITY",
    "relationship_notes": pd.NA,
    "entity_mapping_method": "BLOCK2_MAINLAND_LISTING",
    "entity_mapping_confidence": 1.00,
})

china_cninfo_entity_seed_df = (
    pd.concat(
        [
            direct_mainland_entity_df,
            china_cninfo_entity_override_df,
            china_cninfo_external_override_df,
        ],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(
        [
            "issuer_id",
            "cninfo_stock_code",
            "cninfo_entity_relationship",
        ]
    )
    .reset_index(drop=True)
)

china_cninfo_entity_bridge_df = (
    china_cninfo_entity_seed_df.merge(
        cninfo_stock_metadata_df[
            [
                "stock_code",
                "cninfo_org_id",
                "cninfo_short_name",
                "cninfo_full_name",
                "cninfo_name_normalised",
                "exchange",
            ]
        ].rename(
            columns={
                "stock_code": "cninfo_stock_code",
                "exchange": "cninfo_exchange",
            }
        ),
        on="cninfo_stock_code",
        how="left",
        validate="m:1",
    )
)

china_cninfo_entity_bridge_df["exchange_match"] = (
    china_cninfo_entity_bridge_df["cninfo_exchange"]
    .eq(
        china_cninfo_entity_bridge_df[
            "cninfo_exchange_expected"
        ]
    )
    .fillna(False)
)

china_cninfo_entity_bridge_df["cninfo_entity_mapping_status"] = np.select(
    [
        china_cninfo_entity_bridge_df["cninfo_org_id"].notna()
        & china_cninfo_entity_bridge_df["exchange_match"],
        china_cninfo_entity_bridge_df["cninfo_org_id"].notna(),
    ],
    [
        "CONFIRMED_CODE_AND_EXCHANGE",
        "CONFIRMED_CODE_EXCHANGE_REVIEW",
    ],
    default="UNRESOLVED_CNINFO_ENTITY",
)


for column in [
    "relationship_valid_from",
    "relationship_valid_to",
]:
    china_cninfo_entity_bridge_df[column] = pd.to_datetime(
        china_cninfo_entity_bridge_df.get(
            column,
            pd.Series(pd.NaT, index=china_cninfo_entity_bridge_df.index),
        ),
        errors="coerce",
    )

china_cninfo_entity_bridge_df["cninfo_entity_role"] = (
    china_cninfo_entity_bridge_df["cninfo_entity_role"]
    .fillna("OTHER_CONFIRMED_ENTITY")
)

invalid_relationship_mask = (
    ~china_cninfo_entity_bridge_df[
        "cninfo_entity_relationship"
    ].isin(VALID_CNINFO_ENTITY_RELATIONSHIPS)
)

invalid_role_mask = (
    ~china_cninfo_entity_bridge_df[
        "cninfo_entity_role"
    ].isin(VALID_CNINFO_ENTITY_ROLES)
)

china_cninfo_entity_bridge_df["relationship_validation_status"] = np.select(
    [
        invalid_relationship_mask,
        invalid_role_mask,
    ],
    [
        "INVALID_RELATIONSHIP_TYPE",
        "INVALID_ENTITY_ROLE",
    ],
    default="VALID",
)


china_cninfo_entity_bridge_df["relationship_id"] = (
    china_cninfo_entity_bridge_df.apply(
        lambda row: (
            "CIR_"
            + hashlib.sha256(
                "|".join(
                    [
                        str(row.get("issuer_id", "")),
                        str(row.get("cninfo_stock_code", "")),
                        str(row.get("cninfo_entity_relationship", "")),
                        str(row.get("relationship_valid_from", "")),
                    ]
                ).encode("utf-8")
            ).hexdigest().upper()[:20]
        ),
        axis=1,
    )
)

china_cninfo_entity_bridge_df["cninfo_entity_id"] = (
    china_cninfo_entity_bridge_df.apply(
        lambda row: (
            "CNE_"
            + hashlib.sha256(
                "|".join(
                    [
                        str(row.get("cninfo_stock_code", "")),
                        str(row.get("cninfo_exchange", "")),
                        str(row.get("cninfo_org_id", "")),
                    ]
                ).encode("utf-8")
            ).hexdigest().upper()[:20]
        ),
        axis=1,
    )
)

china_entity_relationship_graph_df = (
    china_cninfo_entity_bridge_df[
        [
            "relationship_id",
            "issuer_id",
            "security_id",
            "economic_issuer_name",
            "cninfo_entity_id",
            "cninfo_stock_code",
            "cninfo_org_id",
            "cninfo_short_name",
            "cninfo_full_name",
            "cninfo_exchange",
            "cninfo_entity_relationship",
            "cninfo_entity_role",
            "relationship_valid_from",
            "relationship_valid_to",
            "relationship_source",
            "relationship_notes",
            "entity_mapping_method",
            "entity_mapping_confidence",
            "cninfo_entity_mapping_status",
            "relationship_validation_status",
        ]
    ]
    .drop_duplicates("relationship_id")
    .reset_index(drop=True)
)

china_cninfo_entities_df = (
    china_entity_relationship_graph_df[
        [
            "cninfo_entity_id",
            "cninfo_stock_code",
            "cninfo_org_id",
            "cninfo_short_name",
            "cninfo_full_name",
            "cninfo_exchange",
            "cninfo_entity_role",
        ]
    ]
    .drop_duplicates("cninfo_entity_id")
    .reset_index(drop=True)
)

china_cninfo_entity_bridge_confirmed_df = (
    china_cninfo_entity_bridge_df[
        china_cninfo_entity_bridge_df[
            "cninfo_entity_mapping_status"
        ].str.startswith(
            "CONFIRMED",
            na=False,
        )
        & china_cninfo_entity_bridge_df[
            "relationship_validation_status"
        ].eq("VALID")
    ]
    .copy()
    .reset_index(drop=True)
)

china_cninfo_entity_bridge_unresolved_df = (
    china_cninfo_entity_bridge_df[
        ~china_cninfo_entity_bridge_df[
            "cninfo_entity_mapping_status"
        ].str.startswith(
            "CONFIRMED",
            na=False,
        )
    ]
    .copy()
    .reset_index(drop=True)
)

china_cninfo_entity_resolution_report_df = (
    china_cninfo_entity_bridge_df
    .groupby(
        [
            "entity_mapping_method",
            "cninfo_entity_relationship",
            "cninfo_entity_mapping_status",
        ],
        dropna=False,
    )
    .agg(
        issuer_count=("issuer_id", "nunique"),
        cninfo_entity_count=("cninfo_stock_code", "nunique"),
        average_mapping_confidence=(
            "entity_mapping_confidence",
            "mean",
        ),
    )
    .reset_index()
)

# Compatibility aliases allow the proven disclosure, extraction and canonical
# mapping engine in Steps 6-13 to operate unchanged.
china_security_universe_df = (
    china_cninfo_entity_bridge_confirmed_df.rename(
        columns={
            "economic_issuer_name": "issuer_name",
            "source_security_ticker": "ticker",
            "cninfo_stock_code": "stock_code",
            "cninfo_exchange": "exchange",
        }
    )
    .copy()
)

china_security_universe_df["listing_ticker"] = (
    china_security_universe_df["stock_code"]
)

china_security_universe_df["share_class"] = np.where(
    china_security_universe_df["stock_code"]
    .astype("string")
    .str.startswith(
        ("200", "900"),
        na=False,
    ),
    "B_SHARE",
    "A_SHARE",
)

china_security_universe_df["listing_country"] = "CN"
china_security_universe_df["mic"] = (
    china_security_universe_df["exchange"].map({
        "SSE": "XSHG",
        "SZSE": "XSHE",
        "BSE": "XBEI",
    })
)

china_security_universe_df["etf_tickers"] = (
    china_security_universe_df["issuer_id"].map(
        china_economic_issuer_universe_df.set_index(
            "issuer_id"
        )["etf_tickers"]
    )
)

china_resolved_securities_df = (
    china_security_universe_df.copy()
)

china_unresolved_securities_df = (
    china_cninfo_entity_bridge_unresolved_df.copy()
)

china_security_bridge_df = (
    china_cninfo_entity_bridge_df.copy()
)

china_issuer_universe_df = (
    china_security_universe_df[
        [
            "issuer_id",
            "issuer_name",
            "security_id",
            "ticker",
            "listing_ticker",
            "stock_code",
            "exchange",
            "mic",
            "listing_country",
            "share_class",
            "etf_tickers",
            "cninfo_entity_relationship",
            "primary_listing_source_system",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

china_etf_membership_intervals_df = (
    china_issuer_etf_membership_intervals_df[
        china_issuer_etf_membership_intervals_df["issuer_id"].isin(
            set(
                china_cninfo_entity_bridge_confirmed_df[
                    "issuer_id"
                ].dropna()
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)

china_cninfo_resolution_report_df = (
    china_cninfo_entity_resolution_report_df.copy()
)

china_universe_exclusions_df = (
    china_cninfo_entity_bridge_unresolved_df.copy()
)

china_universe_quality_df = pd.DataFrame({
    "metric": [
        "china_economic_issuers",
        "china_listed_securities_all_venues",
        "cninfo_entity_relationships_seeded",
        "cninfo_entity_relationships_confirmed",
        "cninfo_enriched_issuers",
        "cninfo_entities",
        "same_issuer_relationships",
        "parent_subsidiary_or_operating_relationships",
        "unresolved_cninfo_relationships",
    ],
    "value": [
        china_economic_issuer_universe_df["issuer_id"].nunique(),
        china_issuer_security_universe_df["security_id"].nunique(),
        len(china_cninfo_entity_seed_df),
        len(china_cninfo_entity_bridge_confirmed_df),
        china_cninfo_entity_bridge_confirmed_df[
            "issuer_id"
        ].nunique(),
        china_cninfo_entity_bridge_confirmed_df[
            "cninfo_stock_code"
        ].nunique(),
        china_cninfo_entity_bridge_confirmed_df[
            "cninfo_entity_relationship"
        ].str.startswith(
            "SAME_ISSUER",
            na=False,
        ).sum(),
        china_cninfo_entity_bridge_confirmed_df[
            "cninfo_entity_relationship"
        ].isin(
            {
                "MAINLAND_PARENT",
                "MAINLAND_SUBSIDIARY",
                "OPERATING_COMPANY",
                "ASSOCIATE_OR_JV",
            }
        ).sum(),
        len(china_cninfo_entity_bridge_unresolved_df),
    ],
})

china_block2_listing_metadata_report_df = (
    china_issuer_security_universe_df
    .groupby(
        [
            "exchange",
            "listing_country",
            "primary_listing_source_system",
        ],
        dropna=False,
    )
    .agg(
        securities=("security_id", "nunique"),
        issuers=("issuer_id", "nunique"),
    )
    .reset_index()
)

print("Official CNINFO stock records:", len(cninfo_stock_metadata_df))
print(
    "Chinese economic issuers:",
    china_economic_issuer_universe_df["issuer_id"].nunique(),
)
print(
    "Confirmed CNINFO enrichment relationships:",
    len(china_cninfo_entity_bridge_confirmed_df),
)
print(
    "Issuers with CNINFO enrichment:",
    china_cninfo_entity_bridge_confirmed_df["issuer_id"].nunique(),
)

display(china_cninfo_entity_resolution_report_df)
display(china_universe_quality_df)
display(china_cninfo_entity_bridge_confirmed_df)
display(china_entity_relationship_graph_df)
display(china_cninfo_entities_df)

if china_cninfo_entity_bridge_confirmed_df.empty:
    display(china_cninfo_entity_bridge_unresolved_df)

    raise RuntimeError(
        "No Chinese ETF issuer was mapped to a confirmed CNINFO entity. "
        "Review CNINFO_ENTITY_OVERRIDES and the Block 2 issuer names/tickers."
    )

Official CNINFO stock records: 6221
Chinese economic issuers: 103
Confirmed CNINFO enrichment relationships: 5
Issuers with CNINFO enrichment: 4


/tmp/ipykernel_1172/3065540756.py:221: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_1172/3065540756.py:221: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_1172/3065540756.py:221: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_1172/3065540756.py:221: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_1172/3065540756.py:221: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_1172/3065540756.py:221: UserWarning: Thi

,entity_mapping_method,cninfo_entity_relationship,cninfo_entity_mapping_status,issuer_count,cninfo_entity_count,average_mapping_confidence
0,BLOCK2_MAINLAND_LISTING,SAME_ISSUER_MAINLAND_LISTING,CONFIRMED_CODE_AND_EXCHANGE,1,1,1.0
1,CURATED_OVERRIDE,SAME_ISSUER_DUAL_LISTING,CONFIRMED_CODE_AND_EXCHANGE,1,1,1.0
2,CURATED_OVERRIDE,SAME_ISSUER_DUAL_LISTING,CONFIRMED_CODE_EXCHANGE_REVIEW,2,2,1.0
3,CURATED_OVERRIDE,SAME_ISSUER_MULTIPLE_SHARE_CLASS,CONFIRMED_CODE_AND_EXCHANGE,1,1,1.0


,metric,value
0,china_economic_issuers,103
1,china_listed_securities_all_venues,79
2,cninfo_entity_relationships_seeded,5
3,cninfo_entity_relationships_confirmed,5
4,cninfo_enriched_issuers,4
5,cninfo_entities,5
6,same_issuer_relationships,5
7,parent_subsidiary_or_operating_relationships,0
8,unresolved_cninfo_relationships,0


,issuer_id,security_id,economic_issuer_name,source_security_ticker,source_exchange,primary_listing_source_system,cninfo_stock_code,cninfo_exchange_expected,cninfo_entity_relationship,cninfo_entity_role,relationship_valid_from,relationship_valid_to,relationship_source,relationship_notes,entity_mapping_method,entity_mapping_confidence,cninfo_org_id,cninfo_short_name,cninfo_full_name,cninfo_name_normalised,cninfo_exchange,exchange_match,cninfo_entity_mapping_status,relationship_validation_status,relationship_id,cninfo_entity_id
0,GAI_1B953A20ED4A32ACF905,GAS_A6B420986A3617E50015,Chongqing Changan Automobile Co Ltd,200625,SZSE,CNINFO,200625,SZSE,SAME_ISSUER_MAINLAND_LISTING,PRIMARY_OPERATING_ENTITY,NaT,NaT,BLOCK2_CANONICAL_LISTING_IDENTITY,<NA>,BLOCK2_MAINLAND_LISTING,1.0,gssz0000625,长安B,None,长安B,SZSE,True,CONFIRMED_CODE_AND_EXCHANGE,VALID,CIR_8E1525C822166172EA14,CNE_D6558279F88A1F44503F
1,GAI_F9BF134E7CCD96A4C594,GAS_7BEEA4BAD5D4252C9825,BYD Co Ltd,1211,HKEX,HKEX,002594,SZSE,SAME_ISSUER_DUAL_LISTING,PRIMARY_OPERATING_ENTITY,NaT,NaT,BUILT_IN_CURATED_OVERRIDE,<NA>,CURATED_OVERRIDE,1.0,gshk0001211,比亚迪,None,比亚迪,SZSE,True,CONFIRMED_CODE_AND_EXCHANGE,VALID,CIR_F2176D5D0D230E8DB533,CNE_C3D379791347A48D08D4
2,GAI_4FE2831532A093228A7A,GAS_84B18AF96EAB297DAA9E,Great Wall Motor Co Ltd,2333,HKEX,HKEX,601633,SSE,SAME_ISSUER_DUAL_LISTING,PRIMARY_OPERATING_ENTITY,NaT,NaT,BUILT_IN_CURATED_OVERRIDE,<NA>,CURATED_OVERRIDE,1.0,gshk0002333,长城汽车,None,长城汽车,SZSE,False,CONFIRMED_CODE_EXCHANGE_REVIEW,VALID,CIR_073109ED26ED8BBBD87D,CNE_628C9045CFA239CCD05A
3,GAI_09DA7F36C135795A0916,GAS_B2C051C672FC07E5EA27,"Guangzhou Automobile Group Co., Ltd",2238,HKEX,HKEX,601238,SSE,SAME_ISSUER_DUAL_LISTING,PRIMARY_OPERATING_ENTITY,NaT,NaT,BUILT_IN_CURATED_OVERRIDE,<NA>,CURATED_OVERRIDE,1.0,9900006006,广汽集团,None,广汽集团,SZSE,False,CONFIRMED_CODE_EXCHANGE_REVIEW,VALID,CIR_A36F7600C5E6BD43DC6D,CNE_8972E24E75AA38DF3780
4,GAI_1B953A20ED4A32ACF905,GAS_A6B420986A3617E50015,Chongqing Changan Automobile Co Ltd,200625,SZSE,CNINFO,000625,SZSE,SAME_ISSUER_MULTIPLE_SHARE_CLASS,PRIMARY_OPERATING_ENTITY,NaT,NaT,BUILT_IN_CURATED_OVERRIDE,<NA>,CURATED_OVERRIDE,1.0,gssz0000625,长安汽车,None,长安汽车,SZSE,True,CONFIRMED_CODE_AND_EXCHANGE,VALID,CIR_430FD1B1D8F05F7C869A,CNE_338C4F34BF44589162A6


,relationship_id,issuer_id,security_id,economic_issuer_name,cninfo_entity_id,cninfo_stock_code,cninfo_org_id,cninfo_short_name,cninfo_full_name,cninfo_exchange,cninfo_entity_relationship,cninfo_entity_role,relationship_valid_from,relationship_valid_to,relationship_source,relationship_notes,entity_mapping_method,entity_mapping_confidence,cninfo_entity_mapping_status,relationship_validation_status
0,CIR_8E1525C822166172EA14,GAI_1B953A20ED4A32ACF905,GAS_A6B420986A3617E50015,Chongqing Changan Automobile Co Ltd,CNE_D6558279F88A1F44503F,200625,gssz0000625,长安B,None,SZSE,SAME_ISSUER_MAINLAND_LISTING,PRIMARY_OPERATING_ENTITY,NaT,NaT,BLOCK2_CANONICAL_LISTING_IDENTITY,<NA>,BLOCK2_MAINLAND_LISTING,1.0,CONFIRMED_CODE_AND_EXCHANGE,VALID
1,CIR_F2176D5D0D230E8DB533,GAI_F9BF134E7CCD96A4C594,GAS_7BEEA4BAD5D4252C9825,BYD Co Ltd,CNE_C3D379791347A48D08D4,002594,gshk0001211,比亚迪,None,SZSE,SAME_ISSUER_DUAL_LISTING,PRIMARY_OPERATING_ENTITY,NaT,NaT,BUILT_IN_CURATED_OVERRIDE,<NA>,CURATED_OVERRIDE,1.0,CONFIRMED_CODE_AND_EXCHANGE,VALID
2,CIR_073109ED26ED8BBBD87D,GAI_4FE2831532A093228A7A,GAS_84B18AF96EAB297DAA9E,Great Wall Motor Co Ltd,CNE_628C9045CFA239CCD05A,601633,gshk0002333,长城汽车,None,SZSE,SAME_ISSUER_DUAL_LISTING,PRIMARY_OPERATING_ENTITY,NaT,NaT,BUILT_IN_CURATED_OVERRIDE,<NA>,CURATED_OVERRIDE,1.0,CONFIRMED_CODE_EXCHANGE_REVIEW,VALID
3,CIR_A36F7600C5E6BD43DC6D,GAI_09DA7F36C135795A0916,GAS_B2C051C672FC07E5EA27,"Guangzhou Automobile Group Co., Ltd",CNE_8972E24E75AA38DF3780,601238,9900006006,广汽集团,None,SZSE,SAME_ISSUER_DUAL_LISTING,PRIMARY_OPERATING_ENTITY,NaT,NaT,BUILT_IN_CURATED_OVERRIDE,<NA>,CURATED_OVERRIDE,1.0,CONFIRMED_CODE_EXCHANGE_REVIEW,VALID
4,CIR_430FD1B1D8F05F7C869A,GAI_1B953A20ED4A32ACF905,GAS_A6B420986A3617E50015,Chongqing Changan Automobile Co Ltd,CNE_338C4F34BF44589162A6,000625,gssz0000625,长安汽车,None,SZSE,SAME_ISSUER_MULTIPLE_SHARE_CLASS,PRIMARY_OPERATING_ENTITY,NaT,NaT,BUILT_IN_CURATED_OVERRIDE,<NA>,CURATED_OVERRIDE,1.0,CONFIRMED_CODE_AND_EXCHANGE,VALID


,cninfo_entity_id,cninfo_stock_code,cninfo_org_id,cninfo_short_name,cninfo_full_name,cninfo_exchange,cninfo_entity_role
0,CNE_D6558279F88A1F44503F,200625,gssz0000625,长安B,None,SZSE,PRIMARY_OPERATING_ENTITY
1,CNE_C3D379791347A48D08D4,002594,gshk0001211,比亚迪,None,SZSE,PRIMARY_OPERATING_ENTITY
2,CNE_628C9045CFA239CCD05A,601633,gshk0002333,长城汽车,None,SZSE,PRIMARY_OPERATING_ENTITY
3,CNE_8972E24E75AA38DF3780,601238,9900006006,广汽集团,None,SZSE,PRIMARY_OPERATING_ENTITY
4,CNE_338C4F34BF44589162A6,000625,gssz0000625,长安汽车,None,SZSE,PRIMARY_OPERATING_ENTITY


In [ ]:
# 20. QUERY OFFICIAL CNINFO PERIODIC REPORT DISCLOSURES

def cninfo_column(exchange):
    return {
        "SSE": "sse",
        "SZSE": "szse",
        "BSE": "bj",
    }.get(exchange, "")


def announcement_search_payload(
    stock_code,
    org_id,
    exchange,
    category,
    page_num,
):
    return {
        "pageNum": str(page_num),
        "pageSize": str(SEARCH_PAGE_SIZE),
        "column": cninfo_column(exchange),
        "tabName": "fulltext",
        "plate": "",
        "stock": f"{stock_code},{org_id}",
        "searchkey": "",
        "secid": "",
        "category": category,
        "trade": "",
        "seDate": f"{DISCOVERY_START_DATE}~{DISCOVERY_END_DATE}",
        "sortName": "time",
        "sortType": "desc",
        "isHLtitle": "true",
    }


def normalise_announcement_record(record, security_row, category):
    adjunct_url = record.get("adjunctUrl") or record.get("adjunct_url")
    announcement_time = record.get("announcementTime")

    if announcement_time is not None:
        numeric_time = pd.to_numeric(announcement_time, errors="coerce")

        if pd.notna(numeric_time):
            release_datetime = pd.to_datetime(
                numeric_time,
                unit="ms",
                utc=True,
                errors="coerce",
            )
        else:
            release_datetime = pd.to_datetime(
                announcement_time,
                errors="coerce",
                utc=True,
            )
    else:
        release_datetime = pd.NaT

    title = BeautifulSoup(
        str(record.get("announcementTitle", "")),
        "lxml",
    ).get_text(" ", strip=True)

    return {
        "announcement_id": record.get("announcementId"),
        "security_id": security_row.security_id,
        "issuer_id": security_row.issuer_id,
        "stock_code": security_row.stock_code,
        "cninfo_org_id": security_row.cninfo_org_id,
        "exchange": security_row.exchange,
        "cninfo_short_name": (
            record.get("secName")
            or security_row.cninfo_short_name
        ),
        "announcement_title": title,
        "announcement_time_raw": announcement_time,
        "release_datetime": release_datetime,
        "release_date": (
            release_datetime.normalize()
            if pd.notna(release_datetime)
            else pd.NaT
        ),
        "category_code": category,
        "report_type": TARGET_CATEGORIES.get(category),
        "adjunct_url": adjunct_url,
        "document_url": (
            urljoin(
                CNINFO_STATIC_BASE_URL,
                str(adjunct_url).lstrip("/"),
            )
            if adjunct_url
            else pd.NA
        ),
        "adjunct_size": record.get("adjunctSize"),
        "adjunct_type": record.get("adjunctType"),
        "storage_time": record.get("storageTime"),
        "is_summary": bool(SUMMARY_TITLE_PATTERN.search(title)),
        "source_record_json": json.dumps(record, ensure_ascii=False),
    }


announcement_rows = []
announcement_logs = []

resolved_query_rows = (
    china_resolved_securities_df[
        ["security_id", "issuer_id", "stock_code", "cninfo_org_id", "cninfo_short_name", "exchange"]
    ]
    .drop_duplicates()
)

for security_row in tqdm(
    resolved_query_rows.itertuples(index=False),
    total=len(resolved_query_rows),
    desc="Querying CNINFO reports",
):
    security_result_count = 0

    for category in TARGET_CATEGORIES:
        page_num = 1

        while True:
            payload_data = announcement_search_payload(
                security_row.stock_code,
                security_row.cninfo_org_id,
                security_row.exchange,
                category,
                page_num,
            )

            try:
                payload, log = cached_post_json(
                    CNINFO_ANNOUNCEMENT_ENDPOINT,
                    payload_data,
                )

                announcements = payload.get("announcements", []) or []

                for record in announcements:
                    normalised = normalise_announcement_record(
                        record,
                        security_row,
                        category,
                    )

                    if TARGET_TITLE_PATTERN.search(
                        normalised["announcement_title"]
                    ):
                        announcement_rows.append(normalised)
                        security_result_count += 1

                log.update({
                    "stock_code": security_row.stock_code,
                    "category_code": category,
                    "page_num": page_num,
                    "api_rows": len(announcements),
                })
                announcement_logs.append(log)

                total_pages = int(payload.get("totalpages", 1) or 1)

                if page_num >= total_pages:
                    break

                if (
                    MAX_ANNOUNCEMENTS_PER_SECURITY is not None
                    and security_result_count >= int(MAX_ANNOUNCEMENTS_PER_SECURITY)
                ):
                    break

                page_num += 1

            except Exception as exc:
                announcement_logs.append({
                    "stock_code": security_row.stock_code,
                    "category_code": category,
                    "page_num": page_num,
                    "status": "FAILED",
                    "api_rows": 0,
                    "error": repr(exc),
                })
                break



def normalise_cninfo_document_url(
    document_url,
    adjunct_url=None,
):
    """
    Return the canonical static.cninfo.com.cn document URL.

    CNINFO announcement search responses provide adjunctUrl values such as
    finalpage/YYYY-MM-DD/<announcement>.PDF. These files are served from the
    static document host rather than the main website host.
    """
    candidate = (
        adjunct_url
        if pd.notna(adjunct_url)
        else document_url
    )

    if pd.isna(candidate):
        return pd.NA

    text = str(candidate).strip()

    if not text:
        return pd.NA

    parsed = urlparse(text)

    if parsed.scheme and parsed.netloc:
        path = parsed.path.lstrip("/")
    else:
        path = text.lstrip("/")

    return urljoin(
        CNINFO_STATIC_BASE_URL,
        path,
    )


cninfo_announcements_discovered_df = (
    pd.DataFrame(announcement_rows)
    .drop_duplicates(
        ["announcement_id", "stock_code", "document_url"]
    )
    .reset_index(drop=True)
    if announcement_rows
    else pd.DataFrame(
        columns=[
            "announcement_id", "security_id", "issuer_id", "stock_code",
            "cninfo_org_id", "exchange", "cninfo_short_name",
            "announcement_title", "announcement_time_raw",
            "release_datetime", "release_date", "category_code",
            "report_type", "adjunct_url", "document_url",
            "adjunct_size", "adjunct_type", "storage_time",
            "is_summary", "source_record_json",
        ]
    )
)

if not cninfo_announcements_discovered_df.empty:
    cninfo_announcements_discovered_df[
        "document_url"
    ] = cninfo_announcements_discovered_df.apply(
        lambda row: normalise_cninfo_document_url(
            row.get("document_url"),
            row.get("adjunct_url"),
        ),
        axis=1,
    )

cninfo_announcement_search_log_df = pd.DataFrame(announcement_logs)

print("Target CNINFO disclosures discovered:", len(cninfo_announcements_discovered_df))

if not cninfo_announcements_discovered_df.empty:
    print(
        "CNINFO static-host URL share:",
        cninfo_announcements_discovered_df[
            "document_url"
        ]
        .astype("string")
        .str.startswith(
            CNINFO_STATIC_BASE_URL,
            na=False,
        )
        .mean(),
    )

display(cninfo_announcements_discovered_df.head(40))

Querying CNINFO reports:   0%|          | 0/5 [00:00<?, ?it/s]

Target CNINFO disclosures discovered: 210
CNINFO static-host URL share: 1.0


,announcement_id,security_id,issuer_id,stock_code,cninfo_org_id,exchange,cninfo_short_name,announcement_title,announcement_time_raw,release_datetime,release_date,category_code,report_type,adjunct_url,document_url,adjunct_size,adjunct_type,storage_time,is_summary,source_record_json
0,1225093379,GAS_A6B420986A3617E50015,GAI_1B953A20ED4A32ACF905,200625,gssz0000625,SZSE,长 安Ｂ,2025年年度报告（英文版）,1775836800000,2026-04-10 16:00:00+00:00,2026-04-10 00:00:00+00:00,category_ndbg_szsh,ANNUAL_REPORT,finalpage/2026-04-11/1225093379.PDF,https://static.cninfo.com.cn/finalpage/2026-04...,5032,PDF,None,False,"{""id"": null, ""secCode"": ""200625"", ""secName"": ""..."
1,1225093378,GAS_A6B420986A3617E50015,GAI_1B953A20ED4A32ACF905,200625,gssz0000625,SZSE,长 安Ｂ,2025年年度报告摘要（英文版）,1775836800000,2026-04-10 16:00:00+00:00,2026-04-10 00:00:00+00:00,category_ndbg_szsh,ANNUAL_REPORT,finalpage/2026-04-11/1225093378.PDF,https://static.cninfo.com.cn/finalpage/2026-04...,155,PDF,None,True,"{""id"": null, ""secCode"": ""200625"", ""secName"": ""..."
2,1223057068,GAS_A6B420986A3617E50015,GAI_1B953A20ED4A32ACF905,200625,gssz0000625,SZSE,长 安Ｂ,2024年年度报告（英文版）,1744300800000,2025-04-10 16:00:00+00:00,2025-04-10 00:00:00+00:00,category_ndbg_szsh,ANNUAL_REPORT,finalpage/2025-04-11/1223057068.PDF,https://static.cninfo.com.cn/finalpage/2025-04...,6486,PDF,None,False,"{""id"": null, ""secCode"": ""200625"", ""secName"": ""..."
3,1219647350,GAS_A6B420986A3617E50015,GAI_1B953A20ED4A32ACF905,200625,gssz0000625,SZSE,长 安Ｂ,2023年年度报告（英文版）,1713369600000,2024-04-17 16:00:00+00:00,2024-04-17 00:00:00+00:00,category_ndbg_szsh,ANNUAL_REPORT,finalpage/2024-04-18/1219647350.PDF,https://static.cninfo.com.cn/finalpage/2024-04...,2389,PDF,None,False,"{""id"": null, ""secCode"": ""200625"", ""secName"": ""..."
4,1216442209,GAS_A6B420986A3617E50015,GAI_1B953A20ED4A32ACF905,200625,gssz0000625,SZSE,长 安Ｂ,2022年年度报告（英文版）,1681747200000,2023-04-17 16:00:00+00:00,2023-04-17 00:00:00+00:00,category_ndbg_szsh,ANNUAL_REPORT,finalpage/2023-04-18/1216442209.PDF,https://static.cninfo.com.cn/finalpage/2023-04...,4160,PDF,None,False,"{""id"": null, ""secCode"": ""200625"", ""secName"": ""..."
5,1213168443,GAS_A6B420986A3617E50015,GAI_1B953A20ED4A32ACF905,200625,gssz0000625,SZSE,长 安Ｂ,2021年年度报告（英文版）,1651075200000,2022-04-27 16:00:00+00:00,2022-04-27 00:00:00+00:00,category_ndbg_szsh,ANNUAL_REPORT,finalpage/2022-04-28/1213168443.PDF,https://static.cninfo.com.cn/finalpage/2022-04...,8737,PDF,None,False,"{""id"": null, ""secCode"": ""200625"", ""secName"": ""..."
6,1209728469,GAS_A6B420986A3617E50015,GAI_1B953A20ED4A32ACF905,200625,gssz0000625,SZSE,长 安Ｂ,2020年年度报告（英文版）,1618848000000,2021-04-19 16:00:00+00:00,2021-04-19 00:00:00+00:00,category_ndbg_szsh,ANNUAL_REPORT,finalpage/2021-04-20/1209728469.PDF,https://static.cninfo.com.cn/finalpage/2021-04...,5176,PDF,None,False,"{""id"": null, ""secCode"": ""200625"", ""secName"": ""..."
7,1207683429,GAS_A6B420986A3617E50015,GAI_1B953A20ED4A32ACF905,200625,gssz0000625,SZSE,长 安Ｂ,2019年年度报告（英文版）,1588176000000,2020-04-29 16:00:00+00:00,2020-04-29 00:00:00+00:00,category_ndbg_szsh,ANNUAL_REPORT,finalpage/2020-04-30/1207683429.PDF,https://static.cninfo.com.cn/finalpage/2020-04...,6112,PDF,None,False,"{""id"": null, ""secCode"": ""200625"", ""secName"": ""..."
8,1206068873,GAS_A6B420986A3617E50015,GAI_1B953A20ED4A32ACF905,200625,gssz0000625,SZSE,长 安Ｂ,2018年年度报告（英文版）,1555948800000,2019-04-22 16:00:00+00:00,2019-04-22 00:00:00+00:00,category_ndbg_szsh,ANNUAL_REPORT,finalpage/2019-04-23/1206068873.PDF,https://static.cninfo.com.cn/finalpage/2019-04...,5553,PDF,None,False,"{""id"": null, ""secCode"": ""200625"", ""secName"": ""..."
9,1224556973,GAS_A6B420986A3617E50015,GAI_1B953A20ED4A32ACF905,200625,gssz0000625,SZSE,长 安Ｂ,2025年半年度报告摘要（英文版）,1755878400000,2025-08-22 16:00:00+00:00,2025-08-22 00:00:00+00:00,category_bndbg_szsh,SEMIANNUAL_REPORT,finalpage/2025-08-23/1224556973.PDF,https://static.cninfo.com.cn/finalpage/2025-08...,146,PDF,None,True,"{""id"": null, ""secCode"": ""200625"", ""

In [ ]:
# 21. LINK DISCLOSURES TO GLOBAL IDS AND SELECT PRIMARY REPORTS

FILING_METADATA_COLUMNS = [
    "document_id",
    "announcement_id",
    "stock_code",
    "cninfo_org_id",
    "exchange",
    "security_id",
    "issuer_id",
    "issuer_name",
    "ticker",
    "cninfo_short_name",
    "announcement_title",
    "report_type",
    "document_url",
    "adjunct_type",
    "release_datetime",
    "available_datetime",
    "available_date",
    "availability_basis",
    "source_system",
    "document_priority",
    "is_summary",
]

if cninfo_announcements_discovered_df.empty:
    china_filing_metadata_df = pd.DataFrame(
        columns=FILING_METADATA_COLUMNS
    )

else:
    disclosures = (
        cninfo_announcements_discovered_df
        .copy()
        .reset_index(drop=True)
    )

    merge_keys = [
        "stock_code",
        "cninfo_org_id",
        "exchange",
    ]

    bridge_columns = [
        "stock_code",
        "cninfo_org_id",
        "exchange",
        "security_id",
        "issuer_id",
        "issuer_name",
        "ticker",
    ]

    bridge_columns = [
        column
        for column in bridge_columns
        if column
        in china_resolved_securities_df.columns
    ]

    missing_keys = [
        key
        for key in merge_keys
        if key not in bridge_columns
    ]

    if missing_keys:
        raise RuntimeError(
            "Resolved CNINFO security table is missing: "
            f"{missing_keys}"
        )

    security_bridge = (
        china_resolved_securities_df[
            bridge_columns
        ]
        .drop_duplicates(
            subset=merge_keys,
            keep="first",
        )
        .reset_index(drop=True)
    )

    payload_columns = (
        merge_keys
        + [
            column
            for column in security_bridge.columns
            if column not in merge_keys
            and column not in disclosures.columns
        ]
    )

    china_filing_metadata_df = (
        disclosures.merge(
            security_bridge[payload_columns],
            on=merge_keys,
            how="left",
            validate="m:1",
        )
    )

    for column, default in {
        "security_id": pd.NA,
        "issuer_id": pd.NA,
        "issuer_name": pd.NA,
        "ticker": pd.NA,
        "adjunct_type": pd.NA,
    }.items():
        if column not in china_filing_metadata_df.columns:
            china_filing_metadata_df[
                column
            ] = default

    china_filing_metadata_df[
        "document_url"
    ] = china_filing_metadata_df.apply(
        lambda row: normalise_cninfo_document_url(
            row.get("document_url"),
            row.get("adjunct_url")
            if "adjunct_url"
            in china_filing_metadata_df.columns
            else pd.NA,
        ),
        axis=1,
    )

    china_filing_metadata_df[
        "available_datetime"
    ] = pd.to_datetime(
        china_filing_metadata_df[
            "release_datetime"
        ],
        errors="coerce",
        utc=True,
    )

    china_filing_metadata_df[
        "available_date"
    ] = (
        china_filing_metadata_df[
            "available_datetime"
        ]
        .dt.normalize()
    )

    china_filing_metadata_df[
        "availability_basis"
    ] = "CNINFO_ANNOUNCEMENT_TIMESTAMP"

    china_filing_metadata_df[
        "source_system"
    ] = "CNINFO"

    summary_mask = (
        china_filing_metadata_df[
            "is_summary"
        ]
        .fillna(False)
        .astype(bool)
    )

    china_filing_metadata_df[
        "document_priority"
    ] = np.where(
        summary_mask,
        2,
        1,
    )

    china_filing_metadata_df[
        "document_id"
    ] = china_filing_metadata_df.apply(
        lambda row: (
            str(row["announcement_id"])
            if pd.notna(row["announcement_id"])
            else hashlib.sha256(
                str(row["document_url"]).encode()
            ).hexdigest()[:24]
        ),
        axis=1,
    )

    duplicate_columns = [
        column
        for column in [
            "security_id",
            "announcement_id",
            "document_url",
        ]
        if column
        in china_filing_metadata_df.columns
    ]

    china_filing_metadata_df = (
        china_filing_metadata_df
        .drop_duplicates(
            subset=duplicate_columns,
            keep="first",
        )
        .reset_index(drop=True)
    )

    for column in FILING_METADATA_COLUMNS:
        if column not in china_filing_metadata_df.columns:
            china_filing_metadata_df[
                column
            ] = pd.NA

    china_filing_metadata_df = (
        china_filing_metadata_df[
            FILING_METADATA_COLUMNS
        ]
        .copy()
    )

print(
    "Linked CNINFO filing rows:",
    len(china_filing_metadata_df),
)

print(
    "Security link rate:",
    (
        china_filing_metadata_df[
            "security_id"
        ]
        .notna()
        .mean()
        if not china_filing_metadata_df.empty
        else np.nan
    ),
)

Linked CNINFO filing rows: 210
Security link rate: 1.0


In [ ]:
# 22. DOWNLOAD OFFICIAL REPORTS AND EXTRACT PDF TEXT

def document_suffix(document_url, adjunct_type=None):
    if pd.notna(adjunct_type) and str(adjunct_type).strip():
        value = str(adjunct_type).lower().lstrip(".")
        return "." + value

    path_suffix = Path(str(document_url).split("?")[0]).suffix.lower()
    return path_suffix if path_suffix else ".pdf"


def download_cninfo_document(
    document_id,
    document_url,
    adjunct_type=None,
):
    suffix = document_suffix(
        document_url,
        adjunct_type,
    )

    # CNINFO report links are expected to be PDF even when adjunct_type is
    # missing or inconsistent.
    if suffix.lower() not in {
        ".pdf",
    }:
        suffix = ".pdf"

    path = (
        CNINFO_DOCUMENT_CACHE_DIR
        / f"{document_id}{suffix}"
    )

    if path.exists():
        cached_bytes = path.read_bytes()

        if cached_bytes.startswith(b"%PDF"):
            return path, {
                "document_id": document_id,
                "status": "CACHE_HIT",
                "cache_path": str(path),
                "document_url": document_url,
                "document_host": urlparse(
                    str(document_url)
                ).netloc,
                "content_length": len(cached_bytes),
                "pdf_magic_valid": True,
            }

        # Remove invalid cached HTML or truncated content.
        path.unlink()

    content, log = cached_get(
        document_url,
        cache_dir=CNINFO_DOCUMENT_CACHE_DIR,
        suffix=suffix,
    )

    if not isinstance(content, (bytes, bytearray)):
        raise TypeError(
            "CNINFO document response was not binary content."
        )

    if not bytes(content).startswith(b"%PDF"):
        preview = bytes(content[:200]).decode(
            "utf-8",
            errors="replace",
        )

        raise RuntimeError(
            "CNINFO document did not return a valid PDF. "
            f"First bytes: {preview!r}"
        )

    path.write_bytes(
        bytes(content)
    )

    log.update({
        "document_id": document_id,
        "document_url": document_url,
        "document_host": urlparse(
            str(document_url)
        ).netloc,
        "cache_path": str(path),
        "content_length": len(content),
        "pdf_magic_valid": True,
    })

    return path, log


def extract_pdf_text(path):
    text_path = CNINFO_TEXT_CACHE_DIR / f"{path.stem}.txt"

    if text_path.exists():
        return text_path.read_text(
            encoding="utf-8",
            errors="replace",
        ), {
            "status": "CACHE_HIT",
            "text_cache_path": str(text_path),
        }

    reader = PdfReader(str(path))
    page_limit = len(reader.pages)

    if MAX_PDF_PAGES is not None:
        page_limit = min(
            page_limit,
            int(MAX_PDF_PAGES),
        )

    pages = []

    for page_number in range(page_limit):
        try:
            pages.append(
                reader.pages[page_number].extract_text() or ""
            )
        except Exception:
            pages.append("")

    text = "\n\n".join(pages)
    text_path.write_text(text, encoding="utf-8")

    return text, {
        "status": "EXTRACTED",
        "text_cache_path": str(text_path),
        "page_count": len(reader.pages),
        "pages_extracted": page_limit,
        "character_count": len(text),
    }


download_rows = []
text_rows = []

documents_to_download = (
    china_filing_metadata_df
    .sort_values(
        ["document_priority", "available_datetime"],
        ascending=[True, True],
    )
    .copy()
)

if MAX_DOCUMENTS_TO_DOWNLOAD is not None:
    documents_to_download = documents_to_download.head(
        int(MAX_DOCUMENTS_TO_DOWNLOAD)
    )

for row in tqdm(
    documents_to_download.itertuples(index=False),
    total=len(documents_to_download),
    desc="Downloading CNINFO reports",
):
    try:
        adjunct_type = getattr(
            row,
            "adjunct_type",
            None,
        )

        path, download_log = download_cninfo_document(
            row.document_id,
            row.document_url,
            adjunct_type,
        )
        download_rows.append(download_log)

        text, text_log = extract_pdf_text(path)

        text_rows.append({
            "document_id": row.document_id,
            "document_text": text,
            **text_log,
        })

    except Exception as exc:
        download_rows.append({
            "document_id": row.document_id,
            "document_url": row.document_url,
            "document_host": urlparse(
                str(row.document_url)
            ).netloc,
            "status": "FAILED",
            "error": repr(exc),
        })


CNINFO_DOCUMENT_DOWNLOAD_LOG_COLUMNS = [
    "document_id",
    "document_url",
    "document_host",
    "status",
    "cache_path",
    "content_length",
    "pdf_magic_valid",
    "error",
]

CNINFO_DOCUMENT_TEXT_COLUMNS = [
    "document_id",
    "document_text",
    "status",
    "text_cache_path",
    "page_count",
    "pages_extracted",
    "character_count",
]

cninfo_document_download_log_df = pd.DataFrame(download_rows)

for column in CNINFO_DOCUMENT_DOWNLOAD_LOG_COLUMNS:
    if column not in cninfo_document_download_log_df.columns:
        cninfo_document_download_log_df[column] = pd.NA

cninfo_document_download_log_df = (
    cninfo_document_download_log_df[
        CNINFO_DOCUMENT_DOWNLOAD_LOG_COLUMNS
    ]
    .copy()
)

cninfo_document_text_df = pd.DataFrame(text_rows)

for column in CNINFO_DOCUMENT_TEXT_COLUMNS:
    if column not in cninfo_document_text_df.columns:
        cninfo_document_text_df[column] = pd.NA

cninfo_document_text_df = (
    cninfo_document_text_df[
        CNINFO_DOCUMENT_TEXT_COLUMNS
    ]
    .copy()
)

print("Documents selected:", len(documents_to_download))
print(
    "Documents with extracted text:",
    len(cninfo_document_text_df),
)

if cninfo_document_text_df.empty:
    print(
        "No document text was extracted. "
        "Step 10 will return empty schema-stable outputs."
    )

Documents selected: 210
Documents with extracted text: 210


In [ ]:
# 23. MAINLAND CHINA ACCOUNT-LABEL DICTIONARY

CHINA_ACCOUNT_MAPPINGS = [
    # standard_concept, Chinese/English label regex, priority

    ("revenue", r"^(营业收入|营业总收入|主营业务收入|收入|revenue|operating revenue)$", 1),
    ("cost_of_revenue", r"^(营业成本|营业总成本|主营业务成本|cost of sales|cost of revenue)$", 1),
    ("gross_profit", r"^(毛利|毛利润|gross profit)$", 1),
    ("operating_income", r"^(营业利润|营业亏损|operating profit|operating loss)$", 1),
    ("profit_before_tax", r"^(利润总额|税前利润|profit before tax)$", 1),
    ("income_tax_expense", r"^(所得税费用|income tax expense)$", 1),
    ("net_income", r"^(净利润|本期利润|归属于.*净利润|net profit|profit for the period)$", 1),
    ("net_income_attributable_to_owners", r"^(归属于母公司股东的净利润|归属于上市公司股东的净利润|profit attributable to owners)$", 1),
    ("net_income_attributable_to_nci", r"^(少数股东损益|非控制性权益应占利润|noncontrolling interests)$", 1),

    ("basic_eps", r"^(基本每股收益|基本每股盈利|basic earnings per share)$", 1),
    ("diluted_eps", r"^(稀释每股收益|摊薄每股收益|diluted earnings per share)$", 1),

    ("research_and_development_expense", r"^(研发费用|研究开发费用|research and development expenses?)$", 1),
    ("selling_expense", r"^(销售费用|selling expenses?)$", 1),
    ("general_and_administrative_expense", r"^(管理费用|administrative expenses?)$", 1),
    ("selling_general_and_administrative_expense", r"^(销售及管理费用|selling general and administrative expenses?)$", 1),
    ("employee_benefit_expense", r"^(职工薪酬|员工成本|employee benefits? expense)$", 1),
    ("depreciation_expense", r"^(折旧费用|固定资产折旧|depreciation expense)$", 1),
    ("amortisation_expense", r"^(摊销费用|无形资产摊销|amortisation expense|amortization expense)$", 1),
    ("depreciation_and_amortisation", r"^(折旧及摊销|depreciation and amorti[sz]ation)$", 1),
    ("impairment_loss", r"^(资产减值损失|信用减值损失|impairment loss)$", 1),
    ("restructuring_expense", r"^(重组费用|结构调整费用|restructuring expenses?)$", 1),
    ("finance_income", r"^(财务收入|金融收益|finance income)$", 1),
    ("finance_costs", r"^(财务费用|融资成本|finance costs?)$", 1),
    ("interest_expense", r"^(利息费用|利息支出|interest expense)$", 1),
    ("interest_income", r"^(利息收入|interest income)$", 1),
    ("share_of_profit_equity_method", r"^(投资收益.*联营|权益法核算的投资收益|share of profit.*associate)$", 1),

    ("other_comprehensive_income", r"^(其他综合收益|other comprehensive income)$", 1),
    ("comprehensive_income", r"^(综合收益总额|total comprehensive income)$", 1),

    ("total_assets", r"^(资产总计|资产合计|总资产|total assets)$", 1),
    ("current_assets", r"^(流动资产合计|流动资产总计|current assets)$", 1),
    ("noncurrent_assets", r"^(非流动资产合计|非流动资产总计|non-current assets|noncurrent assets)$", 1),
    ("cash_and_cash_equivalents", r"^(货币资金|现金及现金等价物|cash and cash equivalents)$", 1),
    ("restricted_cash", r"^(受限资金|受限制货币资金|restricted cash)$", 1),
    ("short_term_investments", r"^(交易性金融资产|短期投资|short-term investments?)$", 1),

    ("trade_receivables", r"^(应收账款|应收票据及应收账款|trade receivables)$", 1),
    ("other_receivables", r"^(其他应收款|other receivables)$", 1),
    ("finance_receivables", r"^(长期应收款|融资租赁应收款|finance receivables)$", 1),
    ("inventory", r"^(存货|inventories|inventory)$", 1),
    ("raw_material_inventory", r"^(原材料|raw materials)$", 1),
    ("work_in_progress_inventory", r"^(在产品|在制品|work in progress)$", 1),
    ("finished_goods_inventory", r"^(库存商品|产成品|finished goods)$", 1),

    ("property_plant_equipment", r"^(固定资产|物业厂房及设备|property plant and equipment)$", 1),
    ("right_of_use_assets", r"^(使用权资产|right-of-use assets)$", 1),
    ("goodwill", r"^(商誉|goodwill)$", 1),
    ("intangible_assets", r"^(无形资产|intangible assets)$", 1),
    ("capitalised_development_costs", r"^(开发支出|资本化开发支出|development costs)$", 1),
    ("investments_in_associates", r"^(长期股权投资|对联营企业投资|investments in associates)$", 1),
    ("deferred_tax_assets", r"^(递延所得税资产|deferred tax assets)$", 1),
    ("pension_assets", r"^(设定受益计划资产|退休福利资产|pension assets)$", 1),

    ("total_liabilities", r"^(负债合计|负债总计|总负债|total liabilities)$", 1),
    ("current_liabilities", r"^(流动负债合计|流动负债总计|current liabilities)$", 1),
    ("noncurrent_liabilities", r"^(非流动负债合计|非流动负债总计|non-current liabilities|noncurrent liabilities)$", 1),
    ("trade_payables", r"^(应付账款|应付票据及应付账款|trade payables)$", 1),
    ("other_payables", r"^(其他应付款|other payables)$", 1),
    ("contract_liabilities", r"^(合同负债|预收款项|contract liabilities)$", 1),

    ("short_term_debt", r"^(短期借款|一年内到期的非流动负债|current borrowings|short-term borrowings)$", 1),
    ("long_term_debt", r"^(长期借款|应付债券|long-term borrowings|bonds payable)$", 1),
    ("total_borrowings", r"^(有息负债|借款总额|total borrowings)$", 1),
    ("current_lease_liabilities", r"^(一年内到期的租赁负债|流动租赁负债|current lease liabilities)$", 1),
    ("noncurrent_lease_liabilities", r"^(租赁负债|非流动租赁负债|non-current lease liabilities)$", 1),

    ("provisions_current", r"^(流动预计负债|current provisions)$", 1),
    ("provisions_noncurrent", r"^(预计负债|非流动预计负债|non-current provisions)$", 1),
    ("warranty_provisions", r"^(产品质量保证|售后服务预计负债|保修准备|warranty provisions)$", 1),
    ("restructuring_provisions", r"^(重组准备|重组预计负债|restructuring provisions)$", 1),
    ("pension_liabilities", r"^(长期应付职工薪酬|退休福利负债|pension liabilities)$", 1),
    ("deferred_tax_liabilities", r"^(递延所得税负债|deferred tax liabilities)$", 1),

    ("total_equity", r"^(所有者权益合计|股东权益合计|权益总额|total equity)$", 1),
    ("equity_attributable_to_owners", r"^(归属于母公司所有者权益合计|归属于上市公司股东的所有者权益|equity attributable to owners)$", 1),
    ("noncontrolling_interests", r"^(少数股东权益|非控制性权益|noncontrolling interests)$", 1),
    ("share_capital", r"^(股本|实收资本|share capital)$", 1),
    ("share_premium", r"^(资本公积|股本溢价|share premium)$", 1),
    ("retained_earnings", r"^(未分配利润|留存收益|retained earnings)$", 1),
    ("treasury_shares", r"^(库存股|treasury shares)$", 1),

    ("operating_cash_flow", r"^(经营活动产生的现金流量净额|net cash flows? from operating activities)$", 1),
    ("investing_cash_flow", r"^(投资活动产生的现金流量净额|net cash flows? from investing activities)$", 1),
    ("financing_cash_flow", r"^(筹资活动产生的现金流量净额|net cash flows? from financing activities)$", 1),
    ("capital_expenditure", r"^(购建固定资产.*支付的现金|资本性支出|capital expenditure)$", 1),
    ("intangible_asset_purchases", r"^(购置无形资产.*支付的现金|purchase of intangible assets)$", 1),
    ("business_acquisition_cash_outflow", r"^(取得子公司.*支付的现金净额|business acquisition cash outflow)$", 1),
    ("business_disposal_cash_inflow", r"^(处置子公司.*收到的现金净额|business disposal cash inflow)$", 1),
    ("debt_issuance", r"^(取得借款收到的现金|发行债券收到的现金|debt issuance)$", 1),
    ("debt_repayment", r"^(偿还债务支付的现金|debt repayment)$", 1),
    ("lease_payments", r"^(支付租赁负债本金|lease payments)$", 1),
    ("dividends_paid", r"^(分配股利.*支付的现金|支付的股利|dividends paid)$", 1),
    ("share_repurchases", r"^(回购股份支付的现金|share repurchases)$", 1),
    ("share_issuance_proceeds", r"^(吸收投资收到的现金|发行股份收到的现金|share issuance proceeds)$", 1),
    ("interest_paid", r"^(偿付利息支付的现金|支付的利息|interest paid)$", 1),
    ("interest_received", r"^(取得利息收到的现金|收到的利息|interest received)$", 1),
    ("income_taxes_paid", r"^(支付的各项税费|支付的所得税|income taxes paid)$", 1),
    ("cash_change", r"^(现金及现金等价物净增加额|net increase in cash and cash equivalents)$", 1),

    ("vehicle_sales_volume", r"^(汽车销量|整车销量|车辆销量|vehicle sales volume)$", 1),
    ("vehicle_production_volume", r"^(汽车产量|整车产量|车辆产量|vehicle production volume)$", 1),
    ("automotive_revenue", r"^(汽车业务收入|整车业务收入|automotive revenue)$", 1),
    ("financial_services_revenue", r"^(汽车金融收入|金融服务收入|financial services revenue)$", 1),
    ("automotive_debt", r"^(汽车业务债务|automotive debt)$", 1),
    ("financial_services_debt", r"^(金融服务债务|financial services debt)$", 1),
]

china_source_account_mapping_proposed_df = pd.DataFrame(
    CHINA_ACCOUNT_MAPPINGS,
    columns=[
        "standard_concept",
        "account_label_regex",
        "priority",
    ],
)

canonical_concepts = set(
    global_canonical_schema_df[
        "standard_concept"
    ]
    .dropna()
    .astype("string")
)

china_unsupported_concept_mappings_df = (
    china_source_account_mapping_proposed_df[
        ~china_source_account_mapping_proposed_df[
            "standard_concept"
        ].isin(canonical_concepts)
    ]
    .copy()
    .reset_index(drop=True)
)

china_source_account_mapping_df = (
    china_source_account_mapping_proposed_df[
        china_source_account_mapping_proposed_df[
            "standard_concept"
        ].isin(canonical_concepts)
    ]
    .copy()
    .reset_index(drop=True)
)

china_standard_concept_dictionary_df = (
    china_source_account_mapping_df.merge(
        global_canonical_schema_df,
        on="standard_concept",
        how="left",
        validate="m:1",
    )
)

unresolved_after_filter = (
    china_standard_concept_dictionary_df[
        china_standard_concept_dictionary_df[
            "statement_type"
        ].isna()
    ]["standard_concept"]
    .drop_duplicates()
    .tolist()
)

if unresolved_after_filter:
    raise RuntimeError(
        "Canonical merge failed after filtering: "
        f"{unresolved_after_filter}"
    )

print(
    "Proposed Chinese mapping rows:",
    len(china_source_account_mapping_proposed_df),
)
print(
    "Supported Chinese mapping rows:",
    len(china_source_account_mapping_df),
)
print(
    "Canonical concepts represented:",
    china_standard_concept_dictionary_df[
        "standard_concept"
    ].nunique(),
)
print(
    "Unsupported proposed concepts:",
    china_unsupported_concept_mappings_df[
        "standard_concept"
    ].nunique(),
)

if not china_unsupported_concept_mappings_df.empty:
    display(
        china_unsupported_concept_mappings_df
    )

Proposed Chinese mapping rows: 96
Supported Chinese mapping rows: 96
Canonical concepts represented: 96
Unsupported proposed concepts: 0


In [ ]:
# 24. EXTRACT ACCOUNT/VALUE CANDIDATES FROM REPORT TEXT

NUMBER_PATTERN = re.compile(
    r"(?P<value>\(?-?\d[\d,]*(?:\.\d+)?\)?)"
    r"(?:\s*(?P<unit>元|万元|亿元|千元|million|billion|thousand))?",
    flags=re.IGNORECASE,
)

CURRENCY_PATTERN = re.compile(
    r"(人民币|RMB|CNY|美元|USD|US\$|港币|港元|HK\$|欧元|EUR)",
    flags=re.IGNORECASE,
)

GENERIC_FUZZY_LABEL_STOPLIST = {
    "本期",
    "上期",
    "本年",
    "上年",
    "合计",
    "其中",
    "项目",
    "附注",
    "total",
    "note",
    "notes",
    "of",
    "and",
    "the",
    "for",
    "from",
    "year",
    "period",
}

FUZZY_SINGLE_TOKEN_WHITELIST = {
    "revenue",
    "turnover",
    "inventory",
    "inventories",
    "goodwill",
    "taxation",
    "cash",
    "equity",
    "borrowings",
}


def normalise_label(text):
    text = html_lib.unescape(str(text))
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(
        r"^[\-–—•·、：:\s\d\.\(\)]+",
        "",
        text,
    )
    text = re.sub(
        r"[\-–—•·、：:\s]+$",
        "",
        text,
    )
    return text.strip()


def canonicalise_label(text):
    text = normalise_label(text).lower()
    text = re.sub(
        r"(附注|注释|单位[:：]?\s*(人民币)?(元|万元|亿元|千元)|"
        r"本期金额|上期金额|期末余额|期初余额)",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"[^\w\u3400-\u9fff&\- ]+",
        " ",
        text,
    )
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_number(value_text, unit_text=None):
    if value_text is None:
        return np.nan

    text = str(value_text).replace(",", "").strip()
    negative = (
        text.startswith("(")
        and text.endswith(")")
    )
    text = text.strip("()")

    value = pd.to_numeric(
        text,
        errors="coerce",
    )

    if pd.isna(value):
        return np.nan

    if negative:
        value = -value

    multiplier = {
        "元": 1,
        "千元": 1_000,
        "万元": 10_000,
        "亿元": 100_000_000,
        "thousand": 1_000,
        "million": 1_000_000,
        "billion": 1_000_000_000,
    }.get(
        str(unit_text).lower()
        if unit_text
        else "",
        1,
    )

    return float(value) * multiplier


def infer_currency(text):
    matches = CURRENCY_PATTERN.findall(
        str(text)[:25_000]
    )

    if not matches:
        return "CNY"

    mapping = {
        "人民币": "CNY",
        "RMB": "CNY",
        "CNY": "CNY",
        "美元": "USD",
        "USD": "USD",
        "US$": "USD",
        "港币": "HKD",
        "港元": "HKD",
        "HK$": "HKD",
        "欧元": "EUR",
        "EUR": "EUR",
    }

    normalised = [
        mapping.get(
            str(match).upper(),
            mapping.get(str(match), str(match)),
        )
        for match in matches
    ]

    return (
        pd.Series(normalised)
        .value_counts()
        .index[0]
    )


def infer_statement_type(lines, line_number):
    context = " ".join(
        lines[max(0, line_number - 12): line_number + 1]
    )

    if re.search(
        r"(资产负债表|财务状况表|balance sheet|statement of financial position)",
        context,
        flags=re.IGNORECASE,
    ):
        return "BALANCE_SHEET"

    if re.search(
        r"(利润表|损益表|income statement|statement of profit)",
        context,
        flags=re.IGNORECASE,
    ):
        return "INCOME_STATEMENT"

    if re.search(
        r"(现金流量表|cash flow statement)",
        context,
        flags=re.IGNORECASE,
    ):
        return "CASH_FLOW"

    if re.search(
        r"(所有者权益变动表|股东权益变动表|statement of changes in equity)",
        context,
        flags=re.IGNORECASE,
    ):
        return "EQUITY_STATEMENT"

    return pd.NA


def fuzzy_label_quality(account_label):
    if pd.isna(account_label):
        return False, "MISSING_LABEL", 0, 0

    text = str(account_label).strip().lower()
    tokens = re.findall(
        r"[a-z0-9]+|[\u3400-\u9fff]+",
        text,
        flags=re.IGNORECASE,
    )
    character_count = len(
        re.sub(r"\s+", "", text)
    )
    token_count = len(tokens)
    contains_cjk = bool(
        re.search(r"[\u3400-\u9fff]", text)
    )

    if not text:
        reason = "EMPTY_LABEL"
    elif text in GENERIC_FUZZY_LABEL_STOPLIST:
        reason = "GENERIC_STOPWORD"
    elif contains_cjk and character_count < 2:
        reason = "CJK_LABEL_TOO_SHORT"
    elif not contains_cjk and character_count < 4:
        reason = "LATIN_LABEL_TOO_SHORT"
    elif (
        not contains_cjk
        and token_count == 1
        and text not in FUZZY_SINGLE_TOKEN_WHITELIST
    ):
        reason = "UNAPPROVED_SINGLE_TOKEN"
    else:
        reason = "ELIGIBLE"

    return (
        reason == "ELIGIBLE",
        reason,
        character_count,
        token_count,
    )


def extract_account_candidates(
    document_id,
    document_text,
):
    lines = [
        normalise_label(line)
        for line in str(document_text).splitlines()
    ]
    lines = [line for line in lines if line]

    currency = infer_currency(document_text)
    rows = []

    for line_number, line in enumerate(lines):
        number_matches = list(
            NUMBER_PATTERN.finditer(line)
        )

        if not number_matches:
            continue

        first_number = number_matches[0]
        raw_label = normalise_label(
            line[: first_number.start()]
        )
        account_label = canonicalise_label(
            raw_label
        )

        if len(account_label) < 2:
            continue

        if len(account_label) > 220:
            continue

        if re.fullmatch(
            r"[\d\W_]+",
            account_label,
        ):
            continue

        if re.fullmatch(
            r"(本期发生额|上期发生额|本期金额|上期金额|"
            r"期末余额|期初余额|current period|prior period|"
            r"ending balance|opening balance)",
            account_label,
            flags=re.IGNORECASE,
        ):
            continue

        statement_type_inferred = (
            infer_statement_type(
                lines,
                line_number,
            )
        )

        eligible, reason, chars, tokens = (
            fuzzy_label_quality(
                account_label
            )
        )

        for value_rank, match in enumerate(
            number_matches[:6],
            start=1,
        ):
            rows.append({
                "document_id": document_id,
                "line_number": line_number,
                "source_line": line,
                "account_label_raw": raw_label,
                "account_label": account_label,
                "value_rank_in_line": value_rank,
                "reported_value_text": match.group("value"),
                "scale_label": match.group("unit"),
                "reported_value": parse_number(
                    match.group("value"),
                    match.group("unit"),
                ),
                "currency": currency,
                "statement_type_inferred": (
                    statement_type_inferred
                ),
                "fuzzy_label_is_eligible": eligible,
                "fuzzy_label_rejection_reason": reason,
                "fuzzy_label_character_count": chars,
                "fuzzy_label_token_count": tokens,
            })

    return pd.DataFrame(rows)


ACCOUNT_CANDIDATE_COLUMNS = [
    "document_id",
    "line_number",
    "source_line",
    "account_label_raw",
    "account_label",
    "value_rank_in_line",
    "reported_value_text",
    "scale_label",
    "reported_value",
    "currency",
    "statement_type_inferred",
    "fuzzy_label_is_eligible",
    "fuzzy_label_rejection_reason",
    "fuzzy_label_character_count",
    "fuzzy_label_token_count",
]

candidate_frames = []

required_document_columns = {
    "document_id",
    "document_text",
}

missing_document_columns = (
    required_document_columns
    .difference(cninfo_document_text_df.columns)
)

if missing_document_columns:
    raise RuntimeError(
        "cninfo_document_text_df is missing required columns: "
        f"{sorted(missing_document_columns)}"
    )

documents_with_usable_text_df = (
    cninfo_document_text_df[
        cninfo_document_text_df["document_id"].notna()
        & cninfo_document_text_df["document_text"].notna()
        & cninfo_document_text_df[
            "document_text"
        ].astype("string").str.strip().ne("")
    ][
        [
            "document_id",
            "document_text",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

for row in tqdm(
    documents_with_usable_text_df.itertuples(index=False),
    total=len(documents_with_usable_text_df),
    desc="Extracting CNINFO account candidates",
):
    frame = extract_account_candidates(
        row.document_id,
        row.document_text,
    )

    if not frame.empty:
        candidate_frames.append(frame)

china_account_candidates_raw_df = (
    pd.concat(
        candidate_frames,
        ignore_index=True,
    )
    if candidate_frames
    else pd.DataFrame(
        columns=ACCOUNT_CANDIDATE_COLUMNS
    )
)

for column in ACCOUNT_CANDIDATE_COLUMNS:
    if column not in china_account_candidates_raw_df.columns:
        china_account_candidates_raw_df[column] = pd.NA

china_account_candidates_raw_df = (
    china_account_candidates_raw_df[
        ACCOUNT_CANDIDATE_COLUMNS
    ]
    .copy()
)

print(
    "Raw Chinese account/value candidate rows:",
    len(china_account_candidates_raw_df),
)

print(
    "Unique cleaned account labels:",
    (
        china_account_candidates_raw_df[
            "account_label"
        ].nunique()
        if not china_account_candidates_raw_df.empty
        else 0
    ),
)

print(
    "Documents with usable extracted text:",
    len(documents_with_usable_text_df),
)

if documents_with_usable_text_df.empty:
    print(
        "No usable report text is available. "
        "Inspect cninfo_document_download_log_df for download or PDF errors."
    )


Extracting CNINFO account candidates:   0%|          | 0/210 [00:00<?, ?it/s]

Raw Chinese account/value candidate rows: 507757
Unique cleaned account labels: 43425
Documents with usable extracted text: 210


In [ ]:
# 25. MAP ACCOUNT LABELS, SELECT FACTS AND ASSIGN SOURCE PRECEDENCE

CANONICAL_ANCHORS = {
    concept: [
        re.sub(
            r"[\^\$\(\)\?\:\[\]\{\}\\\|\*\+]",
            " ",
            pattern,
        )
        for pattern in group[
            "account_label_regex"
        ].astype(str)
    ]
    for concept, group in (
        china_source_account_mapping_df
        .groupby("standard_concept")
    )
}


def deterministic_mapping_candidates(label):
    dictionary = (
        china_standard_concept_dictionary_df
    )

    matched = dictionary[
        dictionary[
            "account_label_regex"
        ].map(
            lambda pattern: bool(
                re.search(
                    pattern,
                    str(label).strip(),
                    flags=re.IGNORECASE,
                )
            )
        )
    ].copy()

    if not matched.empty:
        matched["mapping_method"] = (
            "DETERMINISTIC_REGEX"
        )
        matched["mapping_score"] = (
            100.0
            - matched["priority"].astype(float)
        )

    return matched


def china_block9_registry_mapping_candidates(label):
    """Return one unambiguous, previously accepted Block 9 synonym."""
    normalised_label = normalise_block9_synonym_label(label)

    if pd.isna(normalised_label):
        return pd.DataFrame()

    registry_row = block9_synonym_lookup.get(normalised_label)

    if not registry_row:
        return pd.DataFrame()

    standard_concept = registry_row.get("standard_concept")

    if pd.isna(standard_concept):
        return pd.DataFrame()

    result = pd.DataFrame([
        {
            "standard_concept": standard_concept,
            "anchor_label": normalised_label,
            "mapping_method": "BLOCK9_ACCEPTED_SYNONYM",
            "mapping_score": 100.0,
            "priority": 2,
            "block9_registry_accepted_observations": (
                registry_row.get("accepted_observations")
            ),
            "block9_registry_unique_issuers": (
                registry_row.get("unique_issuers")
            ),
            "block9_registry_unique_filings": (
                registry_row.get("unique_filings")
            ),
            "block9_registry_generation": (
                registry_row.get("registry_generation")
            ),
        }
    ])

    return result.merge(
        global_canonical_schema_df,
        on="standard_concept",
        how="left",
        validate="m:1",
    )


anchor_rows = []

for standard_concept, anchors in CANONICAL_ANCHORS.items():
    for anchor in anchors:
        cleaned = canonicalise_label(anchor)

        if cleaned:
            anchor_rows.append({
                "standard_concept": standard_concept,
                "anchor_label": cleaned,
            })

china_canonical_anchor_df = (
    pd.DataFrame(anchor_rows)
    .drop_duplicates()
    .reset_index(drop=True)
)

anchor_choices = (
    china_canonical_anchor_df[
        "anchor_label"
    ].tolist()
)


def fuzzy_mapping_candidates(label):
    if not label or not anchor_choices:
        return pd.DataFrame()

    matches = process.extract(
        label,
        anchor_choices,
        scorer=fuzz.token_set_ratio,
        limit=5,
    )

    rows = []

    for matched_anchor, score, index in matches:
        standard_concept = (
            china_canonical_anchor_df
            .iloc[index][
                "standard_concept"
            ]
        )

        rows.append({
            "standard_concept": standard_concept,
            "anchor_label": matched_anchor,
            "mapping_method": "FUZZY_ANCHOR",
            "mapping_score": float(score),
            "priority": 99,
        })

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .merge(
            global_canonical_schema_df,
            on="standard_concept",
            how="left",
            validate="m:1",
        )
    )


if china_account_candidates_raw_df.empty:
    china_observed_account_mapping_df = pd.DataFrame()
    china_block9_registry_matches_df = pd.DataFrame()
    china_fundamentals_mapped_df = pd.DataFrame()
    china_mapping_alternatives_df = pd.DataFrame()
    china_mapping_review_queue_df = pd.DataFrame()
    china_low_quality_fuzzy_rejections_df = pd.DataFrame()
    china_fundamentals_standardised_df = pd.DataFrame()
    china_unmapped_account_inventory_df = pd.DataFrame()
    china_automotive_extension_candidates_df = pd.DataFrame()

else:
    mapping_frames = []

    label_quality = (
        china_account_candidates_raw_df[
            [
                "account_label",
                "fuzzy_label_is_eligible",
                "fuzzy_label_rejection_reason",
                "fuzzy_label_character_count",
                "fuzzy_label_token_count",
            ]
        ]
        .drop_duplicates("account_label")
    )

    eligible_label_set = set(
        label_quality.loc[
            label_quality[
                "fuzzy_label_is_eligible"
            ].fillna(False),
            "account_label",
        ].astype("string")
    )

    for label in tqdm(
        china_account_candidates_raw_df[
            "account_label"
        ]
        .dropna()
        .astype("string")
        .drop_duplicates(),
        desc="Mapping Chinese account labels",
    ):
        deterministic = (
            deterministic_mapping_candidates(
                label
            )
        )

        if not deterministic.empty:
            deterministic["account_label"] = label
            mapping_frames.append(deterministic)
            continue

        registry_mapping = (
            china_block9_registry_mapping_candidates(
                label
            )
        )

        if not registry_mapping.empty:
            registry_mapping["account_label"] = label
            mapping_frames.append(registry_mapping)
            continue

        if str(label) not in eligible_label_set:
            continue

        fuzzy = fuzzy_mapping_candidates(label)

        if not fuzzy.empty:
            fuzzy["account_label"] = label
            mapping_frames.append(fuzzy)

    china_observed_account_mapping_df = (
        pd.concat(
            mapping_frames,
            ignore_index=True,
        )
        if mapping_frames
        else pd.DataFrame()
    )


    china_block9_registry_matches_df = (
        china_observed_account_mapping_df.loc[
            china_observed_account_mapping_df[
                "mapping_method"
            ]
            .astype("string")
            .eq("BLOCK9_ACCEPTED_SYNONYM")
        ]
        .copy()
        .reset_index(drop=True)
        if (
            not china_observed_account_mapping_df.empty
            and "mapping_method"
            in china_observed_account_mapping_df.columns
        )
        else pd.DataFrame()
    )

    mapped = (
        china_account_candidates_raw_df
        .merge(
            china_observed_account_mapping_df,
            on="account_label",
            how="left",
            validate="m:m",
        )
    )

    filing_bridge_columns = [
        "document_id",
        "announcement_id",
        "stock_code",
        "cninfo_short_name",
        "announcement_title",
        "report_type",
        "document_url",
        "available_datetime",
        "available_date",
        "availability_basis",
        "document_priority",
        "is_summary",
        "security_id",
        "issuer_id",
        "issuer_name",
        "ticker",
    ]

    filing_bridge = (
        china_filing_metadata_df[
            [
                column
                for column in filing_bridge_columns
                if column
                in china_filing_metadata_df.columns
            ]
        ]
        .drop_duplicates("document_id")
    )

    mapped = mapped.merge(
        filing_bridge,
        on="document_id",
        how="left",
        validate="m:1",
    )

    observed_family = np.where(
        mapped["standard_concept"].isin(
            ["basic_eps", "diluted_eps"]
        ),
        "PER_SHARE",
        np.where(
            mapped["standard_concept"].isin(
                [
                    "vehicle_sales_volume",
                    "vehicle_production_volume",
                    "shares_outstanding",
                    "basic_weighted_average_shares",
                    "diluted_weighted_average_shares",
                ]
            ),
            "COUNT",
            "MONETARY",
        ),
    )

    mapped["observed_unit_family"] = observed_family

    inferred_statement = (
        mapped["statement_type_inferred"]
        .astype("string")
    )
    expected_statement = (
        mapped["statement_type"]
        .astype("string")
    )

    mapped["statement_type_match"] = (
        inferred_statement.isna()
        | expected_statement.isna()
        | inferred_statement.fillna("").eq(
            expected_statement.fillna("")
        )
    ).fillna(True).astype(bool)

    observed_unit = (
        mapped["observed_unit_family"]
        .astype("string")
    )
    expected_unit = (
        mapped["expected_unit_family"]
        .astype("string")
    )

    mapped["unit_family_match"] = (
        observed_unit.isna()
        | expected_unit.isna()
        | observed_unit.fillna("").eq(
            expected_unit.fillna("")
        )
    ).fillna(True).astype(bool)

    mapped["is_numeric_fact"] = (
        mapped["reported_value"]
        .notna()
        .astype(bool)
    )

    deterministic_mask = (
        mapped["mapping_method"]
        .astype("string")
        .eq("DETERMINISTIC_REGEX")
        .fillna(False)
    )

    block9_registry_mask = (
        mapped["mapping_method"]
        .astype("string")
        .eq("BLOCK9_ACCEPTED_SYNONYM")
        .fillna(False)
    )

    fuzzy_accept_mask = (
        pd.to_numeric(
            mapped["mapping_score"],
            errors="coerce",
        )
        .ge(FUZZY_ACCEPT_SCORE)
        .fillna(False)
    )

    mapped["mapping_is_accepted"] = (
        deterministic_mask
        & mapped["is_numeric_fact"]
    ) | (
        block9_registry_mask
        & mapped["statement_type_match"].fillna(False)
        & mapped["unit_family_match"].fillna(False)
        & mapped["is_numeric_fact"]
    ) | (
        ~deterministic_mask
        & ~block9_registry_mask
        & fuzzy_accept_mask
        & mapped[
            "fuzzy_label_is_eligible"
        ].fillna(False)
        & mapped["statement_type_match"]
        & mapped["unit_family_match"]
        & mapped["is_numeric_fact"]
    )

    mapped["mapping_is_accepted"] = (
        mapped["mapping_is_accepted"]
        .fillna(False)
        .astype(bool)
    )

    fuzzy_review_mask = (
        mapped["mapping_method"]
        .astype("string")
        .eq("FUZZY_ANCHOR")
        .fillna(False)
        & pd.to_numeric(
            mapped["mapping_score"],
            errors="coerce",
        )
        .between(
            FUZZY_REVIEW_SCORE,
            FUZZY_ACCEPT_SCORE - 0.0001,
            inclusive="both",
        )
        .fillna(False)
        & mapped[
            "fuzzy_label_is_eligible"
        ].fillna(False)
    )

    china_mapping_review_queue_df = (
        mapped[fuzzy_review_mask]
        .copy()
        .sort_values(
            ["mapping_score", "account_label"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    china_low_quality_fuzzy_rejections_df = (
        mapped[
            mapped["mapping_method"]
            .astype("string")
            .eq("FUZZY_ANCHOR")
            .fillna(False)
            & ~mapped[
                "fuzzy_label_is_eligible"
            ].fillna(False)
        ]
        .copy()
        .reset_index(drop=True)
    )

    accepted = (
        mapped[
            mapped["mapping_is_accepted"]
        ]
        .copy()
    )

    mapping_score = pd.to_numeric(
        accepted["mapping_score"],
        errors="coerce",
    ).fillna(0)

    priority = pd.to_numeric(
        accepted["priority"],
        errors="coerce",
    ).fillna(99)

    value_rank = pd.to_numeric(
        accepted["value_rank_in_line"],
        errors="coerce",
    ).fillna(1)

    document_priority = pd.to_numeric(
        accepted["document_priority"],
        errors="coerce",
    ).fillna(99)

    accepted["selection_score"] = (
        np.where(
            accepted["mapping_method"]
            .astype("string")
            .eq("DETERMINISTIC_REGEX")
            .fillna(False),
            0,
            200 - mapping_score,
        )
        + priority * 2
        + (value_rank - 1) * 10
        + (~accepted[
            "statement_type_match"
        ].astype(bool)).astype(int) * 25
        + (~accepted[
            "unit_family_match"
        ].astype(bool)).astype(int) * 20
        + document_priority * 3
    )

    duplicate_key = [
        "issuer_id",
        "security_id",
        "document_id",
        "standard_concept",
        "available_datetime",
        "currency",
    ]

    for column in duplicate_key:
        if column not in accepted.columns:
            accepted[column] = pd.NA

    accepted = (
        accepted
        .sort_values(
            duplicate_key
            + [
                "selection_score",
                "line_number",
                "value_rank_in_line",
            ]
        )
        .reset_index(drop=True)
    )

    accepted[
        "concept_selection_rank"
    ] = (
        accepted.groupby(
            duplicate_key,
            dropna=False,
        )
        .cumcount()
        + 1
    )

    accepted[
        "is_selected_standard_fact"
    ] = accepted[
        "concept_selection_rank"
    ].eq(1)

    china_fundamentals_mapped_df = (
        accepted.copy()
    )

    china_mapping_alternatives_df = (
        accepted[
            ~accepted[
                "is_selected_standard_fact"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    china_fundamentals_standardised_df = (
        accepted[
            accepted[
                "is_selected_standard_fact"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    mapped_labels = set(
        accepted[
            "account_label"
        ]
        .dropna()
        .astype("string")
    )

    unmapped = (
        china_account_candidates_raw_df[
            ~china_account_candidates_raw_df[
                "account_label"
            ]
            .astype("string")
            .isin(mapped_labels)
        ]
        .copy()
    )

    china_unmapped_account_inventory_df = (
        unmapped
        .groupby(
            "account_label",
            dropna=False,
        )
        .agg(
            candidate_rows=(
                "reported_value",
                "size",
            ),
            document_count=(
                "document_id",
                "nunique",
            ),
            numeric_share=(
                "reported_value",
                lambda s: s.notna().mean(),
            ),
            example_raw_label=(
                "account_label_raw",
                "first",
            ),
            example_line=(
                "source_line",
                "first",
            ),
            fuzzy_label_rejection_reason=(
                "fuzzy_label_rejection_reason",
                "first",
            ),
        )
        .reset_index()
        .sort_values(
            ["document_count", "candidate_rows"],
            ascending=[False, False],
        )
        .reset_index(drop=True)
    )

    automotive_pattern = re.compile(
        r"(汽车|车辆|整车|新能源|电动车|电池|充电|"
        r"产量|销量|交付|保修|售后|自动驾驶|"
        r"vehicle|automotive|battery|charging|"
        r"production|sales volume|delivery|warranty)",
        flags=re.IGNORECASE,
    )

    china_automotive_extension_candidates_df = (
        china_unmapped_account_inventory_df[
            china_unmapped_account_inventory_df[
                "account_label"
            ]
            .astype("string")
            .str.contains(
                automotive_pattern,
                na=False,
            )
        ]
        .copy()
        .reset_index(drop=True)
    )


def first_present_table(table_dict, names):
    for name in names:
        frame = table_dict.get(name)

        if isinstance(frame, pd.DataFrame):
            return name, frame

    return None, pd.DataFrame()


def structured_issuer_coverage(
    source_system,
    table_dict,
    candidate_names,
):
    table_name, frame = first_present_table(
        table_dict,
        candidate_names,
    )

    columns = [
        "issuer_id",
        "upstream_source_system",
        "upstream_source_table",
        "upstream_fact_rows",
        "upstream_unique_concepts",
    ]

    if frame.empty or "issuer_id" not in frame.columns:
        return pd.DataFrame(columns=columns)

    working = frame[
        frame["issuer_id"].notna()
    ].copy()

    if working.empty:
        return pd.DataFrame(columns=columns)

    concept_column = (
        "standard_concept"
        if "standard_concept" in working.columns
        else None
    )

    coverage = (
        working.groupby("issuer_id")
        .size()
        .rename("upstream_fact_rows")
        .reset_index()
    )

    if concept_column:
        concepts = (
            working.groupby("issuer_id")[
                concept_column
            ]
            .nunique()
            .rename("upstream_unique_concepts")
            .reset_index()
        )

        coverage = coverage.merge(
            concepts,
            on="issuer_id",
            how="left",
            validate="1:1",
        )
    else:
        coverage[
            "upstream_unique_concepts"
        ] = 0

    coverage[
        "upstream_source_system"
    ] = source_system

    coverage[
        "upstream_source_table"
    ] = table_name

    return coverage[columns]



upstream_frames = [
    structured_issuer_coverage(
        "SEC",
        block_3_inputs,
        [
            "sec_fundamentals_security_linked_df",
            "sec_fundamentals_standardised_df",
        ],
    ),
    structured_issuer_coverage(
        "HKEX",
        block_7_inputs,
        ["hong_kong_fundamentals_standardised_df"],
    ),
    structured_issuer_coverage(
        "EDINET",
        block_5_inputs,
        ["japan_fundamentals_standardised_df"],
    ),
    structured_issuer_coverage(
        "DART",
        block_6_inputs,
        ["korea_fundamentals_standardised_df"],
    ),
]

upstream_structured_issuer_coverage_df = (
    pd.concat(
        [frame for frame in upstream_frames if not frame.empty],
        ignore_index=True,
    )
    if any(not frame.empty for frame in upstream_frames)
    else pd.DataFrame(
        columns=[
            "issuer_id",
            "upstream_source_system",
            "upstream_source_table",
            "upstream_fact_rows",
            "upstream_unique_concepts",
        ]
    )
)

if upstream_structured_issuer_coverage_df.empty:
    preferred_upstream_issuer_source_df = pd.DataFrame(
        columns=[
            "issuer_id",
            "upstream_source_system",
            "upstream_source_table",
            "upstream_fact_rows",
            "upstream_unique_concepts",
            "precedence_rank",
        ]
    )
else:
    upstream_structured_issuer_coverage_df[
        "precedence_rank"
    ] = (
        upstream_structured_issuer_coverage_df[
            "upstream_source_system"
        ]
        .map(CHINA_PRIMARY_SOURCE_PRECEDENCE)
        .fillna(99)
    )

    preferred_upstream_issuer_source_df = (
        upstream_structured_issuer_coverage_df
        .sort_values(
            [
                "issuer_id",
                "precedence_rank",
                "upstream_fact_rows",
            ],
            ascending=[True, True, False],
        )
        .drop_duplicates("issuer_id")
        .reset_index(drop=True)
    )


def collect_upstream_standardised_facts():
    frames = []

    specifications = [
        (
            "SEC",
            block_3_inputs,
            [
                "sec_fundamentals_standardised_df",
                "sec_fundamentals_security_linked_df",
            ],
        ),
        (
            "HKEX",
            block_7_inputs,
            ["hong_kong_fundamentals_standardised_df"],
        ),
    ]

    for source_system, source_dict, table_names in specifications:
        for table_name in table_names:
            dataframe = source_dict.get(table_name)

            if dataframe is None or dataframe.empty:
                continue

            if "issuer_id" not in dataframe.columns:
                continue

            standard_concept_column = first_existing_column(
                dataframe,
                [
                    "standard_concept",
                    "canonical_concept",
                ],
            )

            if standard_concept_column is None:
                continue

            selected = dataframe.copy()

            selected["standard_concept"] = selected[
                standard_concept_column
            ]

            selected["primary_source_system"] = source_system
            selected["primary_source_table"] = table_name

            frames.append(selected)

            break

    return (
        pd.concat(
            frames,
            ignore_index=True,
            sort=False,
        )
        if frames
        else pd.DataFrame(
            columns=[
                "issuer_id",
                "standard_concept",
                "primary_source_system",
                "primary_source_table",
            ]
        )
    )


china_primary_source_facts_df = collect_upstream_standardised_facts()

china_primary_source_concept_inventory_df = (
    china_primary_source_facts_df[
        [
            "issuer_id",
            "standard_concept",
            "primary_source_system",
            "primary_source_table",
        ]
    ]
    .dropna(
        subset=[
            "issuer_id",
            "standard_concept",
        ]
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

cninfo_issuer_coverage_df = (
    china_fundamentals_standardised_df
    .groupby("issuer_id")
    .agg(
        cninfo_fact_rows=("reported_value", "size"),
        cninfo_unique_concepts=("standard_concept", "nunique"),
        cninfo_filing_count=("document_id", "nunique"),
    )
    .reset_index()
    if not china_fundamentals_standardised_df.empty
    else pd.DataFrame(
        columns=[
            "issuer_id",
            "cninfo_fact_rows",
            "cninfo_unique_concepts",
            "cninfo_filing_count",
        ]
    )
)

china_issuer_enrichment_df = (
    china_economic_issuer_universe_df
    .merge(
        cninfo_issuer_coverage_df,
        on="issuer_id",
        how="left",
        validate="1:1",
    )
    .merge(
        preferred_upstream_issuer_source_df[
            [
                column
                for column in [
                    "issuer_id",
                    "upstream_source_system",
                    "upstream_source_table",
                    "upstream_fact_rows",
                    "upstream_unique_concepts",
                    "precedence_rank",
                ]
                if column
                in preferred_upstream_issuer_source_df.columns
            ]
        ],
        on="issuer_id",
        how="left",
        validate="1:1",
    )
)

for column in [
    "cninfo_fact_rows",
    "cninfo_unique_concepts",
    "cninfo_filing_count",
    "upstream_fact_rows",
    "upstream_unique_concepts",
]:
    china_issuer_enrichment_df[column] = pd.to_numeric(
        china_issuer_enrichment_df.get(
            column,
            pd.Series(0, index=china_issuer_enrichment_df.index),
        ),
        errors="coerce",
    ).fillna(0)

fallback_primary_source = pd.Series(
    np.where(
        china_issuer_enrichment_df["cninfo_fact_rows"].gt(0),
        "CNINFO",
        "UNRESOLVED",
    ),
    index=china_issuer_enrichment_df.index,
    dtype="string",
)

china_issuer_enrichment_df["primary_consolidated_source"] = (
    china_issuer_enrichment_df[
        "upstream_source_system"
    ]
    .astype("string")
    .fillna(fallback_primary_source)
)

if china_issuer_enrichment_df[
    "primary_consolidated_source"
].isna().any():
    raise RuntimeError(
        "Source-precedence assignment left missing primary source values."
    )

china_issuer_enrichment_df["cninfo_ingestion_role"] = np.select(
    [
        china_issuer_enrichment_df[
            "primary_consolidated_source"
        ].eq("CNINFO"),
        china_issuer_enrichment_df["cninfo_fact_rows"].gt(0),
    ],
    [
        "PRIMARY_WHERE_NO_STRUCTURED_UPSTREAM",
        "SUPPLEMENTARY_ISSUER_ENRICHMENT",
    ],
    default="NO_CONFIRMED_CNINFO_ENRICHMENT",
)

china_fundamental_source_precedence_df = (
    china_issuer_enrichment_df.copy()
)

if china_fundamentals_standardised_df.empty:
    china_cninfo_facts_classified_df = pd.DataFrame()
    china_incremental_facts_df = pd.DataFrame()
    china_duplicate_primary_facts_df = pd.DataFrame()
    china_local_metrics_df = pd.DataFrame()
else:
    china_cninfo_facts_classified_df = (
        china_fundamentals_standardised_df
        .merge(
            china_primary_source_concept_inventory_df,
            on=[
                "issuer_id",
                "standard_concept",
            ],
            how="left",
            validate="m:m",
        )
        .merge(
            china_cninfo_entity_bridge_confirmed_df[
                [
                    "issuer_id",
                    "relationship_id",
                    "cninfo_entity_id",
                    "cninfo_stock_code",
                    "cninfo_entity_relationship",
                    "cninfo_entity_role",
                    "entity_mapping_method",
                    "entity_mapping_confidence",
                ]
            ].drop_duplicates(),
            on="issuer_id",
            how="left",
            validate="m:m",
        )
    )

    china_cninfo_facts_classified_df[
        "concept_exists_in_primary_source"
    ] = china_cninfo_facts_classified_df[
        "primary_source_system"
    ].notna()

    source_label_column = first_existing_column(
        china_cninfo_facts_classified_df,
        [
            "account_label",
            "source_account_label",
            "reported_label",
        ],
    )

    if source_label_column is None:
        local_label_mask = pd.Series(
            False,
            index=china_cninfo_facts_classified_df.index,
        )
    else:
        local_label_mask = (
            china_cninfo_facts_classified_df[
                source_label_column
            ]
            .astype("string")
            .str.contains(
                CHINA_LOCAL_DISCLOSURE_LABEL_PATTERN,
                na=False,
            )
        )

    local_concept_mask = (
        china_cninfo_facts_classified_df[
            "standard_concept"
        ].isin(CHINA_LOCAL_ONLY_STANDARD_CONCEPTS)
    )

    different_entity_mask = (
        ~china_cninfo_facts_classified_df[
            "cninfo_entity_relationship"
        ]
        .astype("string")
        .str.startswith(
            "SAME_ISSUER",
            na=False,
        )
    )

    no_primary_mask = (
        ~china_cninfo_facts_classified_df[
            "concept_exists_in_primary_source"
        ]
    )

    china_cninfo_facts_classified_df[
        "fact_origin"
    ] = np.select(
        [
            local_concept_mask | local_label_mask,
            no_primary_mask,
            different_entity_mask,
        ],
        [
            "LOCAL_ONLY",
            "INCREMENTAL",
            "SUPPLEMENTARY",
        ],
        default="DUPLICATE_PRIMARY_CONCEPT",
    )

    china_cninfo_facts_classified_df[
        "supplementary_fact_reason"
    ] = np.select(
        [
            local_concept_mask,
            local_label_mask,
            no_primary_mask,
            different_entity_mask,
        ],
        [
            "LOCAL_CANONICAL_CONCEPT",
            "CHINA_SPECIFIC_SOURCE_LABEL",
            "CONCEPT_ABSENT_FROM_PRIMARY_SOURCE",
            "DISTINCT_CNINFO_ENTITY_OR_CONSOLIDATION_SCOPE",
        ],
        default="CONCEPT_ALREADY_AVAILABLE_FROM_PRIMARY_SOURCE",
    )

    china_cninfo_facts_classified_df[
        "is_primary_consolidated_source"
    ] = (
        china_cninfo_facts_classified_df[
            "primary_source_system"
        ].isna()
    )

    china_cninfo_facts_classified_df[
        "is_incremental_to_primary_source"
    ] = (
        china_cninfo_facts_classified_df[
            "fact_origin"
        ].isin(
            {
                "LOCAL_ONLY",
                "INCREMENTAL",
                "SUPPLEMENTARY",
            }
        )
    )


    def classify_fact_category(row):
        concept = row.get("standard_concept")

        for category, concepts in CHINA_FACT_CATEGORY_RULES.items():
            if concept in concepts:
                return category

        role = row.get("cninfo_entity_role")

        if role == "BATTERY_ENTITY":
            return "BATTERY"

        if role == "COMPONENT_ENTITY":
            return "COMPONENTS"

        if role == "COMMERCIAL_VEHICLE_ENTITY":
            return "COMMERCIAL_VEHICLES"

        if role == "FINANCE_ENTITY":
            return "FINANCE"

        if role == "TECHNOLOGY_ENTITY":
            return "TECHNOLOGY"

        if role == "MANUFACTURING_ENTITY":
            return "MANUFACTURING"

        return "GENERAL_FINANCIAL_OR_DISCLOSURE"

    china_cninfo_facts_classified_df[
        "enrichment_category"
    ] = china_cninfo_facts_classified_df.apply(
        classify_fact_category,
        axis=1,
    )

    china_incremental_facts_df = (
        china_cninfo_facts_classified_df[
            china_cninfo_facts_classified_df[
                "is_incremental_to_primary_source"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

    china_duplicate_primary_facts_df = (
        china_cninfo_facts_classified_df[
            china_cninfo_facts_classified_df[
                "fact_origin"
            ].eq("DUPLICATE_PRIMARY_CONCEPT")
        ]
        .copy()
        .reset_index(drop=True)
    )

    china_local_metrics_df = (
        china_cninfo_facts_classified_df[
            china_cninfo_facts_classified_df[
                "fact_origin"
            ].eq("LOCAL_ONLY")
        ]
        .copy()
        .reset_index(drop=True)
    )

china_disclosure_inventory_df = (
    china_filing_metadata_df.copy()
)

china_operating_statistics_df = (
    china_local_metrics_df[
        china_local_metrics_df[
            "standard_concept"
        ].isin(
            {
                "vehicle_sales_volume",
                "vehicle_production_volume",
            }
        )
    ]
    .copy()
    .reset_index(drop=True)
    if not china_local_metrics_df.empty
    else pd.DataFrame()
)

china_government_subsidies_df = (
    china_local_metrics_df[
        china_local_metrics_df[
            "standard_concept"
        ].isin(
            {
                "government_grants",
                "government_subsidies",
            }
        )
    ]
    .copy()
    .reset_index(drop=True)
    if not china_local_metrics_df.empty
    else pd.DataFrame()
)

china_capex_projects_df = (
    china_local_metrics_df[
        china_local_metrics_df[
            "standard_concept"
        ].isin(
            {
                "capital_expenditure",
                "construction_in_progress",
            }
        )
    ]
    .copy()
    .reset_index(drop=True)
    if not china_local_metrics_df.empty
    else pd.DataFrame()
)

china_related_party_transactions_df = (
    china_local_metrics_df[
        china_local_metrics_df[
            "standard_concept"
        ].astype("string")
        .str.contains(
            "related_party",
            na=False,
        )
    ]
    .copy()
    .reset_index(drop=True)
    if not china_local_metrics_df.empty
    else pd.DataFrame()
)


print(
    "Accepted mapped CNINFO facts:",
    len(china_fundamentals_mapped_df),
)
print(
    "Selected CNINFO standardised facts:",
    len(china_fundamentals_standardised_df),
)
print(
    "Incremental or supplementary CNINFO facts:",
    len(china_incremental_facts_df),
)
print(
    "Duplicate primary-source concepts retained for audit:",
    len(china_duplicate_primary_facts_df),
)

display(china_issuer_enrichment_df)

Mapping Chinese account labels:   0%|          | 0/43425 [00:00<?, ?it/s]

/tmp/ipykernel_1172/3061691470.py:661: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(
/tmp/ipykernel_1172/3061691470.py:1097: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


Accepted mapped CNINFO facts: 44459
Selected CNINFO standardised facts: 6985
Incremental or supplementary CNINFO facts: 8766
Duplicate primary-source concepts retained for audit: 423


,issuer_id,issuer_membership_security_count,issuer_membership_security_ids,issuer_etf_count,issuer_etf_tickers,issuer_first_membership_date,issuer_last_membership_date,issuer_membership_row_count,issuer_name,issuer_country,listed_security_count,listed_security_ids,listing_exchanges,listing_countries,primary_listing_sources,china_economic_issuer_evidence,security_count,security_ids,etf_count,etf_tickers,first_membership_date,last_membership_date,cninfo_fact_rows,cninfo_unique_concepts,cninfo_filing_count,upstream_source_system,upstream_source_table,upstream_fact_rows,upstream_unique_concepts,precedence_rank,primary_consolidated_source,cninfo_ingestion_role
0,GAI_008BF1916FA3C09BB36B,1,GAS_FB0365ACB408D3970591,1,KARS,2022-05-27,2024-05-30,9,Guangzhou Great Power Energy and Technology Co...,CN,1.0,GAS_FB0365ACB408D3970591,<NA>,<NA>,UNRESOLVED,ISSUER_COUNTRY_CN,1,GAS_FB0365ACB408D3970591,1,KARS,2022-05-27,2024-05-30,0.0,0.0,0.0,NaN,NaN,0.0,0.0,NaN,UNRESOLVED,NO_CONFIRMED_CNINFO_ENRICHMENT
1,GAI_01CAE5E9EA608E2482D2,1,GAS_B0B0614B82FE2565480D,1,KARS,2023-11-29,2026-05-29,10,Hunan Yuneng New Energy Battery Material Co Ltd,CN,1.0,GAS_B0B0614B82FE2565480D,<NA>,<NA>,UNRESOLVED,ISSUER_COUNTRY_CN,1,GAS_B0B0614B82FE2565480D,1,KARS,2023-11-29,2026-05-29,0.0,0.0,0.0,NaN,NaN,0.0,0.0,NaN,UNRESOLVED,NO_CONFIRMED_CNINFO_ENRICHMENT
2,GAI_0787D230076902C0E857,1,GAS_9013A32826453D14D079,1,KARS,2021-08-30,2026-05-29,20,"ZHEJIANG HUAYOU COBALT CO., LTD",CN,1.0,GAS_9013A32826453D14D079,<NA>,<NA>,UNRESOLVED,ISSUER_COUNTRY_CN,1,GAS_9013A32826453D14D079,1,KARS,2021-08-30,2026-05-29,0.0,0.0,0.0,NaN,NaN,0.0,0.0,NaN,UNRESOLVED,NO_CONFIRMED_CNINFO_ENRICHMENT
3,GAI_098ABF62DAB5DA55A65D,1,GAS_D3A4BEB9E49457BA8C37,1,KARS,2026-02-27,2026-05-29,2,"NavInfo Co., Ltd.",CN,1.0,GAS_D3A4BEB9E49457BA8C37,<NA>,<NA>,UNRESOLVED,ISSUER_COUNTRY_CN,1,GAS_D3A4BEB9E49457BA8C37,1,KARS,2026-02-27,2026-05-29,0.0,0.0,0.0,NaN,NaN,0.0,0.0,NaN,UNRESOLVED,NO_CONFIRMED_CNINFO_ENRICHMENT
4,GAI_09DA7F36C135795A0916,1,GAS_B2C051C672FC07E5EA27,2,CARZ | KARS,2019-11-19,2026-05-29,37,"Guangzhou Automobile Group Co., Ltd",HK,1.0,GAS_B2C051C672FC07E5EA27,HKEX,HK,HKEX,CHINESE_OFFSHORE_OR_HK_LISTING,1,GAS_B2C051C672FC07E5EA27,2,CARZ | KARS,2019-11-19,2026-05-29,1731.0,74.0,50.0,HKEX,hong_kong_fundamentals_standardised_df,19.0,15.0,2.0,HKEX,SUPPLEMENTARY_ISSUER_ENRICHMENT
5,GAI_0BCA9270FF1670736E5A,1,GAS_67453941547E8CFCA7F1,1,KARS,2026-05-29,NaT,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,GAS_67453941547E8CFCA7F1,1,KARS,2026-05-29,NaT,0.0,0.0,0.0,NaN,NaN,0.0,0.0,NaN,UNRESOLVED,NO_CONFIRMED_CNINFO_ENRICHMENT
6,GAI_10DE387103D01591CDFE,1,GAS_CE5A3C713E42281904FD,1,CARZ,2025-05-22,2025-08-25,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,GAS_CE5A3C713E42281904FD,1,CARZ,2025-05-22,2025-08-25,0.0,0.0,0.0,NaN,NaN,0.0,0.0,NaN,UNRESOLVED,NO_CONFIRMED_CNINFO_ENRICHMENT
7,GAI_118982B3D7A3654914FD,2,GAS_0F0802D8983EE892AD09 | GAS_376976AF42716CB...,4,CARZ | DRIV | IDRV | KARS,2020-04-28,2026-06-25,84,Ganfeng Lithium Group Co Ltd,CN,2.0,GAS_0F0802D8983EE892AD09 | GAS_376976AF42716CB...,HKEX,HK,HKEX | UNRESOLVED,ISSUER_COUNTRY_CN,2,GAS_0F0802D8983EE892AD09 | GAS_376976AF42716CB...,4,CARZ | DRIV | IDRV | KARS,2020-04-28,2026-06-25,0.0,0.0,0.0,HKEX,hong_kong_fundamentals_standardised_df,4.0,4.0,2.0,HKEX,NO_CONFIRMED_CNINFO_ENRICHMENT
8,GAI_11D73C9A6BB9EA6924F2,1,GAS_3BDDC08D09DEB3CF277D,1,KARS,2021-08-30,2026-05-29,20,"GEM Co., Ltd.",CN,1.0,GAS_3BDDC08D09DEB3CF277D,<NA>,<NA>,UNRESOLVED,ISSUER_COUNTRY_CN,1,GAS_3BDDC08D09DEB3CF277D,1,KARS,2021-08-30,2026-05-29,0.0,0.0,0.0,NaN,NaN,0.0,0.0,NaN,UNRESOLVED,NO_CONFIRMED_CNINFO_ENRICHMENT
9,GAI_1230A0A328AB78758D3A,1,GAS_281A00DF016838ECFF9C,2,CARZ | KARS,2026-02-25,2026-05-29,4,WeRide Inc,CN,1.0,GAS_281A00DF016838ECFF9C,<NA>,US,SEC,ISSUER_COUNTRY_CN,1,GAS_281A00DF016838ECFF9C,2,CARZ | KARS,2026-02-25,2026-05-29,0.0,0.0,0.0,SEC,sec_fundamentals_security_linked_df,139.0,29.0,1.0,SEC,NO_CONFIRMED_CNINFO_ENRICHMENT


In [ ]:
# 26. POINT-IN-TIME ACCESS HELPERS

def china_fundamentals_as_of(
    dataframe,
    as_of_date,
    issuer_ids=None,
    security_ids=None,
    standard_concepts=None,
):
    if dataframe.empty:
        return dataframe.copy()

    cutoff = pd.Timestamp(as_of_date)
    cutoff = (
        cutoff.tz_localize("UTC")
        if cutoff.tzinfo is None
        else cutoff.tz_convert("UTC")
    )

    result = dataframe[
        pd.to_datetime(
            dataframe["available_datetime"],
            errors="coerce",
            utc=True,
        ) <= cutoff
    ].copy()

    if issuer_ids is not None:
        result = result[
            result["issuer_id"].isin(set(issuer_ids))
        ]

    if security_ids is not None:
        result = result[
            result["security_id"].isin(set(security_ids))
        ]

    if standard_concepts is not None:
        result = result[
            result["standard_concept"].isin(set(standard_concepts))
        ]

    return result


def latest_china_fact_as_of(dataframe, as_of_date):
    result = china_fundamentals_as_of(
        dataframe,
        as_of_date,
    )

    if result.empty:
        return result

    return (
        result
        .sort_values(
            ["available_datetime", "document_priority", "document_id"]
        )
        .drop_duplicates(
            ["security_id", "issuer_id", "standard_concept"],
            keep="last",
        )
        .reset_index(drop=True)
    )

In [ ]:
# 27. COVERAGE, AVAILABILITY AND QUALITY REPORTS

STANDARD_CONCEPT_COVERAGE_COLUMNS = [
    "standard_concept",
    "statement_type",
    "core_tier",
    "fact_rows",
    "issuer_count",
    "security_count",
    "document_count",
    "earliest_available",
    "latest_available",
    "numeric_fact_share",
    "unit_match_share",
    "deterministic_share",
    "average_mapping_score",
]

OBSERVED_AVAILABILITY_COLUMNS = [
    "standard_concept",
    "issuer_coverage",
    "security_coverage",
    "filing_coverage",
    "fact_rows",
    "first_available_datetime",
    "last_available_datetime",
    "deterministic_fact_share",
    "average_mapping_score",
]

china_filing_coverage_report_df = pd.DataFrame({
    "metric": [
        "china_securities",
        "china_issuers",
        "resolved_cninfo_securities",
        "disclosures_discovered",
        "documents_with_text",
        "raw_account_candidates",
        "unique_cleaned_account_labels",
        "accepted_mapped_fact_rows",
        "standardised_fact_rows",
        "security_link_rate",
    ],
    "value": [
        len(china_security_universe_df),
        china_issuer_universe_df[
            "issuer_id"
        ].nunique(),
        len(china_resolved_securities_df),
        len(china_filing_metadata_df),
        len(cninfo_document_text_df),
        len(china_account_candidates_raw_df),
        (
            china_account_candidates_raw_df[
                "account_label"
            ].nunique()
            if not china_account_candidates_raw_df.empty
            else 0
        ),
        len(china_fundamentals_mapped_df),
        len(china_fundamentals_standardised_df),
        (
            china_fundamentals_standardised_df[
                "security_id"
            ].notna().mean()
            if (
                not china_fundamentals_standardised_df.empty
                and "security_id"
                in china_fundamentals_standardised_df.columns
            )
            else np.nan
        ),
    ],
})

if not china_fundamentals_standardised_df.empty:
    china_standard_concept_coverage_df = (
        china_fundamentals_standardised_df
        .groupby(
            [
                "standard_concept",
                "statement_type",
                "core_tier",
            ],
            dropna=False,
        )
        .agg(
            fact_rows=("reported_value", "size"),
            issuer_count=("issuer_id", "nunique"),
            security_count=("security_id", "nunique"),
            document_count=("document_id", "nunique"),
            earliest_available=("available_datetime", "min"),
            latest_available=("available_datetime", "max"),
            numeric_fact_share=(
                "reported_value",
                lambda s: s.notna().mean(),
            ),
            unit_match_share=(
                "unit_family_match",
                "mean",
            ),
            deterministic_share=(
                "mapping_method",
                lambda s: s.eq(
                    "DETERMINISTIC_REGEX"
                ).mean(),
            ),
            average_mapping_score=(
                "mapping_score",
                "mean",
            ),
        )
        .reset_index()
    )

    observed = (
        china_fundamentals_standardised_df
        .groupby("standard_concept")
        .agg(
            issuer_coverage=("issuer_id", "nunique"),
            security_coverage=("security_id", "nunique"),
            filing_coverage=("document_id", "nunique"),
            fact_rows=("reported_value", "size"),
            first_available_datetime=(
                "available_datetime",
                "min",
            ),
            last_available_datetime=(
                "available_datetime",
                "max",
            ),
            deterministic_fact_share=(
                "mapping_method",
                lambda s: s.eq(
                    "DETERMINISTIC_REGEX"
                ).mean(),
            ),
            average_mapping_score=(
                "mapping_score",
                "mean",
            ),
        )
        .reset_index()
    )

else:
    china_standard_concept_coverage_df = (
        pd.DataFrame(
            columns=STANDARD_CONCEPT_COVERAGE_COLUMNS
        )
    )

    observed = pd.DataFrame(
        columns=OBSERVED_AVAILABILITY_COLUMNS
    )

for column in STANDARD_CONCEPT_COVERAGE_COLUMNS:
    if column not in china_standard_concept_coverage_df.columns:
        china_standard_concept_coverage_df[
            column
        ] = pd.NA

china_standard_concept_coverage_df = (
    china_standard_concept_coverage_df[
        STANDARD_CONCEPT_COVERAGE_COLUMNS
    ]
    .copy()
)

for column in OBSERVED_AVAILABILITY_COLUMNS:
    if column not in observed.columns:
        observed[column] = pd.NA

observed = observed[
    OBSERVED_AVAILABILITY_COLUMNS
].copy()

china_standard_concept_availability_df = (
    global_canonical_schema_df.merge(
        observed,
        on="standard_concept",
        how="left",
        validate="1:1",
    )
)

availability_defaults = {
    "issuer_coverage": 0,
    "security_coverage": 0,
    "filing_coverage": 0,
    "fact_rows": 0,
    "first_available_datetime": pd.NaT,
    "last_available_datetime": pd.NaT,
    "deterministic_fact_share": np.nan,
    "average_mapping_score": np.nan,
}

for column, default_value in availability_defaults.items():
    if column not in china_standard_concept_availability_df.columns:
        china_standard_concept_availability_df[
            column
        ] = default_value

for column in [
    "issuer_coverage",
    "security_coverage",
    "filing_coverage",
    "fact_rows",
]:
    china_standard_concept_availability_df[
        column
    ] = (
        pd.to_numeric(
            china_standard_concept_availability_df[
                column
            ],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )

for column in [
    "first_available_datetime",
    "last_available_datetime",
]:
    china_standard_concept_availability_df[
        column
    ] = pd.to_datetime(
        china_standard_concept_availability_df[
            column
        ],
        errors="coerce",
        utc=True,
    )

total_issuers = max(
    china_issuer_universe_df[
        "issuer_id"
    ].dropna().nunique(),
    1,
)

china_standard_concept_availability_df[
    "issuer_coverage_rate"
] = (
    china_standard_concept_availability_df[
        "issuer_coverage"
    ]
    / total_issuers
)

china_mapping_method_report_df = (
    china_fundamentals_standardised_df[
        "mapping_method"
    ]
    .value_counts(dropna=False)
    .rename_axis("mapping_method")
    .reset_index(name="fact_rows")
    if (
        not china_fundamentals_standardised_df.empty
        and "mapping_method"
        in china_fundamentals_standardised_df.columns
    )
    else pd.DataFrame(
        columns=[
            "mapping_method",
            "fact_rows",
        ]
    )
)

china_fuzzy_label_quality_report_df = (
    china_account_candidates_raw_df[
        [
            "account_label",
            "fuzzy_label_rejection_reason",
        ]
    ]
    .drop_duplicates("account_label")[
        "fuzzy_label_rejection_reason"
    ]
    .value_counts(dropna=False)
    .rename_axis("label_quality_class")
    .reset_index(name="unique_label_count")
    if (
        not china_account_candidates_raw_df.empty
        and {
            "account_label",
            "fuzzy_label_rejection_reason",
        }.issubset(
            china_account_candidates_raw_df.columns
        )
    )
    else pd.DataFrame(
        columns=[
            "label_quality_class",
            "unique_label_count",
        ]
    )
)

china_mapping_quality_df = pd.DataFrame({
    "metric": [
        "canonical_standard_concepts",
        "supported_source_mapping_rows",
        "unique_cleaned_account_labels",
        "observed_mapping_rows",
        "accepted_mapped_fact_rows",
        "selected_standardised_fact_rows",
        "unique_standard_concepts_observed",
        "deterministic_selected_fact_share",
        "average_selected_mapping_score",
        "mapping_review_queue_rows",
        "unmapped_accounts_in_inventory",
        "automotive_extension_candidates",
    ],
    "value": [
        global_canonical_schema_df[
            "standard_concept"
        ].nunique(),
        len(china_source_account_mapping_df),
        (
            china_account_candidates_raw_df[
                "account_label"
            ].nunique()
            if (
                not china_account_candidates_raw_df.empty
                and "account_label"
                in china_account_candidates_raw_df.columns
            )
            else 0
        ),
        len(china_observed_account_mapping_df),
        len(china_fundamentals_mapped_df),
        len(china_fundamentals_standardised_df),
        (
            china_fundamentals_standardised_df[
                "standard_concept"
            ].nunique()
            if (
                not china_fundamentals_standardised_df.empty
                and "standard_concept"
                in china_fundamentals_standardised_df.columns
            )
            else 0
        ),
        (
            china_fundamentals_standardised_df[
                "mapping_method"
            ]
            .eq("DETERMINISTIC_REGEX")
            .mean()
            if (
                not china_fundamentals_standardised_df.empty
                and "mapping_method"
                in china_fundamentals_standardised_df.columns
            )
            else np.nan
        ),
        (
            pd.to_numeric(
                china_fundamentals_standardised_df[
                    "mapping_score"
                ],
                errors="coerce",
            ).mean()
            if (
                not china_fundamentals_standardised_df.empty
                and "mapping_score"
                in china_fundamentals_standardised_df.columns
            )
            else np.nan
        ),
        len(china_mapping_review_queue_df),
        len(china_unmapped_account_inventory_df),
        len(china_automotive_extension_candidates_df),
    ],
})

print(
    "Concepts with observed Chinese facts:",
    int(
        china_standard_concept_availability_df[
            "fact_rows"
        ].gt(0).sum()
    ),
)

if china_fundamentals_standardised_df.empty:
    print(
        "No standardised Chinese facts are currently available. "
        "The QA and persistence tables remain schema-stable."
    )

display(china_filing_coverage_report_df)
display(china_mapping_quality_df)
display(china_mapping_method_report_df)
display(china_fuzzy_label_quality_report_df)
display(
    china_standard_concept_availability_df
    .sort_values(
        [
            "issuer_coverage",
            "fact_rows",
        ],
        ascending=[False, False],
    )
    .head(100)
)

china_enrichment_quality_df = pd.DataFrame({
    "metric": [
        "china_economic_issuers",
        "issuers_with_confirmed_cninfo_entity",
        "cninfo_disclosures",
        "cninfo_standardised_facts",
        "incremental_or_supplementary_facts",
        "local_only_facts",
        "duplicate_primary_concept_facts",
        "issuers_with_sec_primary_source",
        "issuers_with_hkex_primary_source",
        "issuers_with_cninfo_primary_source",
    ],
    "value": [
        china_economic_issuer_universe_df["issuer_id"].nunique(),
        china_cninfo_entity_bridge_confirmed_df["issuer_id"].nunique(),
        len(china_disclosure_inventory_df),
        len(china_fundamentals_standardised_df),
        len(china_incremental_facts_df),
        len(china_local_metrics_df),
        len(china_duplicate_primary_facts_df),
        china_issuer_enrichment_df[
            "primary_consolidated_source"
        ].eq("SEC").sum(),
        china_issuer_enrichment_df[
            "primary_consolidated_source"
        ].eq("HKEX").sum(),
        china_issuer_enrichment_df[
            "primary_consolidated_source"
        ].eq("CNINFO").sum(),
    ],
})

china_fact_origin_report_df = (
    china_cninfo_facts_classified_df[
        "fact_origin"
    ]
    .value_counts(dropna=False)
    .rename_axis("fact_origin")
    .reset_index(name="fact_rows")
    if not china_cninfo_facts_classified_df.empty
    else pd.DataFrame(
        columns=[
            "fact_origin",
            "fact_rows",
        ]
    )
)

display(china_enrichment_quality_df)
display(china_fact_origin_report_df)

china_relationship_quality_df = pd.DataFrame({
    "metric": [
        "economic_issuers",
        "confirmed_cninfo_entities",
        "confirmed_relationships",
        "issuers_with_multiple_cninfo_entities",
        "same_issuer_relationships",
        "parent_relationships",
        "subsidiary_relationships",
        "operating_company_relationships",
        "invalid_relationship_rows",
    ],
    "value": [
        china_economic_issuer_universe_df["issuer_id"].nunique(),
        china_cninfo_entities_df["cninfo_entity_id"].nunique(),
        china_entity_relationship_graph_df["relationship_id"].nunique(),
        (
            china_entity_relationship_graph_df
            .groupby("issuer_id")["cninfo_entity_id"]
            .nunique()
            .gt(1)
            .sum()
        ),
        china_entity_relationship_graph_df[
            "cninfo_entity_relationship"
        ].astype("string").str.startswith(
            "SAME_ISSUER",
            na=False,
        ).sum(),
        china_entity_relationship_graph_df[
            "cninfo_entity_relationship"
        ].eq("MAINLAND_PARENT").sum(),
        china_entity_relationship_graph_df[
            "cninfo_entity_relationship"
        ].isin(
            {
                "MAINLAND_SUBSIDIARY",
                "LISTED_SUBSIDIARY",
                "SPINOFF",
            }
        ).sum(),
        china_entity_relationship_graph_df[
            "cninfo_entity_relationship"
        ].eq("OPERATING_COMPANY").sum(),
        china_entity_relationship_graph_df[
            "relationship_validation_status"
        ].ne("VALID").sum(),
    ],
})

display(china_relationship_quality_df)

Concepts with observed Chinese facts: 84


,metric,value
0,china_securities,5.0
1,china_issuers,4.0
2,resolved_cninfo_securities,5.0
3,disclosures_discovered,210.0
4,documents_with_text,210.0
5,raw_account_candidates,507757.0
6,unique_cleaned_account_labels,43425.0
7,accepted_mapped_fact_rows,44459.0
8,standardised_fact_rows,6985.0
9,security_link_rate,1.0


,metric,value
0,canonical_standard_concepts,100.000000
1,supported_source_mapping_rows,96.000000
2,unique_cleaned_account_labels,43425.000000
3,observed_mapping_rows,208382.000000
4,accepted_mapped_fact_rows,44459.000000
5,selected_standardised_fact_rows,6985.000000
6,unique_standard_concepts_observed,84.000000
7,deterministic_selected_fact_share,0.940587
8,average_selected_mapping_score,99.041429
9,mapping_review_queue_rows,2969.000000


,mapping_method,fact_rows
0,DETERMINISTIC_REGEX,6570
1,BLOCK9_ACCEPTED_SYNONYM,256
2,FUZZY_ANCHOR,159


,label_quality_class,unique_label_count
0,ELIGIBLE,41932
1,UNAPPROVED_SINGLE_TOKEN,1090
2,LATIN_LABEL_TOO_SHORT,389
3,GENERIC_STOPWORD,14


,standard_concept,statement_type,expected_period_type,expected_unit_family,core_tier,is_core,aggregation_policy,issuer_coverage,security_coverage,filing_coverage,fact_rows,first_available_datetime,last_available_datetime,deterministic_fact_share,average_mapping_score,issuer_coverage_rate
77,operating_cash_flow,CASH_FLOW,DURATION,MONETARY,1,True,PERIOD_VALUE,4,4,172,172,2019-03-27 16:00:00+00:00,2026-04-24 16:00:00+00:00,0.889535,99.110465,1.00
0,revenue,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,4,4,166,166,2019-03-22 16:00:00+00:00,2026-04-29 16:00:00+00:00,1.000000,99.000000,1.00
31,total_assets,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,4,4,165,165,2019-03-22 16:00:00+00:00,2026-04-29 16:00:00+00:00,0.993939,99.006061,1.00
7,net_income_attributable_to_owners,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,4,4,138,138,2019-03-27 16:00:00+00:00,2026-04-24 16:00:00+00:00,0.557971,98.974521,1.00
6,net_income,INCOME_STATEMENT,DURATION,MONETARY,1,True,PERIOD_VALUE,4,4,133,133,2019-03-22 16:00:00+00:00,2026-04-24 16:00:00+00:00,1.000000,99.000000,1.00
15,general_and_administrative_expense,INCOME_STATEMENT,DURATION,MONETARY,2,True,PERIOD_VALUE,4,4,131,131,2019-03-27 16:00:00+00:00,2026-04-29 16:00:00+00:00,0.954198,99.045802,1.00
13,research_and_development_expense,INCOME_STATEMENT,DURATION,MONETARY,2,True,PERIOD_VALUE,4,4,129,129,2019-03-27 16:00:00+00:00,2026-04-29 16:00:00+00:00,0.937984,99.062016,1.00
28,other_comprehensive_income,COMPREHENSIVE_INCOME,DURATION,MONETARY,2,True,PERIOD_VALUE,4,4,127,127,2019-03-27 16:00:00+00:00,2026-04-29 16:00:00+00:00,0.984252,99.015748,1.00
14,selling_expense,INCOME_STATEMENT,DURATION,MONETARY,2,True,PERIOD_VALUE,4,4,126,126,2019-03-27 16:00:00+00:00,2026-04-29 16:00:00+00:00,1.000000,99.000000,1.00
52,total_liabilities,BALANCE_SHEET,INSTANT,MONETARY,1,True,LATEST_INSTANT,4,4,124,124,2019-03-27 16:00:00+00:00,2026-04-29 16:00:00+00:00,1.000000,99.000000,1.00


,metric,value
0,china_economic_issuers,103
1,issuers_with_confirmed_cninfo_entity,4
2,cninfo_disclosures,210
3,cninfo_standardised_facts,6985
4,incremental_or_supplementary_facts,8766
5,local_only_facts,17
6,duplicate_primary_concept_facts,423
7,issuers_with_sec_primary_source,5
8,issuers_with_hkex_primary_source,14
9,issuers_with_cninfo_primary_source,2


,fact_origin,fact_rows
0,INCREMENTAL,8749
1,DUPLICATE_PRIMARY_CONCEPT,423
2,LOCAL_ONLY,17


,metric,value
0,economic_issuers,103
1,confirmed_cninfo_entities,5
2,confirmed_relationships,5
3,issuers_with_multiple_cninfo_entities,1
4,same_issuer_relationships,5
5,parent_relationships,0
6,subsidiary_relationships,0
7,operating_company_relationships,0
8,invalid_relationship_rows,0


In [ ]:
# 28. CHINA IN-MEMORY OUTPUT CONTRACT

china_block_data = {
    # Cross-listing Chinese issuer universe
    "china_issuer_security_universe_df": china_issuer_security_universe_df,
    "china_economic_issuer_universe_df": china_economic_issuer_universe_df,
    "china_issuer_etf_membership_intervals_df": china_issuer_etf_membership_intervals_df,
    "china_issuer_universe_quality_df": china_issuer_universe_quality_df,

    # CNINFO entity resolution
    "cninfo_stock_metadata_df": cninfo_stock_metadata_df,
    "cninfo_stock_metadata_log_df": cninfo_stock_metadata_log_df,
    "china_relationship_catalogue_df": china_relationship_catalogue_df,
    "china_cninfo_entity_override_df": china_cninfo_entity_override_df,
    "china_cninfo_external_override_df": china_cninfo_external_override_df,
    "china_cninfo_entity_seed_df": china_cninfo_entity_seed_df,
    "china_cninfo_entity_bridge_df": china_cninfo_entity_bridge_df,
    "china_cninfo_entity_bridge_confirmed_df": china_cninfo_entity_bridge_confirmed_df,
    "china_cninfo_entity_bridge_unresolved_df": china_cninfo_entity_bridge_unresolved_df,
    "china_cninfo_entity_resolution_report_df": china_cninfo_entity_resolution_report_df,
    "china_entity_relationship_graph_df": china_entity_relationship_graph_df,
    "china_cninfo_entities_df": china_cninfo_entities_df,
    "china_relationship_quality_df": china_relationship_quality_df,

    # CNINFO filings and documents
    "cninfo_announcements_discovered_df": cninfo_announcements_discovered_df,
    "cninfo_announcement_search_log_df": cninfo_announcement_search_log_df,
    "china_disclosure_inventory_df": china_disclosure_inventory_df,
    "china_filing_metadata_df": china_filing_metadata_df,
    "cninfo_document_download_log_df": cninfo_document_download_log_df,
    "cninfo_document_text_df": cninfo_document_text_df,

    # Canonical accounting extraction and mapping
    "china_source_account_mapping_proposed_df": china_source_account_mapping_proposed_df,
    "china_unsupported_concept_mappings_df": china_unsupported_concept_mappings_df,
    "china_source_account_mapping_df": china_source_account_mapping_df,
    "china_standard_concept_dictionary_df": china_standard_concept_dictionary_df,
    "china_canonical_anchor_df": china_canonical_anchor_df,
    "china_account_candidates_raw_df": china_account_candidates_raw_df,
    "china_observed_account_mapping_df": china_observed_account_mapping_df,
    "china_block9_registry_matches_df": china_block9_registry_matches_df,
    "china_fundamentals_mapped_df": china_fundamentals_mapped_df,
    "china_mapping_alternatives_df": china_mapping_alternatives_df,
    "china_mapping_review_queue_df": china_mapping_review_queue_df,
    "china_low_quality_fuzzy_rejections_df": china_low_quality_fuzzy_rejections_df,
    "china_fundamentals_standardised_df": china_fundamentals_standardised_df,
    "china_unmapped_account_inventory_df": china_unmapped_account_inventory_df,
    "china_automotive_extension_candidates_df": china_automotive_extension_candidates_df,

    # Primary-source comparison and issuer fundamentals
    "upstream_structured_issuer_coverage_df": upstream_structured_issuer_coverage_df,
    "preferred_upstream_issuer_source_df": preferred_upstream_issuer_source_df,
    "china_primary_source_facts_df": china_primary_source_facts_df,
    "china_primary_source_concept_inventory_df": china_primary_source_concept_inventory_df,
    "cninfo_issuer_coverage_df": cninfo_issuer_coverage_df,
    "china_issuer_enrichment_df": china_issuer_enrichment_df,
    "china_fundamental_source_precedence_df": china_fundamental_source_precedence_df,
    "china_cninfo_facts_classified_df": china_cninfo_facts_classified_df,
    "china_incremental_facts_df": china_incremental_facts_df,
    "china_duplicate_primary_facts_df": china_duplicate_primary_facts_df,
    "china_local_metrics_df": china_local_metrics_df,

    # Accounting-focused supplementary views
    "china_government_subsidies_df": china_government_subsidies_df,
    "china_capex_projects_df": china_capex_projects_df,
    "china_related_party_transactions_df": china_related_party_transactions_df,

    # QA
    "china_filing_coverage_report_df": china_filing_coverage_report_df,
    "china_standard_concept_coverage_df": china_standard_concept_coverage_df,
    "china_standard_concept_availability_df": china_standard_concept_availability_df,
    "china_mapping_method_report_df": china_mapping_method_report_df,
    "china_fuzzy_label_quality_report_df": china_fuzzy_label_quality_report_df,
    "china_mapping_quality_df": china_mapping_quality_df,
    "china_enrichment_quality_df": china_enrichment_quality_df,
    "china_fact_origin_report_df": china_fact_origin_report_df,

    # Compatibility aliases
    "china_security_universe_df": china_security_universe_df,
    "china_issuer_universe_df": china_issuer_universe_df,
    "china_etf_membership_intervals_df": china_etf_membership_intervals_df,
    "china_universe_exclusions_df": china_universe_exclusions_df,
    "china_universe_quality_df": china_universe_quality_df,
    "china_block2_listing_metadata_report_df": china_block2_listing_metadata_report_df,
    "china_security_bridge_df": china_security_bridge_df,
    "china_resolved_securities_df": china_resolved_securities_df,
    "china_unresolved_securities_df": china_unresolved_securities_df,
    "china_cninfo_resolution_report_df": china_cninfo_resolution_report_df,
}

print("China transformations complete.")

for name in [
    "china_economic_issuer_universe_df",
    "china_entity_relationship_graph_df",
    "china_cninfo_entities_df",
    "china_disclosure_inventory_df",
    "china_fundamentals_standardised_df",
    "china_incremental_facts_df",
    "china_issuer_enrichment_df",
]:
    print(
        f"  {name}: "
        f"{len(china_block_data[name]):,} rows"
    )

China transformations complete.
  china_economic_issuer_universe_df: 103 rows
  china_entity_relationship_graph_df: 5 rows
  china_cninfo_entities_df: 5 rows
  china_disclosure_inventory_df: 210 rows
  china_fundamentals_standardised_df: 6,985 rows
  china_incremental_facts_df: 8,766 rows
  china_issuer_enrichment_df: 103 rows


In [ ]:
# 29. BLOCK 7 UNIFIED OUTPUT CONTRACT

duplicate_table_names = sorted(
    set(hong_kong_block_data).intersection(china_block_data)
)

# Some cross-source QA tables intentionally cover the same analytical theme.
# Rename any duplicate China tables before creating the published contract.
china_output_renames = {
    "upstream_structured_issuer_coverage_df": (
        "china_upstream_structured_issuer_coverage_df"
    ),
}

china_block_data_published = {}

for table_name, dataframe in china_block_data.items():
    published_name = china_output_renames.get(table_name, table_name)

    if published_name in hong_kong_block_data:
        raise RuntimeError(
            "Duplicate Block 7 output table after renaming: "
            f"{published_name}"
        )

    china_block_data_published[published_name] = dataframe

block_7_data = {
    **hong_kong_block_data,
    **china_block_data_published,
}

block_7_component_summary_df = pd.DataFrame(
    [
        {
            "component": "Hong Kong / HKEX",
            "table_count": len(hong_kong_block_data),
            "total_rows": int(
                sum(len(table) for table in hong_kong_block_data.values())
            ),
        },
        {
            "component": "China / CNINFO and entity enrichment",
            "table_count": len(china_block_data_published),
            "total_rows": int(
                sum(len(table) for table in china_block_data_published.values())
            ),
        },
        {
            "component": "Combined Block 7",
            "table_count": len(block_7_data),
            "total_rows": int(
                sum(len(table) for table in block_7_data.values())
            ),
        },
    ]
)

block_7_data["block_7_component_summary_df"] = (
    block_7_component_summary_df
)

print(
    "Block 7 transformations complete:",
    f"{len(block_7_data):,} published tables"
)

display(block_7_component_summary_df)


Block 7 transformations complete: 122 published tables


,component,table_count,total_rows
0,Hong Kong / HKEX,51,238720
1,China / CNINFO and entity enrichment,70,1277348
2,Combined Block 7,121,1516068


In [ ]:

# 30. STABLE-SCHEMA, MEMORY-SAFE PERSISTENCE

def release_unused_memory():
    gc.collect()

    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass


def infer_stable_arrow_schema(dataframe):
    template = dataframe.head(0).copy()

    for column in template.columns:
        dtype = dataframe[column].dtype

        if (
            pd.api.types.is_object_dtype(dtype)
            or pd.api.types.is_string_dtype(dtype)
            or isinstance(dtype, pd.CategoricalDtype)
        ):
            template[column] = pd.Series(dtype="string")

        elif pd.api.types.is_bool_dtype(dtype):
            template[column] = pd.Series(dtype="boolean")

    schema = pa.Schema.from_pandas(
        template,
        preserve_index=False,
    )

    del template
    release_unused_memory()

    return schema


def prepare_chunk_for_schema(chunk, arrow_schema):
    output = chunk.copy(deep=False)

    for field in arrow_schema:
        column = field.name

        if column not in output.columns:
            continue

        series = output[column]

        if pa.types.is_string(field.type):
            output[column] = series.astype("string")

        elif pa.types.is_timestamp(field.type):
            converted = pd.to_datetime(
                series,
                errors="coerce",
                utc=field.type.tz is not None,
            )

            if field.type.tz is None:
                try:
                    converted = converted.dt.tz_localize(None)
                except Exception:
                    pass

            output[column] = converted

        elif pa.types.is_boolean(field.type):
            output[column] = series.astype("boolean")

        elif pa.types.is_integer(field.type):
            output[column] = (
                pd.to_numeric(series, errors="coerce")
                .round()
                .astype("Int64")
            )

        elif pa.types.is_floating(field.type):
            output[column] = pd.to_numeric(
                series,
                errors="coerce",
            )

    return output


def persist_dataframe_streaming(
    name,
    dataframe,
    output_dir,
    *,
    chunk_rows=PARQUET_CHUNK_ROWS,
    overwrite=True,
):
    output_path = output_dir / f"{name}.parquet"

    if output_path.exists():
        if overwrite:
            output_path.unlink()
        else:
            raise FileExistsError(output_path)

    row_count = len(dataframe)
    column_count = len(dataframe.columns)

    print(
        f"Persisting {name}: "
        f"{row_count:,} rows × {column_count:,} columns"
    )

    arrow_schema = infer_stable_arrow_schema(dataframe)
    writer = None

    try:
        writer = pq.ParquetWriter(
            output_path,
            arrow_schema,
            compression="snappy",
            use_dictionary=True,
            write_statistics=True,
        )

        if row_count == 0:
            empty = dataframe.head(0).copy()

            for field in arrow_schema:
                if pa.types.is_string(field.type):
                    empty[field.name] = pd.Series(dtype="string")
                elif pa.types.is_boolean(field.type):
                    empty[field.name] = pd.Series(dtype="boolean")

            table = pa.Table.from_pandas(
                empty,
                schema=arrow_schema,
                preserve_index=False,
                safe=False,
            )

            writer.write_table(table)

            del table
            del empty

        else:
            for start in range(0, row_count, chunk_rows):
                end = min(start + chunk_rows, row_count)

                chunk = dataframe.iloc[start:end]
                prepared = prepare_chunk_for_schema(
                    chunk,
                    arrow_schema,
                )

                table = pa.Table.from_pandas(
                    prepared,
                    schema=arrow_schema,
                    preserve_index=False,
                    safe=False,
                )

                writer.write_table(
                    table,
                    row_group_size=len(table),
                )

                del table
                del prepared
                del chunk

                release_unused_memory()

                print(
                    f"  {end:,} / {row_count:,} rows",
                    end="\r",
                )

    except Exception:
        if writer is not None:
            writer.close()
            writer = None

        if output_path.exists():
            output_path.unlink()

        release_unused_memory()
        raise

    finally:
        if writer is not None:
            writer.close()

        del writer
        del arrow_schema
        release_unused_memory()

    print(f"  {row_count:,} / {row_count:,} rows completed")

    return {
        "table_name": name,
        "path": str(output_path),
        "row_count": int(row_count),
        "column_count": int(column_count),
        "columns": list(map(str, dataframe.columns)),
        "file_size_bytes": int(output_path.stat().st_size),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

if PERSIST_BLOCK_7_OUTPUTS:
    manifest_rows = []

    preferred_order = [
        "hong_kong_fundamentals_standardised_df",
        "china_fundamentals_standardised_df",
        "hong_kong_filing_metadata_df",
        "china_filing_metadata_df",
        "hong_kong_standard_concept_dictionary_df",
        "china_standard_concept_dictionary_df",
        "hong_kong_mapping_review_queue_df",
        "china_mapping_review_queue_df",
        "hong_kong_unmapped_account_inventory_df",
        "china_unmapped_account_inventory_df",
        "china_entity_relationship_graph_df",
        "china_cninfo_entities_df",
        "china_issuer_enrichment_df",
        "block_7_component_summary_df",
    ]

    persistence_order = (
        [name for name in preferred_order if name in block_7_data]
        + [name for name in block_7_data if name not in preferred_order]
    )

    for table_name in persistence_order:
        record = persist_dataframe_streaming(
            table_name,
            block_7_data[table_name],
            BLOCK_7_OUTPUT_DIR,
            chunk_rows=PARQUET_CHUNK_ROWS,
            overwrite=OVERWRITE_PERSISTED_OUTPUTS,
        )

        manifest_rows.append(record)
        release_unused_memory()

    block_7_manifest = {
        "block": 7,
        "block_name": (
            "Hong Kong and China fundamentals, filings and "
            "Chinese issuer enrichment"
        ),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "project_root": str(PROJECT_ROOT),
        "input_manifests": [
            str(BLOCK_2_MANIFEST_PATH),
            str(BLOCK_3_MANIFEST_PATH),
            str(BLOCK_4_MANIFEST_PATH),
            str(BLOCK_5_MANIFEST_PATH),
            str(BLOCK_6_MANIFEST_PATH),
        ],
        "output_directory": str(BLOCK_7_OUTPUT_DIR),
        "source_systems": [
            "HKEXNEWS",
            "CNINFO official disclosure archive",
        ],
        "universe_policy": (
            "All relevant Hong Kong-listed securities and Chinese economic "
            "issuers represented in the point-in-time ETF universe, including "
            "ADR, H-share, A-share and offshore listing relationships."
        ),
        "extraction_policy": (
            "Deterministic document acquisition, text extraction, account "
            "mapping, fuzzy matching and accounting validation. Unresolved "
            "records are retained for Block 9 AI enrichment and quality control."
        ),
        "fundamental_source_policy": (
            "Prefer usable structured SEC, ESEF, EDINET or DART issuer "
            "fundamentals where appropriate; use HKEX and CNINFO as primary "
            "or supplementary sources according to issuer coverage, listing "
            "relationships and canonical-concept availability."
        ),
        "hkex_stock_identifier_source": HKEX_ACTIVE_STOCKS_URL,
        "hkex_title_search_url": HKEX_TITLE_SEARCH_URL,
        "cninfo_source": "CNINFO official disclosure archive",
        "discovery_start_date": DISCOVERY_START_DATE,
        "discovery_end_date": DISCOVERY_END_DATE,
        "canonical_concept_count": int(
            global_canonical_schema_df["standard_concept"].nunique()
        ),
        "ai_assistance_used": False,
        "downstream_ai_qc_block": 9,
        "parquet_chunk_rows": PARQUET_CHUNK_ROWS,
        "tables": manifest_rows,
    }

    with BLOCK_7_MANIFEST_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            block_7_manifest,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print(
        f"Persisted {len(manifest_rows):,} Block 7 tables."
    )
    print("Manifest:", BLOCK_7_MANIFEST_PATH)


Persisting hong_kong_fundamentals_standardised_df: 534 rows × 61 columns
  534 / 534 rows completed
Persisting china_fundamentals_standardised_df: 6,985 rows × 54 columns
  6,985 / 6,985 rows completed
Persisting hong_kong_filing_metadata_df: 90 rows × 26 columns
  90 / 90 rows completed
Persisting china_filing_metadata_df: 210 rows × 21 columns
  210 / 210 rows completed
Persisting hong_kong_standard_concept_dictionary_df: 90 rows × 9 columns
  90 / 90 rows completed
Persisting china_standard_concept_dictionary_df: 96 rows × 9 columns
  96 / 96 rows completed
Persisting hong_kong_mapping_review_queue_df: 642 rows × 58 columns
  642 / 642 rows completed
Persisting china_mapping_review_queue_df: 2,969 rows × 51 columns
  2,969 / 2,969 rows completed
Persisting hong_kong_unmapped_account_inventory_df: 14,533 rows × 7 columns
  14,533 / 14,533 rows completed
Persisting china_unmapped_account_inventory_df: 43,047 rows × 7 columns
  43,047 / 43,047 rows completed
Persisting china_entity_rel

In [ ]:
# 31. PERSISTENCE AND OUTPUT-CONTRACT VALIDATION

required_tables = {
    "hong_kong_security_universe_df",
    "hong_kong_filing_metadata_df",
    "hong_kong_fundamentals_standardised_df",
    "hong_kong_mapping_review_queue_df",
    "china_economic_issuer_universe_df",
    "china_entity_relationship_graph_df",
    "china_cninfo_entities_df",
    "china_filing_metadata_df",
    "china_fundamentals_standardised_df",
    "china_mapping_review_queue_df",
    "china_issuer_enrichment_df",
    "block_7_component_summary_df",
}

missing_runtime_tables = required_tables.difference(block_7_data)

if missing_runtime_tables:
    raise RuntimeError(
        "Block 7 output contract is missing required runtime tables: "
        f"{sorted(missing_runtime_tables)}"
    )

validation_rows = []

if PERSIST_BLOCK_7_OUTPUTS:
    manifest_names = {
        item["table_name"]
        for item in block_7_manifest["tables"]
    }

    missing_persisted_tables = required_tables.difference(
        manifest_names
    )

    if missing_persisted_tables:
        raise RuntimeError(
            "Block 7 manifest is missing required persisted tables: "
            f"{sorted(missing_persisted_tables)}"
        )

    for table_name in sorted(required_tables):
        table_path = BLOCK_7_OUTPUT_DIR / f"{table_name}.parquet"

        if not table_path.exists():
            raise FileNotFoundError(table_path)

        parquet_file = pq.ParquetFile(table_path)
        persisted_rows = parquet_file.metadata.num_rows
        original_rows = len(block_7_data[table_name])

        if persisted_rows != original_rows:
            raise RuntimeError(
                f"Row-count mismatch for {table_name}: "
                f"{original_rows:,} runtime versus "
                f"{persisted_rows:,} persisted."
            )

        validation_rows.append(
            {
                "table_name": table_name,
                "runtime_rows": original_rows,
                "persisted_rows": persisted_rows,
                "persisted_columns": (
                    parquet_file.metadata.num_columns
                ),
                "status": "PASSED",
            }
        )

        del parquet_file
        gc.collect()

else:
    for table_name in sorted(required_tables):
        validation_rows.append(
            {
                "table_name": table_name,
                "runtime_rows": len(block_7_data[table_name]),
                "persisted_rows": pd.NA,
                "persisted_columns": pd.NA,
                "status": "RUNTIME_ONLY",
            }
        )

block_7_validation_report_df = pd.DataFrame(validation_rows)

display(block_7_validation_report_df)

print(
    "Block 7 persistence validation passed. Block 9 can consume the deterministic "
    "Hong Kong and China outputs for AI enrichment and quality control."
)

,table_name,runtime_rows,persisted_rows,persisted_columns,status
0,block_7_component_summary_df,3,3,3,PASSED
1,china_cninfo_entities_df,5,5,7,PASSED
2,china_economic_issuer_universe_df,103,103,22,PASSED
3,china_entity_relationship_graph_df,5,5,20,PASSED
4,china_filing_metadata_df,210,210,21,PASSED
5,china_fundamentals_standardised_df,6985,6985,54,PASSED
6,china_issuer_enrichment_df,103,103,32,PASSED
7,china_mapping_review_queue_df,2969,2969,51,PASSED
8,hong_kong_filing_metadata_df,90,90,26,PASSED
9,hong_kong_fundamentals_standardised_df,534,534,61,PASSED


Block 7 persistence validation passed. Block 9 can consume the deterministic Hong Kong and China outputs for AI enrichment and quality control.


## Operating notes

### Scope

Block 7 covers:

- Hong Kong-listed securities and HKEXnews filings;
- mainland Chinese entities and CNINFO periodic reports;
- Chinese economic issuers represented through ADRs, H-shares, A-shares or offshore ordinary shares;
- deterministic entity resolution, filing acquisition, account mapping and canonical standardisation.

### Deterministic extraction

The notebook uses official disclosure archives, PDF text extraction, curated account dictionaries, exact matching, fuzzy matching and accounting validation. It does not call an LLM, a vision-language model or an external AI document service.

### Review queues

Ambiguous records are retained rather than silently forced into the canonical dataset. The principal downstream review inputs include:

```python
hong_kong_mapping_review_queue_df
hong_kong_unmapped_account_inventory_df
china_mapping_review_queue_df
china_unmapped_account_inventory_df
china_cninfo_entity_bridge_unresolved_df
china_duplicate_primary_facts_df
```

These tables provide the controlled exception set for Block 9.

### Source precedence

Structured regulatory fundamentals remain preferred where they provide sufficient issuer-level coverage. HKEX and CNINFO facts are classified as primary, supplementary or duplicative according to source availability, issuer relationships and canonical-concept coverage.

### Published contract

All persisted files are written to:

```python
data/interim/block_7/
```

The manifest is:

```python
data/interim/block_7/block_7_manifest.json
```

Block 10 should consume the Block 9 quality-controlled outputs rather than bypassing the enrichment and validation layer.
